In [82]:
THE_FOLD_I_WANT=1
THE_FOLDs_I_WANT=[3,4,5]
tile_size_I_want=128*2
reg_model_name='mit-b5'
seg_model_name='mit-b0'

In [ ]:
TRAIN_REGRESSOR=False

In [ ]:
print("starting.......")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os
# import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:

def scale(arr):
    arr = arr.astype(np.float32)
    if arr.ndim == 4:
        for i in range(len(arr)):
            arr[i,:,:,:] =  (arr[i,:,:,:] - arr[i,:,:,:].min())/ (arr[i,:,:,:].max() - arr[i,:,:,:].min() + 1e-6)
    elif arr.ndim ==3:
        arr[:,:,:] =  (arr[:,:,:] - arr[:,:,:].min())/ (arr[:,:,:].max() - arr[:,:,:].min() + 1e-6)
    else :
        raise Exception("Unknown dimension array provided, expected 3 or 4, got", arr.ndim)
    return arr


def get_RGB(S1, S2):
    blue_c = S1[:,:,:,0] - S1[:,:,:,1]
    blue_c = blue_c[:,:,:,np.newaxis]
    S1_rgb = np.concatenate([S1,blue_c], axis=3)
    S2_rgb = S2[:,:,:,:3]
    return S1_rgb, S2_rgb



def show_preds(S1,S2,S1_pred,S2_pred,epoch, save=True):
    for i in range(len(S1)):
        # plt.figure(figsize=(10,8))
        fig, ax = plt.subplots(nrows=1, ncols=4,figsize= (16,12))
        # plt.subplot(1,4,1)
        ax[0].imshow(scale(S1)[i])
        ax[0].set_title('Original S1')

        ax[1].imshow(scale(S1_pred)[i])
        ax[1].set_title('Generated S1')
        # plt.subplot(1,4,2)
        ax[2].imshow(scale(S2_pred)[i])
        ax[2].set_title('Generated S2')
        # plt.subplot(1,4,3)
        ax[3].imshow(scale(S2)[i])
        ax[3].set_title('Original S2')
        # plt.subplot(1,4,4)
        fig.tight_layout()
        #Save this fig
        if save:
            fig.savefig(f"/content/drive/MyDrive/Flood_Monitoring/CycleGAN2/Pred_Epoch{epoch}_{i}.png")
        plt.show()


In [ ]:
DATA_PATH = "/content/drive/MyDrive/Flood_Monitoring/Cloud2Street/chips"
OUT_PATH = "/content/drive/MyDrive/Flood_Monitoring/Cloud2Street/outputs"

In [ ]:
! ls "/kaggle/input/datasets/sakibahmed91/new-flood-data/chips/0c7daa97-37f6-4862-867f-b3843f298d9e/s1"

In [ ]:
import os
import glob
import numpy as np
import rasterio
import re
from tqdm import tqdm
from rasterio.warp import transform_bounds
DATA_PATH = "/kaggle/input/datasets/sakibahmed91/new-flood-data/chips"

# ==========================================================
# Discover events
# ==========================================================

flood_events = sorted([
    d for d in os.listdir(DATA_PATH)
    if os.path.isdir(os.path.join(DATA_PATH, d))
])

event_to_id = {e: i for i, e in enumerate(flood_events)}
id_to_event = {i: e for e, i in event_to_id.items()}

# ==========================================================
# Matching helper
# ==========================================================

def get_key(folder_name):
    """
    Example: S1B_..._01845-00514 -> 01845-00514
    """
    return folder_name.split("_")[-1]

# ==========================================================
# Build matched pair list
# ==========================================================

pairs = []

for event in flood_events:
    s2_lookup = {}
    s2_folders = glob.glob(os.path.join(DATA_PATH, event, "s2", "*"))

    for folder in s2_folders:
        key = get_key(os.path.basename(folder))
        s2_lookup[key] = folder

    s1_folders = glob.glob(os.path.join(DATA_PATH, event, "s1", "*"))

    for s1_folder in s1_folders:
        key = get_key(os.path.basename(s1_folder))
        if key in s2_lookup:
            pairs.append((event, s1_folder, s2_lookup[key]))

print("Matched pairs:", len(pairs))

# ==========================================================
# Allocate arrays (Including Spatial and Temporal)
# ==========================================================

N = len(pairs)

s1_arr = np.empty((N, 512, 512, 2), dtype=np.float32)
s2_arr = np.empty((N, 512, 512, 4), dtype=np.float32)
s1_water_arr = np.empty((N, 512, 512), dtype=np.uint8)
s2_water_arr = np.empty((N, 512, 512), dtype=np.uint8)
cloud_arr = np.empty((N, 512, 512), dtype=np.uint8)
event_arr = np.empty((N,), dtype=np.int32)

# 🌍 NEW: Spatial Bounds Array (Left, Bottom, Right, Top)
bounds_arr = np.empty((N, 4), dtype=np.float32)

# ⏱️ NEW: Temporal Date Array (Stores YYYYMMDD as string)
date_arr = np.empty((N,), dtype=object)

# ==========================================================
# Load data
# ==========================================================

for idx, (event, s1_folder, s2_folder) in enumerate(tqdm(pairs)):

    # --------------------------
    # S1 & Extract Metadata
    # --------------------------
    with rasterio.open(os.path.join(s1_folder, "VV.tif")) as src:
        vv = src.read(1)
        
        # 🌍 FIX: Transform native bounds to Lat/Lon (EPSG:4326) degrees
        left, bottom, right, top = transform_bounds(
            src.crs, 
            'EPSG:4326', 
            src.bounds.left, 
            src.bounds.bottom, 
            src.bounds.right, 
            src.bounds.top
        )
        bounds_arr[idx] = [left, bottom, right, top]

    with rasterio.open(os.path.join(s1_folder, "VH.tif")) as src:
        vh = src.read(1)

    s1_arr[idx] = np.stack([vv, vh], axis=-1)
    
    # ⏱️ Extract Temporal Information from folder name
    # Sentinel standard strings contain dates like 20190810T224512
    # This regex pulls the 8-digit YYYYMMDD sequence
    s1_folder_name = os.path.basename(s1_folder)
    date_match = re.search(r'(20\d{6})', s1_folder_name)
    if date_match:
        date_arr[idx] = date_match.group(1)
    else:
        date_arr[idx] = "Unknown"

    # --------------------------
    # S2 RGB + NIR
    # --------------------------
    bands = []
    for band in ["B4", "B3", "B2", "B8"]:
        with rasterio.open(os.path.join(s2_folder, f"{band}.tif")) as src:
            bands.append(src.read(1))
    s2_arr[idx] = np.stack(bands, axis=-1)

    # --------------------------
    # S1 Water Label
    # --------------------------
    with rasterio.open(os.path.join(s1_folder, "LabelWater.tif")) as src:
        s1_water_arr[idx] = src.read(1)

    # --------------------------
    # S2 Water Label
    # --------------------------
    with rasterio.open(os.path.join(s2_folder, "LabelWater.tif")) as src:
        s2_water_arr[idx] = src.read(1)

    # --------------------------
    # Cloud
    # --------------------------
    with rasterio.open(os.path.join(s2_folder, "LabelCloud.tif")) as src:
        cloud_arr[idx] = src.read(1)

    # --------------------------
    # Event ID
    # --------------------------
    event_arr[idx] = event_to_id[event]

# ==========================================================
# Check shapes
# ==========================================================

print("S1:", s1_arr.shape)
print("S2:", s2_arr.shape)
print("S1 Water:", s1_water_arr.shape)
print("S2 Water:", s2_water_arr.shape)
print("Cloud:", cloud_arr.shape)
print("Event:", event_arr.shape)
print("Bounds:", bounds_arr.shape)
print("Dates:", date_arr.shape)

print('Data loaded successfully with spatial and temporal metadata.')

In [ ]:
pip install reverse_geocoder

In [ ]:
S1_images = s1_arr
S2_images = s2_arr
S2_clouds = cloud_arr
S2_water = s2_water_arr
S1_water= s1_water_arr
S1_images.shape, S2_images.shape, S2_water.shape

In [ ]:
# cloud2Street_data.keys()

In [ ]:
import folium
import numpy as np
from datetime import datetime

# ---------------------------------------------------------
# 1. Map Setup
# ---------------------------------------------------------
mean_lon = np.mean([b[0] + (b[2] - b[0]) / 2 for b in bounds_arr])
mean_lat = np.mean([b[1] + (b[3] - b[1]) / 2 for b in bounds_arr])

m = folium.Map(location=[mean_lat, mean_lon], zoom_start=3, tiles="CartoDB positron")

# ---------------------------------------------------------
# 2. Define Temporal Color Palette (By Year)
# ---------------------------------------------------------
year_colors = {
    '2016': '#4575b4', 
    '2017': '#91bfdb', 
    '2018': '#fee090', 
    '2019': '#fc8d59', 
    '2020': '#d73027'  
}
fallback_color = '#808080' 

print("Calculating extents, applying colors, generating labels, and rendering map...")

# ---------------------------------------------------------
# 3. Group by Event, Assign Color, Label, and Plot
# ---------------------------------------------------------
unique_events = np.unique(event_arr)

for ev_id in unique_events:
    indices = np.where(event_arr == ev_id)[0]
    
    ev_bounds = bounds_arr[indices]
    ev_dates = date_arr[indices]
    
    min_lon = np.min(ev_bounds[:, 0])
    min_lat = np.min(ev_bounds[:, 1])
    max_lon = np.max(ev_bounds[:, 2])
    max_lat = np.max(ev_bounds[:, 3])
    
    # --- Calculate Center for the Label ---
    center_lat = (min_lat + max_lat) / 2
    center_lon = (min_lon + max_lon) / 2
    
    unique_dates = np.unique(ev_dates)
    
    # --- Color Assignment Logic ---
    valid_dates = sorted([d for d in unique_dates if d != "Unknown"])
    
    if len(valid_dates) > 0:
        primary_year = valid_dates[0][:4] 
        poly_color = year_colors.get(primary_year, fallback_color)
    else:
        poly_color = fallback_color
        
    # --- Date Formatting Logic ---
    human_readable_dates = []
    for d in unique_dates:
        if d == "Unknown":
            human_readable_dates.append(d)
        else:
            try:
                parsed_date = datetime.strptime(str(d), "%Y%m%d")
                human_readable_dates.append(parsed_date.strftime("%B %d, %Y"))
            except ValueError:
                human_readable_dates.append(str(d)) 
                
    dates_str = " | ".join(human_readable_dates)
    event_name = id_to_event[ev_id]
    extent_bounds = [[min_lat, min_lon], [max_lat, max_lon]]
    
    popup_text = f"<b>Event Name:</b> {event_name}<br><b>Event ID:</b> {ev_id}<br><b>Time(s):</b> {dates_str}"
    
    # Draw the rectangle
    folium.Rectangle(
        bounds=extent_bounds,
        color=poly_color,      
        weight=3,
        fill=True,
        fill_color=poly_color, 
        fill_opacity=0.5,     
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)
    
    # --- NEW: Add the Event ID Label as a DivIcon ---
    folium.Marker(
        location=[center_lat, center_lon],
        icon=folium.DivIcon(
            html=f"""
                <div style="
                    font-size: 16px; 
                    font-weight: bold; 
                    color: black; 
                    text-shadow: 2px 2px 4px white, -2px -2px 4px white, 2px -2px 4px white, -2px 2px 4px white;
                    text-align: center;
                    width: 40px;
                    margin-left: -20px;
                    margin-top: -10px;
                ">
                    {ev_id}
                </div>
            """
        )
    ).add_to(m)

# ---------------------------------------------------------
# 4. Add a Custom HTML Legend
# ---------------------------------------------------------
legend_html = '''
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 110px; height: 160px; 
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white; padding: 10px; opacity: 0.9;">
    <b>Timeline</b><br>
    <i style="background:#4575b4; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>2016<br>
    <i style="background:#91bfdb; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>2017<br>
    <i style="background:#fee090; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>2018<br>
    <i style="background:#fc8d59; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>2019<br>
    <i style="background:#d73027; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>2020<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Display the map
m

In [ ]:
np.unique(date_arr)

In [ ]:
def NDWI(arr):
    if arr.ndim == 4:
        return (arr[:,:,:,0] - arr[:,:,:,1])/(arr[:,:,:,0] + arr[:,:,:,1] + 1e-6)
    elif arr.ndim ==3:
        return (arr[:,:,0] - arr[:,:,1])/(arr[:,:,0] + arr[:,:,1] + 1e-6)
    else :
        raise Exception("Unknown dimension array provided, expected 3 or 4, got", arr.ndim)

def scale(arr):
    arr = arr.astype(np.float32)
    if arr.ndim == 4:
        for i in range(len(arr)):
            arr[i,:,:,:] =  (arr[i,:,:,:] - arr[i,:,:,:].min())/ (arr[i,:,:,:].max() - arr[i,:,:,:].min() + 1e-6)
    elif arr.ndim ==3:
        arr[:,:,:] =  (arr[:,:,:] - arr[:,:,:].min())/ (arr[:,:,:].max() - arr[:,:,:].min() + 1e-6)
    else :
        raise Exception("Unknown dimension array provided, expected 3 or 4, got", arr.ndim)
    return arr
def tile(im, M, N):
    return np.array([im[x:x+M,y:y+N,:] for x in range(0,im.shape[0],M) for y in range(0,im.shape[1],N)])

In [ ]:
# S2_NDWI = NDWI(scale(S2_images[:,:,:,1]),scale(S2_images[:,:,:,3]))[:,:,:,None]
S2_NDWI = scale(S2_images[:,:,:,[1,3]])

In [ ]:
n_samples = np.random.choice(S1_images.shape[0], 5, replace=False)
fig, ax = plt.subplots(nrows=5, ncols=5, figsize=(20,12))

for i in range(len(n_samples)):
    c1 = ax[0,i].imshow(scale(S1_images[n_samples[i]])[:,:,0], cmap="RdBu")
    c2 = ax[1,i].imshow(scale(S1_images[n_samples[i]])[:,:,1], cmap="RdBu")
    c3 = ax[2,i].imshow(NDWI(S2_NDWI[n_samples[i]]), cmap="winter_r")
    ax[3,i].imshow(scale(NDWI(S2_NDWI[n_samples[i]][:,:,None])), cmap="winter_r")
    c4 = ax[4,i].imshow(S2_water[n_samples[i]], cmap="winter_r")

    # Column titles (only once per column)
    ax[0,i].set_title(f"Sample {i+1}")

# Row titles
ax[0,0].set_ylabel("VV", fontsize=14)
ax[1,0].set_ylabel("VH", fontsize=14)
ax[2,0].set_ylabel("NDWI (S2)", fontsize=14)
ax[3,0].set_ylabel("Scaled NDWI", fontsize=14)
ax[4,0].set_ylabel("Water Mask", fontsize=14)

# Colorbars
plt.colorbar(c1, ax=ax[0,4], shrink=0.8)
plt.colorbar(c2, ax=ax[1,4], shrink=0.8)
plt.colorbar(c3, ax=ax[2,4], shrink=0.8)
plt.colorbar(c4, ax=ax[4,4], shrink=0.8)

plt.tight_layout()
plt.show()


In [ ]:
S2_water.shape,S1_images.shape,S2_clouds.shape

In [ ]:
import numpy as np

# ==========================================================
# Mathematical Knee Finder Function
# ==========================================================
def find_negligible_threshold(disagreement_pct):
    """
    Finds the mathematical knee point of the live distribution curve 
    using the maximum orthogonal distance to the secant line.
    """
    sorted_disagreements = np.sort(disagreement_pct)
    n_points = len(sorted_disagreements)
    
    # Coordinates: X is the index, Y is the sorted disagreement value
    x = np.arange(n_points)
    y = sorted_disagreements
    
    # Endpoints of the baseline secant line
    x1, y1 = x[0], y[0]
    x2, y2 = x[-1], y[-1]
    
    # Orthogonal distance calculation from every coordinate to the baseline line
    numerator = np.abs((y2 - y1) * x - (x2 - x1) * y + x2 * y1 - y2 * x1)
    denominator = np.sqrt((y2 - y1)**2 + (x2 - x1)**2)
    distances = numerator / denominator
    
    # Identify the index where the distance curve bends sharpest (the knee)
    knee_idx = np.argmax(distances)
    knee_value = sorted_disagreements[knee_idx]
    
    return knee_value


# ==========================================================
# Tiling Pipeline (Your Existing Variables)
# ==========================================================
tile_size = tile_size_I_want

# Tiling your active dataset images
S1_tiles = [tile(im, tile_size, tile_size) for im in S1_images]
S2_tiles = [tile(im, tile_size, tile_size) for im in S2_NDWI]
S1_water_tiles = [tile(im, tile_size, tile_size) for im in S1_water[:,:,:,None]]
S2_water_tiles = [tile(im, tile_size, tile_size) for im in S2_water[:,:,:,None]]
S2_cloud_tiles = [tile(im, tile_size, tile_size) for im in S2_clouds[:,:,:,None]]

# 1. Count exactly how many tiles each individual image produces
tiles_per_image_counts = [len(tile(im, tile_size, tile_size)) for im in S1_images]

# 2. Repeat each event ID and date according to its specific image's tile count
event_arr_tile = np.repeat(event_arr, tiles_per_image_counts)
date_arr_tile = np.repeat(date_arr, tiles_per_image_counts)  # Added time array tiling

# Stacking them into uniform NumPy arrays
S1_tiles = np.vstack(S1_tiles)
S2_tiles = np.vstack(S2_tiles)
S1_water_tiles = np.vstack(S1_water_tiles)
S2_water_tiles = np.vstack(S2_water_tiles)
S2_cloud_tiles = np.vstack(S2_cloud_tiles)


# ==========================================================
# Run Metrics Collection & Dynamic Filtering
# ==========================================================

# 1. Calculate the standard cloud percentage per tile
# cloud_PCT = (S2_cloud_tiles.sum(axis=(1, 2)) / (tile_size * tile_size)).squeeze()

# 2. Calculate the raw pixel disagreement percentage between S1 and S2 water targets
disagreement_PCT = ((S1_water_tiles != S2_water_tiles).sum(axis=(1, 2)) / (tile_size * tile_size)).squeeze()

# 3. Dynamically compute the absolute data-driven threshold from your active tiles
Threshold = find_negligible_threshold(disagreement_PCT)
Cloud_Threshold = 0.00  # Strict cloud ceiling constraint

# 4. Generate the Boolean indexing mask using the calculated knee threshold
keep_mask = (disagreement_PCT <= Threshold)

# 5. Filter all parallel tile blocks simultaneously down to matching distributions
S1_tiles = S1_tiles[keep_mask]
S2_tiles = S2_tiles[keep_mask]
S1_water_tiles = S1_water_tiles[keep_mask]
S2_water_tiles = S2_water_tiles[keep_mask]
S2_cloud_tiles = S2_cloud_tiles[keep_mask]
event_arr_tile = event_arr_tile[keep_mask]
date_arr_tile = date_arr_tile[keep_mask]  # Added time array filtering


# ==========================================================
# Verification Report
# ==========================================================
print(f"Calculated Data Knee Threshold : {Threshold * 100:.2f}%")
print(f"Total aligned tiles remaining  : {S1_tiles.shape[0]}")
print(f"Dates Array Shape              : {date_arr_tile.shape}")  # Added verification

# 1. Calculate cloud percentage per tile
cloud_PCT = (S2_cloud_tiles.sum(axis=(1, 2)) / (tile_size * tile_size)).squeeze()
Water_tiles = S2_water_tiles

In [ ]:
S2_tiles.shape

In [ ]:
# # Filter tiles using cloud mask
# S1_filtered    = S1_tiles[cloud_PCT == 0]
# S2_filtered    = S2_tiles[cloud_PCT == 0]
# Water_filtered = Water_tiles[cloud_PCT == 0]

# print(S1_filtered.shape, S2_filtered.shape, Water_filtered.shape)
# # 
# # Scale S1 & S2
# S1_filtered = scale(S1_filtered)
# S2_filtered = scale(NDWI(scale(S2_filtered)))[:, :, :, None]   # keep 4D shape (N,H,W,1)
# from sklearn.model_selection import train_test_split

# # Step 1: train + temp (val+test)
# S1_train, S1_temp, \
# S2_train, S2_temp, \
# Water_train, Water_temp = train_test_split(
#     S1_filtered, S2_filtered, Water_filtered,
#     test_size=0.3,  # 30% goes to temp
#     random_state=47
# )

# # Step 2: split temp into val and test
# S1_val, S1_test, \
# S2_val, S2_test, \
# Water_val, Water_test = train_test_split(
#     S1_temp, S2_temp, Water_temp,
#     test_size=0.5,  # split 30% into 15% val, 15% test
#     random_state=47
# )

# print(S1_train.shape, S1_val.shape, S1_test.shape)



In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import StratifiedGroupKFold

# # ==========================================
# # 1. FILTERING
# # ==========================================
# S1_filtered = S1_tiles[cloud_PCT == 0]
# S2_filtered = S2_tiles[cloud_PCT == 0]
# Water_filtered = S2_water_tiles[cloud_PCT == 0] # Assuming this is your target mask
# event_arr_filtered = event_arr_tile[cloud_PCT == 0]

# print("Filtered Shapes:")
# print(f"S1: {S1_filtered.shape}, S2: {S2_filtered.shape}, Water: {Water_filtered.shape}")
# print("-" * 50)

# # ==========================================
# # 2. SCALING
# # ==========================================
# # Assuming scale() and NDWI() are your custom functions

# # Scale S1 & S2
# S1_filtered = scale(S1_filtered)
# S2_filtered = scale(NDWI(scale(S2_filtered)))[:, :, :, None]   # keep 4D shape (N,H,W,1)
# from sklearn.model_selection import train_test_split

# # ==========================================
# # 3. PREPARE EVENT-LEVEL STRATIFICATION
# # ==========================================
# unique_events = np.unique(event_arr_filtered)
# event_water_pct = []

# # Calculate water ratio for each event after cloud filtering
# for event in unique_events:
#     event_mask = (event_arr_filtered == event)
#     event_tiles = Water_filtered[event_mask]
#     water_pct = np.sum(event_tiles == 1) / event_tiles.size
#     event_water_pct.append(water_pct)

# # Bin events into Low, Med, High water to keep splits balanced
# strata = pd.qcut(event_water_pct, q=3, labels=["low", "med", "high"], duplicates='drop')

# event_meta = pd.DataFrame({'event_id': unique_events, 'strata': strata})

# # Map the strata back to the individual tiles
# tile_strata = np.array([event_meta.loc[event_meta['event_id'] == ev, 'strata'].values[0] for ev in event_arr_filtered])

# # ==========================================
# # 4. PERFORM THE LEAK-FREE SPLIT
# # ==========================================
# X_dummy = np.arange(len(event_arr_filtered))

# # Step A: Isolate Test Set (~15%)
# # n_splits=7 means 1 split is ~14.2% of the data
# sgkf_test = StratifiedGroupKFold(n_splits=7)
# train_val_idx, test_idx = next(sgkf_test.split(X_dummy, y=tile_strata, groups=event_arr_filtered))

# # Step B: Split the remaining data into Train and Val (~15% Val, ~70% Train)
# X_train_val = X_dummy[train_val_idx]
# strata_train_val = tile_strata[train_val_idx]
# groups_train_val = event_arr_filtered[train_val_idx]

# # n_splits=6 on the remaining 85% yields ~14.1% for Val, leaving ~71% for Train
# sgkf_val = StratifiedGroupKFold(n_splits=6)
# train_idx_rel, val_idx_rel = next(sgkf_val.split(X_train_val, y=strata_train_val, groups=groups_train_val))

# # Convert relative indices back to original absolute indices
# train_idx = X_train_val[train_idx_rel]
# val_idx = X_train_val[val_idx_rel]
# S1_cv_pool = S1_filtered[train_val_idx]
# S2_cv_pool = S2_filtered[train_val_idx]
# Water_cv_pool = Water_filtered[train_val_idx]
# events_cv_pool = event_arr_filtered[train_val_idx]
# strata_cv_pool = tile_strata[train_val_idx]

# # ==========================================
# # 5. ASSIGN FINAL DATASETS
# # ==========================================
# S1_train, S1_val, S1_test = S1_filtered[train_idx], S1_filtered[val_idx], S1_filtered[test_idx]
# S2_train, S2_val, S2_test = S2_filtered[train_idx], S2_filtered[val_idx], S2_filtered[test_idx]
# Water_train, Water_val, Water_test = Water_filtered[train_idx], Water_filtered[val_idx], Water_filtered[test_idx]

# # Print final verification
# print("\nFinal Split Shapes:")
# print(f"Train S1: {S1_train.shape} | Val S1: {S1_val.shape} | Test S1: {S1_test.shape}")
# print(f"Train S2: {S2_train.shape} | Val S2: {S2_val.shape} | Test S2: {S2_test.shape}")
# print(f"Train Water: {Water_train.shape} | Val Water: {Water_val.shape} | Test Water: {Water_test.shape}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import reverse_geocoder as rg

# ==========================================
# 1. FILTERING & SCALING
# ==========================================
S1_filtered = S1_tiles[cloud_PCT == 0]
S2_filtered = S2_tiles[cloud_PCT == 0]
Water_filtered = S2_water_tiles[cloud_PCT == 0]
event_arr_filtered = event_arr_tile[cloud_PCT == 0]
date_arr_filtered = date_arr_tile[cloud_PCT == 0]  # Swapped to date_arr_tile

S1_filtered = scale(S1_filtered)
S2_filtered = scale(NDWI(scale(S2_filtered)))[:, :, :, None]


# ==========================================
# 2. GEOGRAPHIC & SPATIAL EVENT COMPILATION
# ==========================================
print("\n[Data Profiling] Compiling event spatial metrics...")
unique_events = np.unique(event_arr_filtered)
event_stats = []

for ev in unique_events:
    indices = np.where(event_arr == ev)[0]
    ev_bounds = bounds_arr[indices]
    
    center_lat = np.mean([np.min(ev_bounds[:, 1]), np.max(ev_bounds[:, 3])])
    center_lon = np.mean([np.min(ev_bounds[:, 0]), np.max(ev_bounds[:, 2])])
    
    event_stats.append({
        'event_id': ev, 
        'lat': center_lat,
        'lon': center_lon
    })

df_events = pd.DataFrame(event_stats)

# Run geographic KMeans clustering (4 pseudo-continents)
coords = df_events[['lat', 'lon']].values
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_events['spatial_cluster'] = kmeans.fit_predict(coords)

# Reverse geocode event centers to fetch country codes
print("[Data Profiling] Reverse geocoding event locations...")
geo_results = rg.search(list(zip(df_events['lat'], df_events['lon'])), verbose=False)
df_events['country_code'] = [res['cc'] for res in geo_results]

# --- EXHAUSTIVE GLOBAL ISO-2 TO CONTINENT MAPPING ---
ASIA = ['AF','AM','AZ','BD','BH','BN','BT','KH','CN','CY','GE','IN','ID','IR','IQ','IL','JP','JO','KZ','KP','KR','KW','KG','LA','LB','MY','MV','MN','MM','NP','OM','PK','PS','PH','QA','SA','SG','LK','SY','TW','TJ','TH','TL','TR','TM','AE','UZ','VN','YE']
EUROPE = ['AL','AD','AT','BY','BE','BA','BG','HR','CZ','DK','EE','FI','FR','DE','GR','HU','IS','IE','IT','LV','LI','LT','LU','MT','MD','MC','ME','NL','MK','NO','PL','PT','RO','RU','SM','RS','SK','SI','ES','SE','CH','UA','GB','VA']
AFRICA = ['DZ','AO','BJ','BW','BF','BI','CV','CM','CF','TD','KM','CD','CG','CI','DJ','EG','GQ','ER','SZ','ET','GA','GM','GH','GN','GW','KE','LS','LR','LY','MG','MW','ML','MR','MA','MZ','NA','NE','NG','RW','ST','SN','SC','SL','SO','ZA','SS','SD','TZ','TG','TN','UG','ZM','ZW']
NORTH_AMERICA = ['AG','BS','BB','BZ','CA','CR','CU','DM','DO','SV','GD','GT','HT','HN','JM','MX','NI','PA','KN','LC','VC','TT','US','PR','GL']
SOUTH_AMERICA = ['AR','BO','BR','CL','CO','EC','GY','PY','PE','SR','UY','VE','FK','GF']
OCEANIA = ['AU','FJ','KI','MH','FM','NR','NZ','PW','PG','WS','SB','TO','TV','VU','AS','GU','PF','NC']

CC_TO_CONTINENT = {}
for cc in ASIA: CC_TO_CONTINENT[cc] = 'Asia'
for cc in EUROPE: CC_TO_CONTINENT[cc] = 'Europe'
for cc in AFRICA: CC_TO_CONTINENT[cc] = 'Africa'
for cc in NORTH_AMERICA: CC_TO_CONTINENT[cc] = 'North America'
for cc in SOUTH_AMERICA: CC_TO_CONTINENT[cc] = 'South America'
for cc in OCEANIA: CC_TO_CONTINENT[cc] = 'Oceania'

df_events['continent'] = df_events['country_code'].map(CC_TO_CONTINENT).fillna('Other / Island')


# ==========================================
# 3. BUILD TILE-LEVEL DATAFRAME FOR ANALYSES
# ==========================================
df_tiles = pd.DataFrame({
    'event_id': event_arr_filtered,
    'date': pd.to_datetime(date_arr_filtered)  # Swapped to date_arr_filtered
})

# Extract granular time horizons
df_tiles['year'] = df_tiles['date'].dt.year
df_tiles['month'] = df_tiles['date'].dt.strftime('%B')
df_tiles['year_month'] = df_tiles['date'].dt.to_period('M')

# Map event spatial metadata back down to individual tiles
df_profile = df_tiles.merge(df_events, on='event_id', how='left')


# ==========================================
# 4. EXECUTE UNTRUNCATED DISTRIBUTION REPORT
# ==========================================
# Configure pandas to output EVERYTHING without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

def print_dist(df, column, title):
    print(f"\n{'='*15} {title} {'='*15}")
    counts = df[column].value_counts()
    pcts = df[column].value_counts(normalize=True) * 100
    
    report = pd.DataFrame({'Tile Count': counts, 'Percentage (%)': pcts.round(2)})
    print(report.to_string())

# 1. Cluster-wise Percentage
print_dist(df_profile, 'spatial_cluster', 'CLUSTER-WISE TILE DISTRIBUTION')

# 2. Continent-wise Percentage
print_dist(df_profile, 'continent', 'CONTINENT-WISE TILE DISTRIBUTION')

# 3. Time-wise Percentage
print_dist(df_profile, 'year', 'TIME-WISE TILE DISTRIBUTION (BY YEAR)')
print_dist(df_profile, 'month', 'SEASONAL TILE DISTRIBUTION (BY MONTH)')

# 4. Country-wise Percentage (Prints EVERY country detected)
print_dist(df_profile, 'country_code', 'COMPLETE COUNTRY-WISE TILE DISTRIBUTION')

# 5. Event-wise Percentage (Prints EVERY single event code)
print(f"\n{'='*15} COMPLETE EVENT-WISE TILE DISTRIBUTION {'='*15}")
event_counts = df_profile['event_id'].value_counts()
event_pcts = df_profile['event_id'].value_counts(normalize=True) * 100
df_ev_dist = pd.DataFrame({'Tile Count': event_counts, 'Percentage (%)': event_pcts.round(2)})

print(f"Total Unique Cloud-Free Events: {len(df_ev_dist)}\n")
print(df_ev_dist.to_string())

print("\nEvent Size Distribution Summary Statistics:")
print(df_ev_dist['Tile Count'].describe().to_string())
print(f"\n{'='*55}\n")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ==========================================
# 1. FILTERING & SCALING
# ==========================================
S1_filtered = S1_tiles[cloud_PCT == 0]
S2_filtered = S2_tiles[cloud_PCT == 0]
Water_filtered = S2_water_tiles[cloud_PCT == 0]
event_arr_filtered = event_arr_tile[cloud_PCT == 0]

S1_filtered = scale(S1_filtered)
S2_filtered = scale(NDWI(scale(S2_filtered)))[:, :, :, None]

# ==========================================
# 2. EVENT-LEVEL STRATIFICATION
# ==========================================
unique_events = np.unique(event_arr_filtered)
event_water_pct = []

for event in unique_events:
    event_mask = (event_arr_filtered == event)
    event_tiles = Water_filtered[event_mask]
    water_pct = np.sum(event_tiles == 1) / event_tiles.size
    event_water_pct.append(water_pct)

# Absolute bins optimized for your exact distribution
bins = [-np.inf, 0.10, 0.40, np.inf]
strata = pd.cut(event_water_pct, bins=bins, labels=["low", "med", "high"])
tile_strata = np.array([strata[np.where(unique_events == ev)[0][0]] for ev in event_arr_filtered])

event_meta = pd.DataFrame({'event_id': unique_events, 'strata': strata})

# ==========================================
# 3. LEAK-FREE SPLIT WITH SINGLETON PROTECTION
# ==========================================
# Identify strata that only have 1 single event (your 94.6% event)
strata_counts = event_meta['strata'].value_counts()
singleton_strata = strata_counts[strata_counts == 1].index

is_singleton = event_meta['strata'].isin(singleton_strata)
singletons_df = event_meta[is_singleton]
splittable_df = event_meta[~is_singleton]

# Isolate 15% of events for the Test Set
events_cv_pool_splittable, events_test = train_test_split(
    splittable_df, 
    test_size=0.15, 
    stratify=splittable_df['strata'], 
    random_state=42
)

# Force the 94.6% singleton event into the CV pool so it's available for training folds
events_cv_pool_meta = pd.concat([events_cv_pool_splittable, singletons_df]).reset_index(drop=True)

# ==========================================
# 4. MAP BACK TO TILES & GENERATE TARGET POOLS
# ==========================================
# Create event-wise boolean masks for the tiles
cv_pool_mask = np.isin(event_arr_filtered, events_cv_pool_meta['event_id'])
test_mask = np.isin(event_arr_filtered, events_test['event_id'])

# Your exact target CV pool variables
S1_cv_pool = S1_filtered[cv_pool_mask]
S2_cv_pool = S2_filtered[cv_pool_mask]
Water_cv_pool = Water_filtered[cv_pool_mask]
events_cv_pool = event_arr_filtered[cv_pool_mask]
strata_cv_pool = tile_strata[cv_pool_mask]

# Isolated Test Set (Holdout)
S1_test = S1_filtered[test_mask]
S2_test = S2_filtered[test_mask]
Water_test = Water_filtered[test_mask]

# ==========================================
# 5. VERIFICATION
# ==========================================
print("\nFinal Split Shapes:")
print(f"CV Pool S1:    {S1_cv_pool.shape} | Test S1:    {S1_test.shape}")
print(f"CV Pool Water: {Water_cv_pool.shape} | Test Water: {Water_test.shape}")
print("-" * 50)
print(f"Total Events in CV Pool: {len(events_cv_pool_meta)} (Includes the 94.6% flood)")
print(f"Total Events in Test Set: {len(events_test)}")

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.cluster import KMeans

# # ==========================================
# # 1. BUILD GLOBAL EVENT STATS & WATER PCT
# # ==========================================
# unique_events = np.unique(event_arr_filtered)
# event_stats = []

# for ev in unique_events:
#     # --- Spatial Bounds ---
#     indices = np.where(event_arr == ev)[0]
#     ev_bounds = bounds_arr[indices]
    
#     center_lat = np.mean([np.min(ev_bounds[:, 1]), np.max(ev_bounds[:, 3])])
#     center_lon = np.mean([np.min(ev_bounds[:, 0]), np.max(ev_bounds[:, 2])])
    
#     # --- Target (Water) Distribution ---
#     event_mask = (event_arr_filtered == ev)
#     tile_count = np.sum(event_mask)
    
#     event_tiles = Water_filtered[event_mask]
#     # Handle edge case where event might have 0 tiles after cloud filtering
#     water_pct = np.sum(event_tiles == 1) / event_tiles.size if event_tiles.size > 0 else 0
    
#     event_stats.append({
#         'event_id': ev, 
#         'tile_count': tile_count,
#         'lat': center_lat,
#         'lon': center_lon,
#         'water_pct': water_pct
#     })

# df_events = pd.DataFrame(event_stats)

# # ==========================================
# # 2. GEOGRAPHIC CLUSTERING & STRATIFICATION
# # ==========================================
# # A. Group into 4 spatial regions
# coords = np.column_stack((df_events['lat'], df_events['lon']))
# kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
# df_events['spatial_cluster'] = kmeans.fit_predict(coords)

# # B. Bin the water percentage
# bins = [-np.inf, 0.10, 0.40, np.inf]
# df_events['water_strata'] = pd.cut(df_events['water_pct'], bins=bins, labels=["low", "med", "high"])

# # C. Create a combined grouping variable (e.g., "Cluster 0 + High Water")
# df_events['stratify_group'] = df_events['spatial_cluster'].astype(str) + "_" + df_events['water_strata'].astype(str)

# # ==========================================
# # 3. CUSTOM STRATIFIED SPLIT (SMALL DATASET SAFE)
# # ==========================================
# TEST_SIZE = 0.10 # Set back to 10% to match your print statement
# total_dataset_tiles = df_events['tile_count'].sum()
# target_test_tiles = int(total_dataset_tiles * TEST_SIZE)

# # 1. Protect singletons (force them into Train so CV doesn't break)
# group_counts = df_events['stratify_group'].value_counts()
# singleton_groups = group_counts[group_counts == 1].index

# # 2. Filter splittable events and randomly shuffle them to remove size bias
# splittable_df = df_events[~df_events['stratify_group'].isin(singleton_groups)].sample(frac=1, random_state=42)

# test_events = []
# current_test_tiles = 0
# groups_available = splittable_df['stratify_group'].unique()

# # 3. Create a queue of randomized events for each stratum
# group_queues = {grp: splittable_df[splittable_df['stratify_group'] == grp]['event_id'].tolist() for grp in groups_available}

# # 4. Round-robin selection to guarantee balanced diversity
# while current_test_tiles < target_test_tiles:
#     added_this_round = False
    
#     for grp in groups_available:
#         if len(group_queues[grp]) > 0:
#             ev_candidate = group_queues[grp][0] # Peek at the next event in this stratum's queue
#             ev_tiles = df_events[df_events['event_id'] == ev_candidate]['tile_count'].values[0]
            
#             # Check if adding this event wildly overshoots our target (allow a tiny 4% margin of error)
#             if current_test_tiles + ev_tiles <= target_test_tiles + (total_dataset_tiles * 0.04):
#                 test_events.append(ev_candidate)
#                 current_test_tiles += ev_tiles
#                 group_queues[grp].pop(0) # Remove the used event from the queue
#                 added_this_round = True
                
#     # If we cycled through all strata groups and couldn't add a single event without massive overshoot, stop
#     if not added_this_round:
#         break

# train_events = df_events[~df_events['event_id'].isin(test_events)]['event_id'].tolist()
# total_test_tiles = df_events[df_events['event_id'].isin(test_events)]['tile_count'].sum()

# # ==========================================
# # 4. MAP MASKS & CREATE FINAL SETS
# # ==========================================
# cv_pool_mask = np.isin(event_arr_filtered, train_events)
# test_mask = np.isin(event_arr_filtered, test_events)

# # CV / Train Pool
# S1_cv_pool = S1_filtered[cv_pool_mask]
# S2_cv_pool = S2_filtered[cv_pool_mask]
# Water_cv_pool = Water_filtered[cv_pool_mask]
# events_cv_pool = event_arr_filtered[cv_pool_mask]

# # Isolated Test Set
# S1_test = S1_filtered[test_mask]
# S2_test = S2_filtered[test_mask]
# Water_test = Water_filtered[test_mask]

# # ==========================================
# # 5. VERIFICATION REPORT
# # ==========================================
# actual_test_pct = (total_test_tiles / total_dataset_tiles) * 100

# print("\n--- Spatially Blocked Stratified Split ---")
# print(f"Total Cloud-Free Tiles : {total_dataset_tiles}")
# print(f"Target Test Size (15%) : {int(total_dataset_tiles * TEST_SIZE)}")
# print(f"Actual Test Size       : {total_test_tiles} tiles ({actual_test_pct:.2f}%)")
# print(f"Test Set Event IDs     : {test_events}")
# print("-" * 42)
# print(f"Train/CV Pool Shapes -> S1: {S1_cv_pool.shape} | Water: {Water_cv_pool.shape}")
# print(f"Test Set Shapes      -> S1: {S1_test.shape} | Water: {Water_test.shape}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# ==========================================
# 1. BUILD GLOBAL EVENT STATS
# ==========================================
unique_events = np.unique(event_arr_filtered)
event_stats = []

for ev in unique_events:
    # Get spatial bounds for the event to find its center
    indices = np.where(event_arr == ev)[0]
    ev_bounds = bounds_arr[indices]
    
    center_lat = np.mean([np.min(ev_bounds[:, 1]), np.max(ev_bounds[:, 3])])
    center_lon = np.mean([np.min(ev_bounds[:, 0]), np.max(ev_bounds[:, 2])])
    
    # Count valid tiles
    tile_count = np.sum(event_arr_filtered == ev)
    
    event_stats.append({
        'event_id': ev, 
        'tile_count': tile_count,
        'lat': center_lat,
        'lon': center_lon
    })

df_events = pd.DataFrame(event_stats)

# ==========================================
# 2. GEOGRAPHIC CLUSTERING
# ==========================================
# Group into 4 regions (representing pseudo-continents)
coords = np.column_stack((df_events['lat'], df_events['lon']))
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_events['spatial_cluster'] = kmeans.fit_predict(coords)

# ==========================================
# 3. PER-REGION SMALL EVENT ACCUMULATION
# ==========================================
test_events = []
total_test_tiles = 0
total_dataset_tiles = df_events['tile_count'].sum()
TEST_SIZE = 0.1
for cluster_id, cluster_df in df_events.groupby('spatial_cluster'):
    cluster_total_tiles = cluster_df['tile_count'].sum()
    
    # Target exactly 15% of the data WITHIN this specific continent/region
    target_cluster_test_tiles = int(cluster_total_tiles * TEST_SIZE)
    
    # Sort from smallest to largest to maximize the number of distinct regions
    cluster_df = cluster_df.sort_values(by='tile_count', ascending=True)
    
    # Protect against regions that only have 1 event (must stay in train so the model learns the region)
    if len(cluster_df) == 1:
        continue
        
    current_cluster_test_tiles = 0
    
    for _, row in cluster_df.iterrows():
        ev_id = row['event_id']
        ev_tiles = row['tile_count']
        
        # Add the event if we haven't hit the target yet
        if current_cluster_test_tiles < target_cluster_test_tiles:
            
            # Guardrail: Prevent a single mid-sized event from accidentally eating 
            # more than 40% of a continent's entire dataset during accumulation
            if (current_cluster_test_tiles + ev_tiles) / cluster_total_tiles <= 0.40:
                test_events.append(ev_id)
                current_cluster_test_tiles += ev_tiles
                
    total_test_tiles += current_cluster_test_tiles

# The rest of the events go to the Training/CV pool
train_events = df_events[~df_events['event_id'].isin(test_events)]['event_id'].tolist()

# ==========================================
# 4. MAP MASKS & CREATE FINAL SETS
# ==========================================
cv_pool_mask = np.isin(event_arr_filtered, train_events)
test_mask = np.isin(event_arr_filtered, test_events)

# CV / Train Pool
S1_cv_pool = S1_filtered[cv_pool_mask]
S2_cv_pool = S2_filtered[cv_pool_mask]
Water_cv_pool = Water_filtered[cv_pool_mask]

# Isolated Test Set
S1_test = S1_filtered[test_mask]
S2_test = S2_filtered[test_mask]
Water_test = Water_filtered[test_mask]

# ==========================================
# 5. VERIFICATION REPORT
# ==========================================
actual_test_pct = (total_test_tiles / total_dataset_tiles) * 100

print("\n--- Geographically Balanced Split ---")
print(f"Total Cloud-Free Tiles : {total_dataset_tiles}")
print(f"Target Test Size (15%) : {int(total_dataset_tiles * TEST_SIZE)}")
print(f"Actual Test Size       : {total_test_tiles} tiles ({actual_test_pct:.2f}%)")
print(f"Test Set Event IDs     : {test_events}")
print("-" * 40)
print(f"Train/CV Pool Shapes -> S1: {S1_cv_pool.shape} | Water: {Water_cv_pool.shape}")
print(f"Test Set Shapes      -> S1: {S1_test.shape} | Water: {Water_test.shape}")

In [ ]:
# Create the event tracking array for the CV pool
events_cv_pool = event_arr_filtered[cv_pool_mask]

# Create a dictionary mapping each event_id to its spatial_cluster (continent)
cluster_mapping = dict(zip(df_events['event_id'], df_events['spatial_cluster']))

# Map those cluster IDs onto every single tile in the CV pool
geo_clusters_cv_pool = np.array([cluster_mapping[ev] for ev in events_cv_pool])

In [ ]:
test_events

In [ ]:
import folium
import numpy as np
from datetime import datetime

# ---------------------------------------------------------
# 1. Map Setup
# ---------------------------------------------------------
mean_lon = np.mean([b[0] + (b[2] - b[0]) / 2 for b in bounds_arr])
mean_lat = np.mean([b[1] + (b[3] - b[1]) / 2 for b in bounds_arr])

m = folium.Map(location=[mean_lat, mean_lon], zoom_start=3, tiles="CartoDB positron")

# ---------------------------------------------------------
# 2. Extract Split Sets from your New Split Logic
# ---------------------------------------------------------
# Convert the newly generated lists to sets for fast lookup
test_events_set = set(test_events)
cv_events_set = set(train_events)

print("Mapping data splits, calculating tile counts, and rendering map...")

# ---------------------------------------------------------
# 3. Group by Event, Assign Colors, Labels, and Plot
# ---------------------------------------------------------
# We use the original event_arr to draw the bounding boxes
unique_events = np.unique(event_arr)

for ev_id in unique_events:
    
    # --- Check Split Status & Assign Color ---
    if ev_id in test_events_set:
        split_status = "Test Set (Holdout)"
        poly_color = "#e31a1c" # Red
    elif ev_id in cv_events_set:
        split_status = "CV / Train Pool"
        poly_color = "#1f78b4" # Blue
    else:
        # If an event was completely filtered out by the cloud mask
        split_status = "Discarded (100% Cloud)"
        poly_color = "#808080" # Grey
        
    # --- Calculate Available Cloud-Free Tiles ---
    # We count occurrences in the NEW filtered array
    tile_count = np.sum(event_arr_filtered == ev_id)
    
    # Skip drawing if the event was completely dropped and you don't want to see it
    # if tile_count == 0: continue 

    # --- Bounding Box Calculation ---
    indices = np.where(event_arr == ev_id)[0]
    ev_bounds = bounds_arr[indices]
    ev_dates = date_arr[indices]
    
    min_lon = np.min(ev_bounds[:, 0])
    min_lat = np.min(ev_bounds[:, 1])
    max_lon = np.max(ev_bounds[:, 2])
    max_lat = np.max(ev_bounds[:, 3])
    
    center_lat = (min_lat + max_lat) / 2
    center_lon = (min_lon + max_lon) / 2
    
    # --- Date Formatting ---
    unique_dates = np.unique(ev_dates)
    human_readable_dates = []
    for d in unique_dates:
        if d == "Unknown":
            human_readable_dates.append(d)
        else:
            try:
                parsed_date = datetime.strptime(str(d), "%Y%m%d")
                human_readable_dates.append(parsed_date.strftime("%B %d, %Y"))
            except ValueError:
                human_readable_dates.append(str(d)) 
                
    dates_str = " | ".join(human_readable_dates)
    # Safely handle id_to_event if it exists in the environment
    try:
        event_name = id_to_event.get(ev_id, f"Event_{ev_id}")
    except NameError:
        event_name = f"Event_{ev_id}"
        
    extent_bounds = [[min_lat, min_lon], [max_lat, max_lon]]
    
    # --- Build Popup Content ---
    popup_text = f"""
        <b>Event Name:</b> {event_name}<br>
        <b>Event ID:</b> {ev_id}<br>
        <b>Split:</b> {split_status}<br>
        <b>Usable Tiles:</b> {tile_count}<br>
        <b>Time(s):</b> {dates_str}
    """
    
    # Draw the rectangle
    folium.Rectangle(
        bounds=extent_bounds,
        color=poly_color,      
        weight=3,
        fill=True,
        fill_color=poly_color, 
        fill_opacity=0.4,     
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)
    
    # --- Add the Multi-Line Text Label (DivIcon) ---
    # Shows Event ID and Tile count stacked
    folium.Marker(
        location=[center_lat, center_lon],
        icon=folium.DivIcon(
            html=f"""
                <div style="
                    font-size: 13px; 
                    font-weight: bold; 
                    color: black; 
                    background-color: rgba(255, 255, 255, 0.7);
                    border: 1px solid black;
                    border-radius: 4px;
                    padding: 2px 4px;
                    text-align: center;
                    width: max-content;
                    transform: translate(-50%, -50%);
                ">
                    ID: {ev_id}<br>
                    {tile_count} tiles
                </div>
            """
        )
    ).add_to(m)

# ---------------------------------------------------------
# 4. Add Split Legend
# ---------------------------------------------------------
legend_html = '''
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 160px; height: 110px; 
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white; padding: 10px; opacity: 0.9;">
    <b>Data Split</b><br>
    <i style="background:#1f78b4; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>CV / Train Pool<br>
    <i style="background:#e31a1c; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>Test Set<br>
    <i style="background:#808080; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>Filtered Out<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Display the map
m

In [ ]:
import folium
import numpy as np
from datetime import datetime

# ---------------------------------------------------------
# 1. Map Setup
# ---------------------------------------------------------
mean_lon = np.mean([b[0] + (b[2] - b[0]) / 2 for b in bounds_arr])
mean_lat = np.mean([b[1] + (b[3] - b[1]) / 2 for b in bounds_arr])

m = folium.Map(location=[mean_lat, mean_lon], zoom_start=3, tiles="CartoDB positron")

# ---------------------------------------------------------
# 2. Extract Split Sets from your Pandas DataFrames
# ---------------------------------------------------------
# Convert to sets/lists for fast lookup
test_events = set(events_test['event_id'].values)
cv_events = set(events_cv_pool_meta['event_id'].values)

print("Mapping data splits, calculating tile counts, and rendering map...")

# ---------------------------------------------------------
# 3. Group by Event, Assign Colors, Labels, and Plot
# ---------------------------------------------------------
# We use the original event_arr to draw the bounding boxes
unique_events = np.unique(event_arr)

for ev_id in unique_events:
    
    # --- Check Split Status & Assign Color ---
    if ev_id in test_events:
        split_status = "Test Set (Holdout)"
        poly_color = "#e31a1c" # Red
    elif ev_id in cv_events:
        split_status = "CV / Train Pool"
        poly_color = "#1f78b4" # Blue
    else:
        # If an event was completely filtered out by the cloud mask
        split_status = "Discarded (100% Cloud)"
        poly_color = "#808080" # Grey
        
    # --- Calculate Available Cloud-Free Tiles ---
    # We count occurrences in the NEW filtered array
    tile_count = np.sum(event_arr_filtered == ev_id)
    
    # Skip drawing if the event was completely dropped and you don't want to see it
    # if tile_count == 0: continue 

    # --- Bounding Box Calculation ---
    indices = np.where(event_arr == ev_id)[0]
    ev_bounds = bounds_arr[indices]
    ev_dates = date_arr[indices]
    
    min_lon = np.min(ev_bounds[:, 0])
    min_lat = np.min(ev_bounds[:, 1])
    max_lon = np.max(ev_bounds[:, 2])
    max_lat = np.max(ev_bounds[:, 3])
    
    center_lat = (min_lat + max_lat) / 2
    center_lon = (min_lon + max_lon) / 2
    
    # --- Date Formatting ---
    unique_dates = np.unique(ev_dates)
    human_readable_dates = []
    for d in unique_dates:
        if d == "Unknown":
            human_readable_dates.append(d)
        else:
            try:
                parsed_date = datetime.strptime(str(d), "%Y%m%d")
                human_readable_dates.append(parsed_date.strftime("%B %d, %Y"))
            except ValueError:
                human_readable_dates.append(str(d)) 
                
    dates_str = " | ".join(human_readable_dates)
    event_name = id_to_event.get(ev_id, f"Event_{ev_id}")
    extent_bounds = [[min_lat, min_lon], [max_lat, max_lon]]
    
    # --- Build Popup Content ---
    popup_text = f"""
        <b>Event Name:</b> {event_name}<br>
        <b>Event ID:</b> {ev_id}<br>
        <b>Split:</b> {split_status}<br>
        <b>Usable Tiles:</b> {tile_count}<br>
        <b>Time(s):</b> {dates_str}
    """
    
    # Draw the rectangle
    folium.Rectangle(
        bounds=extent_bounds,
        color=poly_color,      
        weight=3,
        fill=True,
        fill_color=poly_color, 
        fill_opacity=0.4,     
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)
    
    # --- Add the Multi-Line Text Label (DivIcon) ---
    # Shows Event ID and Tile count stacked
    folium.Marker(
        location=[center_lat, center_lon],
        icon=folium.DivIcon(
            html=f"""
                <div style="
                    font-size: 13px; 
                    font-weight: bold; 
                    color: black; 
                    background-color: rgba(255, 255, 255, 0.7);
                    border: 1px solid black;
                    border-radius: 4px;
                    padding: 2px 4px;
                    text-align: center;
                    width: max-content;
                    transform: translate(-50%, -50%);
                ">
                    ID: {ev_id}<br>
                    {tile_count} tiles
                </div>
            """
        )
    ).add_to(m)

# ---------------------------------------------------------
# 4. Add Split Legend
# ---------------------------------------------------------
legend_html = '''
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 160px; height: 110px; 
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white; padding: 10px; opacity: 0.9;">
    <b>Data Split</b><br>
    <i style="background:#1f78b4; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>CV / Train Pool<br>
    <i style="background:#e31a1c; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>Test Set<br>
    <i style="background:#808080; width:12px; height:12px; float:left; margin-right:8px; margin-top:4px;"></i>Filtered Out<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Display the map
m

In [ ]:
event_water_pct

In [ ]:
S1_cloudy    = S1_tiles[cloud_PCT == 1]
S2_cloudy    = S2_tiles[cloud_PCT == 1]
Water_cloudy = Water_tiles[cloud_PCT == 1]

print(S1_cloudy.shape, S2_cloudy.shape, Water_cloudy.shape)

# Scale S1 & S2
S1_cloudy = scale(S1_cloudy)
S2_cloudy = scale(NDWI(scale(S2_cloudy)))[:, :, :, None]

In [ ]:
Water_filtered.shape,S1_filtered.shape,S2_filtered.shape

In [ ]:
# S2_filtered[S1_filtered[:,:,:,1]==0]

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

def create_dataloader(inputs, targets, batch_size=32, shuffle=True):
    """
    Helper function to convert numpy/pandas data to a PyTorch DataLoader
    """
    # 1. Convert to PyTorch Tensors
    # .float() is used because Neural Networks usually require float32
    # .unsqueeze(1) is added in case inputs are 1D arrays (N,) -> (N, 1)
    input_tensor = torch.tensor(inputs, dtype=torch.float32)
    target_tensor = torch.tensor(targets, dtype=torch.float32)

    # Ensure inputs have at least 2 dimensions [Batch_Size, Features]
    if len(input_tensor.shape) == 1:
        input_tensor = input_tensor.unsqueeze(1)
    if len(target_tensor.shape) == 1:
        target_tensor = target_tensor.unsqueeze(1)

    # 2. Create the Dataset
    dataset = TensorDataset(input_tensor, target_tensor)

    # 3. Create the DataLoader
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return loader

# # --- Scenario 1: S1 is Input, Water is Output ---
# train_loader_s1 = create_dataloader(S1_train, Water_train, batch_size=64, shuffle=True)
# val_loader_s1   = create_dataloader(S1_val, Water_val, batch_size=64, shuffle=False)
# test_loader_s1  = create_dataloader(S1_test, Water_test, batch_size=64, shuffle=False)

# # --- Scenario 2: S2 is Input, Water is Output ---
# train_loader_s2 = create_dataloader(S2_train, Water_train, batch_size=64, shuffle=True)
# val_loader_s2   = create_dataloader(S2_val, Water_val, batch_size=64, shuffle=False)
# test_loader_s2  = create_dataloader(S2_test, Water_test, batch_size=64, shuffle=False)

# # --- Verification ---
# print("--- S1 Loader Check ---")
# for X_batch, y_batch in train_loader_s1:
#     print(f"Input S1 Batch Shape: {X_batch.shape}")
#     print(f"Target Water Batch Shape: {y_batch.shape}")
#     break # Just check the first batch

# print("\n--- S2 Loader Check ---")
# for X_batch, y_batch in train_loader_s2:
#     print(f"Input S2 Batch Shape: {X_batch.shape}")
#     print(f"Target Water Batch Shape: {y_batch.shape}")
#     break

In [ ]:
test_loader_cloudy  = create_dataloader(S1_cloudy, Water_cloudy, batch_size=64, shuffle=False)


In [ ]:
vals = Water_filtered.flatten()

num_zeros = np.sum(vals == 0)
num_ones  = np.sum(vals == 1)
total     = len(vals)

ratio_zeros = num_zeros / total
ratio_ones  = num_ones / total

num_zeros, num_ones, ratio_zeros, ratio_ones


In [ ]:
plt.hist(Water_filtered.flatten())
plt.show()

## Regression S1->S2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation Block.
    Adaptive channel-wise attention mechanism.
    """
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.SiLU(inplace=True), # Modern activation
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ModernDoubleConv(nn.Module):
    """
    (Conv2d => GroupNorm => SiLU) * 2
    + Residual Connection
    + SE Attention
    + Optional Dropout
    """
    def __init__(self, in_channels, out_channels, groups=8, dropout_prob=0.0):
        super().__init__()

        # 1. The main Convolutional path
        self.double_conv = nn.Sequential(
            # Use 'reflect' padding to avoid edge artifacts
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect', bias=False),
            nn.GroupNorm(num_groups=groups, num_channels=out_channels),
            nn.SiLU(inplace=True), # Swish/SiLU is often better than ReLU

            # Spatial Dropout (drops whole feature maps)
            nn.Dropout2d(p=dropout_prob),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect', bias=False),
            nn.GroupNorm(num_groups=groups, num_channels=out_channels),
            nn.SiLU(inplace=True)
        )

        # 2. Squeeze-and-Excitation Block (Channel Attention)
        self.se = SEBlock(out_channels)

        # 3. Residual Projection
        # If input and output channels differ, we need a 1x1 conv to match them for the add operation
        self.identity = nn.Sequential()
        if in_channels != out_channels:
            self.identity = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.GroupNorm(num_groups=groups, num_channels=out_channels)
            )

        # Apply He Normal Initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        identity = self.identity(x)
        out = self.double_conv(x)
        out = self.se(out) # Apply Attention
        return out + identity # Residual connection
import torch
import torch.nn as nn
import torch.nn.functional as F

class ASPP(nn.Module):
    """
    Atrous Spatial Pyramid Pooling.
    Captures context at multiple scales (rates: 1, 6, 12, 18).
    """
    def __init__(self, in_channels, out_channels):
        super(ASPP, self).__init__()
        modules = []
        # 1x1 Conv
        modules.append(nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(inplace=True)
        ))

        # Atrous Convolutions (Dilated)
        rates = [6, 12, 18]
        for rate in rates:
            modules.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=rate, dilation=rate, bias=False),
                nn.GroupNorm(8, out_channels),
                nn.SiLU(inplace=True)
            ))

        # Global Average Pooling
        modules.append(nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(inplace=True)
        ))

        self.convs = nn.ModuleList(modules)

        # Project back to desired dimension
        self.project = nn.Sequential(
            nn.Conv2d(len(modules) * out_channels, out_channels, 1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        res = []
        for conv in self.convs:
            # Handle the global average pooling branch specifically
            if getattr(conv[0], 'output_size', None) is not None:
                res.append(F.interpolate(conv(x), size=x.shape[2:], mode='bilinear', align_corners=False))
            else:
                res.append(conv(x))
        res = torch.cat(res, dim=1)
        return self.project(res)

class AttentionGate(nn.Module):
    """
    Filters the skip connection features (x) using the decoder features (g).
    """
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.GroupNorm(8, F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.GroupNorm(8, F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.GroupNorm(1, 1),
            nn.Sigmoid()
        )

        self.relu = nn.SiLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi
class ScalableUltraUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, base_width=32, groups=8, dropout_rate=0.2):
        super(ScalableUltraUNet, self).__init__()

        # Define filter sizes based on base_width
        # e.g., if base_width=32: [32, 64, 128, 256, 512] (Much lighter than original)
        f = [base_width, base_width*2, base_width*4, base_width*8, base_width*16]

        # --- Encoder ---
        self.inc = ModernDoubleConv(n_channels, f[0], groups=groups)

        # Increase dropout slightly as we go deeper
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(f[0], f[1], groups=groups, dropout_prob=0.1))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(f[1], f[2], groups=groups, dropout_prob=0.2))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(f[2], f[3], groups=groups, dropout_prob=0.3))

        # --- Bottleneck (ASPP) ---
        self.pool4 = nn.MaxPool2d(2)
        # Note: We scale ASPP to match the bottleneck output
        self.aspp = ASPP(f[3], f[4])

        # --- Decoder ---
        # Up 1
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv1 = nn.Conv2d(f[4], f[3], kernel_size=1)
        self.att1 = AttentionGate(F_g=f[3], F_l=f[3], F_int=f[2])
        self.conv_up1 = ModernDoubleConv(f[4], f[3], groups=groups)

        # Up 2
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv2 = nn.Conv2d(f[3], f[2], kernel_size=1)
        self.att2 = AttentionGate(F_g=f[2], F_l=f[2], F_int=f[1])
        self.conv_up2 = ModernDoubleConv(f[3], f[2], groups=groups)

        # Up 3
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv3 = nn.Conv2d(f[2], f[1], kernel_size=1)
        self.att3 = AttentionGate(F_g=f[1], F_l=f[1], F_int=f[0])
        self.conv_up3 = ModernDoubleConv(f[2], f[1], groups=groups)

        # Up 4
        self.up4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv4 = nn.Conv2d(f[1], f[0], kernel_size=1)
        self.att4 = AttentionGate(F_g=f[0], F_l=f[0], F_int=f[0]//2)
        self.conv_up4 = ModernDoubleConv(f[1], f[0], groups=groups)

        # --- Output ---
        self.outc = nn.Conv2d(f[0], n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)       # 64
        x2 = self.down1(x1)    # 128
        x3 = self.down2(x2)    # 256
        x4 = self.down3(x3)    # 512

        # Bottleneck (ASPP)
        x5 = self.pool4(x4)
        x5 = self.aspp(x5)     # 1024

        # Decoder 1
        x = self.up1(x5)
        x = self.up_conv1(x)
        # Handle odd shapes
        if x.shape != x4.shape: x = F.interpolate(x, size=x4.shape[2:], mode='bilinear', align_corners=True)

        # Attention Gate: Filter x4 using current features x
        x4_att = self.att1(g=x, x=x4)

        x = torch.cat([x, x4_att], dim=1)
        x = self.conv_up1(x)

        # Decoder 2
        x = self.up2(x)
        x = self.up_conv2(x)
        if x.shape != x3.shape: x = F.interpolate(x, size=x3.shape[2:], mode='bilinear', align_corners=True)

        x3_att = self.att2(g=x, x=x3)

        x = torch.cat([x, x3_att], dim=1)
        x = self.conv_up2(x)

        # Decoder 3
        x = self.up3(x)
        x = self.up_conv3(x)
        if x.shape != x2.shape: x = F.interpolate(x, size=x2.shape[2:], mode='bilinear', align_corners=True)

        x2_att = self.att3(g=x, x=x2)

        x = torch.cat([x, x2_att], dim=1)
        x = self.conv_up3(x)

        # Decoder 4
        x = self.up4(x)
        x = self.up_conv4(x)
        if x.shape != x1.shape: x = F.interpolate(x, size=x1.shape[2:], mode='bilinear', align_corners=True)

        x1_att = self.att4(g=x, x=x1)

        x = torch.cat([x, x1_att], dim=1)
        x = self.conv_up4(x)

        return self.outc(x)
class UltraUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, groups=8, dropout_rate=0.2):
        super(UltraUNet, self).__init__()

        # --- Encoder ---
        self.inc = ModernDoubleConv(n_channels, 64, groups=groups)

        self.down1 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(64, 128, groups=groups, dropout_prob=0.1))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(128, 256, groups=groups, dropout_prob=0.2))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(256, 512, groups=groups, dropout_prob=0.3))

        # --- Bottleneck (ASPP) ---
        # Note: We use a 1x1 conv to reduce dimensions before ASPP to save memory if needed,
        # but here we go for power.
        self.pool4 = nn.MaxPool2d(2)
        self.aspp = ASPP(512, 1024)

        # --- Decoder ---
        # We need to calculate input channels for DoubleConv carefully after concatenation

        # Up 1
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv1 = nn.Conv2d(1024, 512, kernel_size=1) # Reduce channels after upsample
        self.att1 = AttentionGate(F_g=512, F_l=512, F_int=256)
        self.conv_up1 = ModernDoubleConv(1024, 512, groups=groups)

        # Up 2
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv2 = nn.Conv2d(512, 256, kernel_size=1)
        self.att2 = AttentionGate(F_g=256, F_l=256, F_int=128)
        self.conv_up2 = ModernDoubleConv(512, 256, groups=groups)

        # Up 3
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv3 = nn.Conv2d(256, 128, kernel_size=1)
        self.att3 = AttentionGate(F_g=128, F_l=128, F_int=64)
        self.conv_up3 = ModernDoubleConv(256, 128, groups=groups)

        # Up 4
        self.up4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up_conv4 = nn.Conv2d(128, 64, kernel_size=1)
        self.att4 = AttentionGate(F_g=64, F_l=64, F_int=32)
        self.conv_up4 = ModernDoubleConv(128, 64, groups=groups)

        # --- Output ---
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)       # 64
        x2 = self.down1(x1)    # 128
        x3 = self.down2(x2)    # 256
        x4 = self.down3(x3)    # 512

        # Bottleneck (ASPP)
        x5 = self.pool4(x4)
        x5 = self.aspp(x5)     # 1024

        # Decoder 1
        x = self.up1(x5)
        x = self.up_conv1(x)
        # Handle odd shapes
        if x.shape != x4.shape: x = F.interpolate(x, size=x4.shape[2:], mode='bilinear', align_corners=True)

        # Attention Gate: Filter x4 using current features x
        x4_att = self.att1(g=x, x=x4)

        x = torch.cat([x, x4_att], dim=1)
        x = self.conv_up1(x)

        # Decoder 2
        x = self.up2(x)
        x = self.up_conv2(x)
        if x.shape != x3.shape: x = F.interpolate(x, size=x3.shape[2:], mode='bilinear', align_corners=True)

        x3_att = self.att2(g=x, x=x3)

        x = torch.cat([x, x3_att], dim=1)
        x = self.conv_up2(x)

        # Decoder 3
        x = self.up3(x)
        x = self.up_conv3(x)
        if x.shape != x2.shape: x = F.interpolate(x, size=x2.shape[2:], mode='bilinear', align_corners=True)

        x2_att = self.att3(g=x, x=x2)

        x = torch.cat([x, x2_att], dim=1)
        x = self.conv_up3(x)

        # Decoder 4
        x = self.up4(x)
        x = self.up_conv4(x)
        if x.shape != x1.shape: x = F.interpolate(x, size=x1.shape[2:], mode='bilinear', align_corners=True)

        x1_att = self.att4(g=x, x=x1)

        x = torch.cat([x, x1_att], dim=1)
        x = self.conv_up4(x)

        return self.outc(x)

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# def get_groups(num_channels):
#     for g in range(8, 0, -1):
#         if num_channels % g == 0:
#             return g
#     return 1

# class SEBlock(nn.Module):
#     """
#     Squeeze-and-Excitation Block.
#     Adaptive channel-wise attention mechanism.
#     """
#     def __init__(self, channel, reduction=16):
#         super(SEBlock, self).__init__()
#         self.avg_pool = nn.AdaptiveAvgPool2d(1)
#         self.fc = nn.Sequential(
#             nn.Linear(channel, channel // reduction, bias=False),
#             nn.SiLU(inplace=True), # Modern activation
#             nn.Linear(channel // reduction, channel, bias=False),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         b, c, _, _ = x.size()
#         y = self.avg_pool(x).view(b, c)
#         y = self.fc(y).view(b, c, 1, 1)
#         return x * y.expand_as(x)

# class ModernDoubleConv(nn.Module):
#     """
#     (Conv2d => GroupNorm => SiLU) * 2
#     + Residual Connection
#     + SE Attention
#     + Optional Dropout
#     """
#     def __init__(self, in_channels, out_channels, groups=8, dropout_prob=0.0):
#         super().__init__()

#         # 1. The main Convolutional path
#         self.double_conv = nn.Sequential(
#             # Use 'reflect' padding to avoid edge artifacts
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect', bias=False),
#             nn.GroupNorm(num_groups=groups, num_channels=out_channels),
#             nn.SiLU(inplace=True), # Swish/SiLU is often better than ReLU

#             # Spatial Dropout (drops whole feature maps)
#             nn.Dropout2d(p=dropout_prob),

#             nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect', bias=False),
#             nn.GroupNorm(num_groups=groups, num_channels=out_channels),
#             nn.SiLU(inplace=True)
#         )

#         # 2. Squeeze-and-Excitation Block (Channel Attention)
#         self.se = SEBlock(out_channels)

#         # 3. Residual Projection
#         # If input and output channels differ, we need a 1x1 conv to match them for the add operation
#         self.identity = nn.Sequential()
#         if in_channels != out_channels:
#             self.identity = nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
#                 nn.GroupNorm(num_groups=groups, num_channels=out_channels)
#             )

#         # Apply He Normal Initialization
#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

#     def forward(self, x):
#         identity = self.identity(x)
#         out = self.double_conv(x)
#         out = self.se(out) # Apply Attention
#         return out + identity # Residual connection

# class ASPP(nn.Module):
#     """
#     Atrous Spatial Pyramid Pooling.
#     Captures context at multiple scales (rates: 1, 6, 12, 18).
#     """
#     def __init__(self, in_channels, out_channels):
#         super(ASPP, self).__init__()
#         modules = []
#         # 1x1 Conv
#         modules.append(nn.Sequential(
#             nn.Conv2d(in_channels, out_channels, 1, bias=False),
#             nn.GroupNorm(get_groups(out_channels), out_channels),
#             nn.SiLU(inplace=True)
#         ))

#         # Atrous Convolutions (Dilated)
#         rates = [6, 12, 18]
#         for rate in rates:
#             modules.append(nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, 3, padding=rate, dilation=rate, bias=False),
#                 nn.GroupNorm(get_groups(out_channels), out_channels),
#                 nn.SiLU(inplace=True)
#             ))

#         # Global Average Pooling
#         modules.append(nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Conv2d(in_channels, out_channels, 1, bias=False),
#             nn.GroupNorm(get_groups(out_channels), out_channels),
#             nn.SiLU(inplace=True)
#         ))

#         self.convs = nn.ModuleList(modules)

#         # Project back to desired dimension
#         self.project = nn.Sequential(
#             nn.Conv2d(len(modules) * out_channels, out_channels, 1, bias=False),
#             nn.GroupNorm(get_groups(out_channels), out_channels),
#             nn.SiLU(inplace=True),
#             nn.Dropout(0.5)
#         )

#     def forward(self, x):
#         res = []
#         for conv in self.convs:
#             # Handle the global average pooling branch specifically
#             if getattr(conv[0], 'output_size', None) is not None:
#                 res.append(F.interpolate(conv(x), size=x.shape[2:], mode='bilinear', align_corners=False))
#             else:
#                 res.append(conv(x))
#         res = torch.cat(res, dim=1)
#         return self.project(res)

# class AttentionGate(nn.Module):
#     """
#     Filters the skip connection features (x) using the decoder features (g).
#     """
#     def __init__(self, F_g, F_l, F_int):
#         super(AttentionGate, self).__init__()
#         self.W_g = nn.Sequential(
#             nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.GroupNorm(get_groups(F_int), F_int)
#         )

#         self.W_x = nn.Sequential(
#             nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.GroupNorm(get_groups(F_int), F_int)
#         )

#         self.psi = nn.Sequential(
#             nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.GroupNorm(1, 1),
#             nn.Sigmoid()
#         )

#         self.relu = nn.SiLU(inplace=True)

#     def forward(self, g, x):
#         g1 = self.W_g(g)
#         x1 = self.W_x(x)
#         psi = self.relu(g1 + x1)
#         psi = self.psi(psi)
#         return x * psi

# class ScalableUltraUNet(nn.Module):
#     def __init__(self, n_channels=3, n_classes=1, base_width=32, groups=8, dropout_rate=0.2):
#         super(ScalableUltraUNet, self).__init__()

#         # Define filter sizes based on base_width
#         # e.g., if base_width=32: [32, 64, 128, 256, 512] (Much lighter than original)
#         f = [base_width, base_width*2, base_width*4, base_width*8, base_width*16]

#         # --- Encoder ---
#         self.inc = ModernDoubleConv(n_channels, f[0], groups=groups)

#         # Increase dropout slightly as we go deeper
#         self.down1 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(f[0], f[1], groups=groups, dropout_prob=0.1))
#         self.down2 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(f[1], f[2], groups=groups, dropout_prob=0.2))
#         self.down3 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(f[2], f[3], groups=groups, dropout_prob=0.3))

#         # --- Bottleneck (ASPP) ---
#         self.pool4 = nn.MaxPool2d(2)
#         # Note: We scale ASPP to match the bottleneck output
#         self.aspp = ASPP(f[3], f[4])

#         # --- Decoder ---
#         # Up 1
#         self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv1 = nn.Conv2d(f[4], f[3], kernel_size=1)
#         self.att1 = AttentionGate(F_g=f[3], F_l=f[3], F_int=f[2])
#         self.conv_up1 = ModernDoubleConv(f[4], f[3], groups=groups)

#         # Up 2
#         self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv2 = nn.Conv2d(f[3], f[2], kernel_size=1)
#         self.att2 = AttentionGate(F_g=f[2], F_l=f[2], F_int=f[1])
#         self.conv_up2 = ModernDoubleConv(f[3], f[2], groups=groups)

#         # Up 3
#         self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv3 = nn.Conv2d(f[2], f[1], kernel_size=1)
#         self.att3 = AttentionGate(F_g=f[1], F_l=f[1], F_int=f[0])
#         self.conv_up3 = ModernDoubleConv(f[2], f[1], groups=groups)

#         # Up 4
#         self.up4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv4 = nn.Conv2d(f[1], f[0], kernel_size=1)
#         self.att4 = AttentionGate(F_g=f[0], F_l=f[0], F_int=f[0]//2)
#         self.conv_up4 = ModernDoubleConv(f[1], f[0], groups=groups)

#         # --- Output ---
#         self.outc = nn.Conv2d(f[0], n_classes, kernel_size=1)

#     def forward(self, x):
#         # Encoder
#         x1 = self.inc(x)       # 64
#         x2 = self.down1(x1)    # 128
#         x3 = self.down2(x2)    # 256
#         x4 = self.down3(x3)    # 512

#         # Bottleneck (ASPP)
#         x5 = self.pool4(x4)
#         x5 = self.aspp(x5)     # 1024

#         # Decoder 1
#         x = self.up1(x5)
#         x = self.up_conv1(x)
#         # Handle odd shapes
#         if x.shape != x4.shape: x = F.interpolate(x, size=x4.shape[2:], mode='bilinear', align_corners=True)

#         # Attention Gate: Filter x4 using current features x
#         x4_att = self.att1(g=x, x=x4)

#         x = torch.cat([x, x4_att], dim=1)
#         x = self.conv_up1(x)

#         # Decoder 2
#         x = self.up2(x)
#         x = self.up_conv2(x)
#         if x.shape != x3.shape: x = F.interpolate(x, size=x3.shape[2:], mode='bilinear', align_corners=True)

#         x3_att = self.att2(g=x, x=x3)

#         x = torch.cat([x, x3_att], dim=1)
#         x = self.conv_up2(x)

#         # Decoder 3
#         x = self.up3(x)
#         x = self.up_conv3(x)
#         if x.shape != x2.shape: x = F.interpolate(x, size=x2.shape[2:], mode='bilinear', align_corners=True)

#         x2_att = self.att3(g=x, x=x2)

#         x = torch.cat([x, x2_att], dim=1)
#         x = self.conv_up3(x)

#         # Decoder 4
#         x = self.up4(x)
#         x = self.up_conv4(x)
#         if x.shape != x1.shape: x = F.interpolate(x, size=x1.shape[2:], mode='bilinear', align_corners=True)

#         x1_att = self.att4(g=x, x=x1)

#         x = torch.cat([x, x1_att], dim=1)
#         x = self.conv_up4(x)

#         return self.outc(x)

# class UltraUNet(nn.Module):
#     def __init__(self, n_channels=3, n_classes=1, groups=8, dropout_rate=0.2):
#         super(UltraUNet, self).__init__()

#         # --- Encoder ---
#         self.inc = ModernDoubleConv(n_channels, 64, groups=groups)

#         self.down1 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(64, 128, groups=groups, dropout_prob=0.1))
#         self.down2 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(128, 256, groups=groups, dropout_prob=0.2))
#         self.down3 = nn.Sequential(nn.MaxPool2d(2), ModernDoubleConv(256, 512, groups=groups, dropout_prob=0.3))

#         # --- Bottleneck (ASPP) ---
#         # Note: We use a 1x1 conv to reduce dimensions before ASPP to save memory if needed,
#         # but here we go for power.
#         self.pool4 = nn.MaxPool2d(2)
#         self.aspp = ASPP(512, 1024)

#         # --- Decoder ---
#         # We need to calculate input channels for DoubleConv carefully after concatenation

#         # Up 1
#         self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv1 = nn.Conv2d(1024, 512, kernel_size=1) # Reduce channels after upsample
#         self.att1 = AttentionGate(F_g=512, F_l=512, F_int=256)
#         self.conv_up1 = ModernDoubleConv(1024, 512, groups=groups)

#         # Up 2
#         self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv2 = nn.Conv2d(512, 256, kernel_size=1)
#         self.att2 = AttentionGate(F_g=256, F_l=256, F_int=128)
#         self.conv_up2 = ModernDoubleConv(512, 256, groups=groups)

#         # Up 3
#         self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv3 = nn.Conv2d(256, 128, kernel_size=1)
#         self.att3 = AttentionGate(F_g=128, F_l=128, F_int=64)
#         self.conv_up3 = ModernDoubleConv(256, 128, groups=groups)

#         # Up 4
#         self.up4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#         self.up_conv4 = nn.Conv2d(128, 64, kernel_size=1)
#         self.att4 = AttentionGate(F_g=64, F_l=64, F_int=32)
#         self.conv_up4 = ModernDoubleConv(128, 64, groups=groups)

#         # --- Output ---
#         self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

#     def forward(self, x):
#         # Encoder
#         x1 = self.inc(x)       # 64
#         x2 = self.down1(x1)    # 128
#         x3 = self.down2(x2)    # 256
#         x4 = self.down3(x3)    # 512

#         # Bottleneck (ASPP)
#         x5 = self.pool4(x4)
#         x5 = self.aspp(x5)     # 1024

#         # Decoder 1
#         x = self.up1(x5)
#         x = self.up_conv1(x)
#         # Handle odd shapes
#         if x.shape != x4.shape: x = F.interpolate(x, size=x4.shape[2:], mode='bilinear', align_corners=True)

#         # Attention Gate: Filter x4 using current features x
#         x4_att = self.att1(g=x, x=x4)

#         x = torch.cat([x, x4_att], dim=1)
#         x = self.conv_up1(x)

#         # Decoder 2
#         x = self.up2(x)
#         x = self.up_conv2(x)
#         if x.shape != x3.shape: x = F.interpolate(x, size=x3.shape[2:], mode='bilinear', align_corners=True)

#         x3_att = self.att2(g=x, x=x3)

#         x = torch.cat([x, x3_att], dim=1)
#         x = self.conv_up2(x)

#         # Decoder 3
#         x = self.up3(x)
#         x = self.up_conv3(x)
#         if x.shape != x2.shape: x = F.interpolate(x, size=x2.shape[2:], mode='bilinear', align_corners=True)

#         x2_att = self.att3(g=x, x=x2)

#         x = torch.cat([x, x2_att], dim=1)
#         x = self.conv_up3(x)

#         # Decoder 4
#         x = self.up4(x)
#         x = self.up_conv4(x)
#         if x.shape != x1.shape: x = F.interpolate(x, size=x1.shape[2:], mode='bilinear', align_corners=True)

#         x1_att = self.att4(g=x, x=x1)

#         x = torch.cat([x, x1_att], dim=1)
#         x = self.conv_up4(x)

#         return self.outc(x)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation Block.
    Adaptive channel-wise attention mechanism.
    """
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ModernDoubleConv(nn.Module):
    """
    (Conv2d => GroupNorm => SiLU) * 2
    + Residual Connection
    + SE Attention
    + Optional Dropout
    """
    def __init__(self, in_channels, out_channels, groups=8, dropout_prob=0.0):
        super().__init__()

        self.double_conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode="reflect",
                bias=False
            ),
            nn.GroupNorm(num_groups=groups, num_channels=out_channels),
            nn.SiLU(inplace=True),

            nn.Dropout2d(p=dropout_prob),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode="reflect",
                bias=False
            ),
            nn.GroupNorm(num_groups=groups, num_channels=out_channels),
            nn.SiLU(inplace=True)
        )

        self.se = SEBlock(out_channels)

        self.identity = nn.Sequential()
        if in_channels != out_channels:
            self.identity = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.GroupNorm(num_groups=groups, num_channels=out_channels)
            )

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        identity = self.identity(x)
        out = self.double_conv(x)
        out = self.se(out)
        return out + identity


class PowerfulUNet(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, groups=8, dropout_rate=0.2):
        super(PowerfulUNet, self).__init__()

        self.inc = ModernDoubleConv(n_channels, 64, groups=groups)

        self.down1 = nn.Sequential(
            nn.MaxPool2d(2),
            ModernDoubleConv(64, 128, groups=groups, dropout_prob=dropout_rate * 0.5)
        )

        self.down2 = nn.Sequential(
            nn.MaxPool2d(2),
            ModernDoubleConv(128, 256, groups=groups, dropout_prob=dropout_rate)
        )

        self.down3 = nn.Sequential(
            nn.MaxPool2d(2),
            ModernDoubleConv(256, 512, groups=groups, dropout_prob=dropout_rate)
        )

        self.down4 = nn.Sequential(
            nn.MaxPool2d(2),
            ModernDoubleConv(512, 1024, groups=groups, dropout_prob=dropout_rate * 1.5)
        )

        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = ModernDoubleConv(1024, 512, groups=groups, dropout_prob=dropout_rate)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = ModernDoubleConv(512, 256, groups=groups, dropout_prob=dropout_rate)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = ModernDoubleConv(256, 128, groups=groups, dropout_prob=dropout_rate * 0.5)

        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = ModernDoubleConv(128, 64, groups=groups)

        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        if x.shape != x4.shape:
            x = F.interpolate(x, size=x4.shape[2:], mode="bilinear", align_corners=True)
        x = torch.cat([x, x4], dim=1)
        x = self.conv_up1(x)

        x = self.up2(x)
        if x.shape != x3.shape:
            x = F.interpolate(x, size=x3.shape[2:], mode="bilinear", align_corners=True)
        x = torch.cat([x, x3], dim=1)
        x = self.conv_up2(x)

        x = self.up3(x)
        if x.shape != x2.shape:
            x = F.interpolate(x, size=x2.shape[2:], mode="bilinear", align_corners=True)
        x = torch.cat([x, x2], dim=1)
        x = self.conv_up3(x)

        x = self.up4(x)
        if x.shape != x1.shape:
            x = F.interpolate(x, size=x1.shape[2:], mode="bilinear", align_corners=True)
        x = torch.cat([x, x1], dim=1)
        x = self.conv_up4(x)

        return self.outc(x)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SEBlock(nn.Module):
    """Squeeze-and-Excitation Block (kept your original)"""
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ModernDoubleConv(nn.Module):
    """
    Updated 2026-friendly version:
    - Keeps your structure & names
    - Adds stochastic depth (drop-path)
    - Uses better init pattern
    """
    def __init__(self, in_channels, out_channels, groups=8, dropout_prob=0.0, drop_path_prob=0.0):
        super().__init__()

        self.drop_path = DropPath(drop_path_prob) if drop_path_prob > 0. else nn.Identity()

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, padding_mode="reflect", bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),

            nn.Dropout2d(p=dropout_prob) if dropout_prob > 0 else nn.Identity(),

            nn.Conv2d(out_channels, out_channels, 3, padding=1, padding_mode="reflect", bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
        )

        self.se = SEBlock(out_channels)

        # Shortcut
        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.GroupNorm(groups, out_channels)
            )

        # Better init (helps quite a bit)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='silu')
            elif isinstance(m, nn.GroupNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.double_conv(x)
        out = self.se(out)
        out = self.drop_path(out)
        return out + identity


class DropPath(nn.Module):
    """Stochastic Depth (DropPath) - very useful at deeper levels"""
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()  # binarize
        output = x.div(keep_prob) * random_tensor
        return output


class PowerfulUNet(nn.Module):
    def __init__(
        self,
        n_channels=2,
        n_classes=1,
        base_channels=64,
        groups=8,
        dropout_rate=0.15,
        drop_path_rate=0.10
    ):
        super().__init__()

        ch = [base_channels, base_channels*2, base_channels*4, base_channels*8]

        # Linear drop-path schedule across all blocks
        total_blocks = 8  # 4 down + 4 up
        drop_rates = torch.linspace(0, drop_path_rate, total_blocks).tolist()

        block_idx = 0

        self.inc = ModernDoubleConv(
            n_channels, ch[0], groups=groups, dropout_prob=dropout_rate*0.3,
            drop_path_prob=drop_rates[block_idx]
        )
        block_idx += 1

        self.down1 = nn.Sequential(
            nn.MaxPool2d(2),
            ModernDoubleConv(ch[0], ch[1], groups=groups, dropout_prob=dropout_rate*0.6,
                            drop_path_prob=drop_rates[block_idx])
        )
        block_idx += 1

        self.down2 = nn.Sequential(
            nn.MaxPool2d(2),
            ModernDoubleConv(ch[1], ch[2], groups=groups, dropout_prob=dropout_rate,
                            drop_path_prob=drop_rates[block_idx])
        )
        block_idx += 1

        self.down3 = nn.Sequential(
            nn.MaxPool2d(2),
            ModernDoubleConv(ch[2], ch[3], groups=groups, dropout_prob=dropout_rate*1.1,
                            drop_path_prob=drop_rates[block_idx])
        )
        block_idx += 1

        # Bottleneck
        self.bottleneck = ModernDoubleConv(
            ch[3], ch[3], groups=groups, dropout_prob=dropout_rate*1.3,
            drop_path_prob=drop_rates[block_idx]
        )
        block_idx += 1

        # Up path
        self.up3 = nn.ConvTranspose2d(ch[3], ch[2], kernel_size=2, stride=2)
        self.conv_up3 = ModernDoubleConv(ch[2]*2, ch[2], groups=groups, dropout_prob=dropout_rate*0.6,
                                        drop_path_prob=drop_rates[block_idx])
        block_idx += 1

        self.up2 = nn.ConvTranspose2d(ch[2], ch[1], kernel_size=2, stride=2)
        self.conv_up2 = ModernDoubleConv(ch[1]*2, ch[1], groups=groups, dropout_prob=dropout_rate*0.4,
                                        drop_path_prob=drop_rates[block_idx])
        block_idx += 1

        self.up1 = nn.ConvTranspose2d(ch[1], ch[0], kernel_size=2, stride=2)
        self.conv_up1 = ModernDoubleConv(ch[0]*2, ch[0], groups=groups, dropout_prob=dropout_rate*0.2,
                                        drop_path_prob=drop_rates[block_idx])
        block_idx += 1

        # Final
        self.outc = nn.Conv2d(ch[0], n_classes, kernel_size=1)
        self.final_activation = nn.Sigmoid()   # ← This is the key change for [0,1] target

    def forward(self, x):
        x1 = self.inc(x)          # 64
        x2 = self.down1(x1)       # 128
        x3 = self.down2(x2)       # 256
        x4 = self.down3(x3)       # 512
        x5 = self.bottleneck(x4)  # bottleneck

        d3 = self.up3(x5)
        d3 = torch.cat([d3, x3], dim=1)
        d3 = self.conv_up3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, x2], dim=1)
        d2 = self.conv_up2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, x1], dim=1)
        d1 = self.conv_up1(d1)

        out = self.outc(d1)
        out = self.final_activation(out)      # ← Sigmoid for stable [0,1] output

        return out

In [ ]:
import os

# 2. Define and Create Directory
save_folder = '/content/drive/MyDrive/SAR2NDWI'
os.makedirs(save_folder, exist_ok=True)

print(f"Checkpoints will be saved to: {save_folder}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import os
import glob
import re


def calculate_r2(preds, targets):
    target_mean = torch.mean(targets)
    ss_tot = torch.sum((targets - target_mean) ** 2)
    ss_res = torch.sum((targets - preds) ** 2)
    r2 = 1 - ss_res / (ss_tot + 1e-6)
    return r2.item()

def train_regression_resumable(model, train_loader, test_loader, optimizer, scheduler=None, criterion=None, total_epochs=50, save_dir='.'):
    print(f"--- Setup Regression Training (S1 -> S2) ---")
    print(f"Device: {device}")
    print(f"Save Directory: {save_dir}")

    if criterion is None:
        criterion = nn.MSELoss()

    # --- 1. RESUME LOGIC ---
    start_epoch = 0
    best_val_mse = float('inf')

    # Search for checkpoints
    checkpoint_files = glob.glob(os.path.join(save_dir, "checkpoint_epoch_*.pth"))

    if checkpoint_files:
        # Sort files by epoch number (using regex to extract the number)
        # format: checkpoint_epoch_10.pth
        latest_checkpoint = max(checkpoint_files, key=lambda x: int(re.search(r'epoch_(\d+)', x).group(1)))
        print(f">> Found checkpoint: {latest_checkpoint}")

        # Load the checkpoint
        checkpoint = torch.load(latest_checkpoint, map_location=device)

        # Restore states
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        if scheduler is not None and 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        start_epoch = checkpoint['epoch']
        best_val_mse = checkpoint.get('best_val_mse', float('inf'))

        print(f">> Resuming from Epoch {start_epoch}. Previous Best MSE: {best_val_mse:.5f}")
    else:
        print(">> No checkpoint found. Starting from scratch (Epoch 0).")

    # --- 2. TRAINING LOOP ---
    # Note: range goes from start_epoch to total_epochs
    for epoch in range(start_epoch, total_epochs):
        model.train()
        train_mse = 0
        train_r2 = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs} [Train]")
        for inputs, targets in loop:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs.permute(0, 3, 1, 2)
            targets = targets.permute(0, 3, 1, 2)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_mse += loss.item()
            train_r2 += calculate_r2(outputs, targets)
            loop.set_postfix(loss=loss.item())

        # Validation
        model.eval()
        val_mse = 0
        val_r2 = 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                inputs = inputs.permute(0, 3, 1, 2)
                targets = targets.permute(0, 3, 1, 2)
                outputs = model(inputs)
                val_mse += criterion(outputs, targets).item()
                val_r2 += calculate_r2(outputs, targets)

        avg_train_mse = train_mse / len(train_loader)
        avg_train_r2 = train_r2 / len(train_loader)
        avg_val_mse = val_mse / len(test_loader)
        avg_val_r2 = val_r2 / len(test_loader)

        print(f"Epoch {epoch+1}/{total_epochs}")
        print(f"  Train MSE: {avg_train_mse:.5f} | R2: {avg_train_r2:.4f}")
        print(f"  Test  MSE: {avg_val_mse:.5f} | R2: {avg_val_r2:.4f}")

        # --- 3. SAVING LOGIC ---

        # A. Save Best Model (Overwrites previous best)
        if avg_val_mse < best_val_mse:
            best_val_mse = avg_val_mse
            best_path = os.path.join(save_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)
            print(f"  [SAVED] New Best Model (MSE: {best_val_mse:.5f})")

        # B. Save Checkpoint (Every 10 epochs OR last epoch)
        # We include scheduler and best_val_mse now for perfect resuming
        if (epoch + 1) % 10 == 0 or (epoch + 1) == total_epochs:
            checkpoint_path = os.path.join(save_dir, f"checkpoint_epoch_{epoch+1}.pth")
            torch.save({
                'epoch': epoch + 1, # Save next epoch index
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                'loss': avg_val_mse,     # Current loss
                'best_val_mse': best_val_mse # Carry over the best score
            }, checkpoint_path)
            print(f"  [SAVED] Checkpoint: {checkpoint_path}")

        # Scheduler Step
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(avg_val_mse)
            else:
                scheduler.step()

    return model

### Syntetic Model

In [ ]:
# Assuming S1_train, S1_val, Water_train, etc., are already tensors or numpy arrays
# We convert them to Float Tensors if they aren't already

def prepare_tensor(data):
    # Handles if data is DataFrame, Numpy, or Tensor
    if hasattr(data, 'values'): data = data.values
    t = torch.tensor(data, dtype=torch.float32)
    # Ensure standard dimensions: [N, H, W, C] is what we stored initially
    return t

# # Prepare Inputs
# t_s1_train = prepare_tensor(S1_train)
# t_s1_val   = prepare_tensor(S1_val)

# t_s2_train = prepare_tensor(S2_train)
# t_s2_val   = prepare_tensor(S2_val)

# # Prepare Targets
# t_w_train  = prepare_tensor(Water_train)
# t_w_val    = prepare_tensor(Water_val)

# (Test Loaders should already exist from previous steps, we reuse them)

In [ ]:
import torch
import torch.optim as optim
import os
import numpy as np

# --- 1. The Metric Calculator (Handle 9:1 Imbalance) ---
def get_detailed_metrics(logits, targets, threshold=0.5):
    # Convert raw logits to probabilities -> binary mask
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    # Flatten to vectors [Batch * H * W]
    preds = preds.view(-1)
    targets = targets.view(-1)

    # Confusion Matrix Components
    # TP: Predicted Water (1) & Actual Water (1)
    TP = (preds * targets).sum()

    # FP: Predicted Water (1) & Actual Land (0) -> "Over-painting"
    FP = preds.sum() - TP

    # FN: Predicted Land (0) & Actual Water (1) -> "Missing water"
    FN = targets.sum() - TP

    # TN: Predicted Land (0) & Actual Land (0)
    # Total Pixels - (TP + FP + FN)
    TN = preds.numel() - (TP + FP + FN)

    smooth = 1e-6

    # Metrics
    iou = (TP + smooth) / (TP + FP + FN + smooth)
    f1 = (2 * TP + smooth) / (2 * TP + FP + FN + smooth)
    precision = (TP + smooth) / (TP + FP + smooth)
    recall = (TP + smooth) / (TP + FN + smooth)
    specificity = (TN + smooth) / (TN + FP + smooth)

    return {
        "IoU": iou.item(),
        "F1": f1.item(),
        "Precision": precision.item(),
        "Recall": recall.item(),
        "Specificity": specificity.item()
    }

In [ ]:
def calculate_r2(preds, targets):
    target_mean = torch.mean(targets)
    ss_tot = torch.sum((targets - target_mean) ** 2)
    ss_res = torch.sum((targets - preds) ** 2)
    r2 = 1 - ss_res / (ss_tot + 1e-6)
    return r2.item()

In [ ]:
! pip install torchvision>=0.16

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
from tqdm import tqdm
from torchvision.transforms import v2  # The modern, GPU-optimized API

def train_regression_fold(model, train_loader, val_loader, fold_idx, save_dir, epochs=50, patience=25):
    print(f"  [Fold {fold_idx}] Starting Regression Training (with Torchvision v2)...")

    # --- 1. CONFIGURATION ---
    # Robust Loss: Smoothed L1 is less sensitive to outliers than MSE
    criterion = nn.SmoothL1Loss(beta=0.1)

    # Optimizer: Adjusted for noisy SAR data
    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)

    # Scheduler: More patience to prevent premature LR drops
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
    )

    # --- 2. AUGMENTATION PIPELINE (GPU Optimized) ---
    # The v2 API automatically applies the same random parameters to both
    # the image (SAR) and the target (NDWI) when passed together.
    # --- 2. AUGMENTATION PIPELINE (GPU Optimized & Corrected) ---
    train_transforms = v2.Compose([
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),

        # CORRECTED: Use RandomChoice for discrete 90-degree rotations
        # This randomly selects ONE of the transforms in the list
        v2.RandomChoice([
            v2.Identity(),                                      # 0 degrees
            v2.RandomRotation(degrees=(90, 90), expand=False),  # 90 degrees
            v2.RandomRotation(degrees=(180, 180), expand=False),# 180 degrees
            v2.RandomRotation(degrees=(270, 270), expand=False) # 270 degrees
        ]),
    ])

    # Paths
    best_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_ckpt.pth")

    # State variables
    start_epoch = 0
    best_val_mse = float('inf')
    epochs_no_improve = 0

    # --- 3. RESUME LOGIC ---
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['opt'])
        if 'scheduler' in ckpt: scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_mse = ckpt['best_val_mse']
        epochs_no_improve = ckpt['epochs_no_improve']
        print(f"  [Resume] Fold {fold_idx} Regression at Epoch {start_epoch}")
    else:
        print()

    # --- 4. TRAINING LOOP ---
    for epoch in range(start_epoch, epochs):
        model.train()
        train_mse = 0
        train_r2 = 0

        loop = tqdm(train_loader, desc=f"  Fold {fold_idx} Reg Epoch {epoch+1}/{epochs}", leave=False)

        for inputs, targets in loop:
            # Move to GPU first (Critical for v2 optimization)
            # Permute to (Batch, Channel, Height, Width) if coming from (B, H, W, C)
            inputs = inputs.to(device).permute(0, 3, 1, 2)
            targets = targets.to(device).permute(0, 3, 1, 2)

            # --- APPLY AUGMENTATION ---
            # 'train_transforms' handles the pairing logic automatically.
            # We use no_grad() because we don't want to backprop through the augmentation itself.
            with torch.no_grad():
                inputs, targets = train_transforms(inputs, targets)
            # --------------------------

            optimizer.zero_grad()

            # Forward Pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            # Backward Pass
            loss.backward()

            # Clip Gradients (Stability fix)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Tracking
            train_mse += loss.item()
            train_r2 += calculate_r2(outputs, targets)
            loop.set_postfix(loss=loss.item())

        # --- 5. VALIDATION LOOP (No Augmentation) ---
        model.eval()
        val_mse = 0
        val_r2 = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2)
                targets = targets.to(device).permute(0, 3, 1, 2)

                outputs = model(inputs)

                val_mse += criterion(outputs, targets).item()
                val_r2 += calculate_r2(outputs, targets)

        # Averages
        avg_train_mse = train_mse / len(train_loader)
        avg_train_r2 = train_r2 / len(train_loader)
        avg_val_mse = val_mse / len(val_loader)
        avg_val_r2 = val_r2 / len(val_loader)

        # Print Stats
        print(f"    Epoch {epoch+1} | Train MSE: {avg_train_mse:.5f} R2: {avg_train_r2:.4f} | Val MSE: {avg_val_mse:.5f} R2: {avg_val_r2:.4f}")

        # --- 6. SAVING & EARLY STOPPING ---
        if avg_val_mse < best_val_mse:
            best_val_mse = avg_val_mse
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"    [SAVED] New Best Model (MSE: {best_val_mse:.5f}) to {best_path}")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"  [Early Stopping] Regression converged.")
            break

        # Save Regular Checkpoint
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'opt': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_mse': best_val_mse,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        # Step Scheduler
        scheduler.step(avg_val_mse)

    # Cleanup and Return
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path))
    if os.path.exists(ckpt_path): os.remove(ckpt_path)

    return model

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# # ──────────────────────────────────────────────────────────────
# #                     Modern components
# # ──────────────────────────────────────────────────────────────

# class DropPath(nn.Module):
#     """Drop paths (stochastic depth) per sample"""
#     def __init__(self, drop_prob: float = 0.):
#         super().__init__()
#         self.drop_prob = drop_prob

#     def forward(self, x):
#         if self.drop_prob == 0. or not self.training:
#             return x
#         keep_prob = 1 - self.drop_prob
#         shape = (x.shape[0],) + (1,) * (x.ndim - 1)
#         random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
#         random_tensor.floor_()  # binarize
#         return x.div(keep_prob) * random_tensor


# class ConvNeXtBlock(nn.Module):
#     """ConvNeXt style block – very strong in 2024–2025 segmentation"""
#     def __init__(self, dim, drop_path=0., layer_scale_init=1e-6):
#         super().__init__()
#         self.dwconv = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)  # depthwise
#         self.norm   = nn.LayerNorm(dim, eps=1e-6)
#         self.pwconv1 = nn.Linear(dim, 4 * dim)
#         self.act    = nn.GELU()
#         self.pwconv2 = nn.Linear(4 * dim, dim)
#         self.gamma   = nn.Parameter(layer_scale_init * torch.ones(dim)) if layer_scale_init > 0 else None
#         self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

#     def forward(self, x):
#         shortcut = x
#         x = self.dwconv(x)
#         x = x.permute(0, 2, 3, 1)          # (N, C, H, W) -> (N, H, W, C)
#         x = self.norm(x)
#         x = self.pwconv1(x)
#         x = self.act(x)
#         x = self.pwconv2(x)
#         if self.gamma is not None:
#             x = self.gamma * x
#         x = x.permute(0, 3, 1, 2)          # (N, H, W, C) -> (N, C, H, W)
#         x = shortcut + self.drop_path(x)
#         return x


# class ModernDoubleConv(nn.Module):
#     """ConvNeXt-style + residual + SE + drop-path"""
#     def __init__(self, in_ch, out_ch, drop_path=0., layer_scale=True):
#         super().__init__()
#         self.conv = nn.Sequential(
#             ConvNeXtBlock(in_ch, drop_path=drop_path),
#             ConvNeXtBlock(in_ch, drop_path=drop_path),
#         )
#         if in_ch != out_ch:
#             self.shortcut = nn.Conv2d(in_ch, out_ch, 1, bias=False)
#         else:
#             self.shortcut = nn.Identity()

#     def forward(self, x):
#         return self.shortcut(x) + self.conv(x)


# # ──────────────────────────────────────────────────────────────
# #                           Loss functions
# # ──────────────────────────────────────────────────────────────

# def huber_loss(pred, target, delta=0.1):
#     """Smooth L1 — behaves better than pure L1 on outliers"""
#     diff = (pred - target).abs()
#     quadratic = torch.minimum(diff, torch.full_like(diff, delta))
#     linear = diff - quadratic
#     return 0.5 * quadratic ** 2 + delta * linear


# def asymmetric_loss_regression(pred, target, clip=0.05, focus_factor=2.0):
#     """Emphasize errors when target is near 1 (water)"""
#     diff = (pred - target).abs()
#     weight = 1.0 + focus_factor * torch.clamp(target, 0, 1)
#     # also give a little more weight to very small predictions when target is 1
#     weight += 4.0 * (target > (1 - clip)).float() * (pred < clip).float()
#     return (weight * diff).mean()


# # ──────────────────────────────────────────────────────────────
# #                         Final model suggestion
# # ──────────────────────────────────────────────────────────────

# class StrongWaterUNet(nn.Module):
#     def __init__(self, in_ch=2, base=64, drop_path_rate=0.15):
#         super().__init__()

#         depths = [2, 2, 9, 2]          # similar to ConvNeXt-T/S
#         channels = [base, base*2, base*4, base*8]

#         drop_rates = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

#         # Encoder
#         self.stem = nn.Sequential(
#             nn.Conv2d(in_ch, channels[0], 4, 2, 1, bias=False),
#             nn.BatchNorm2d(channels[0]),
#             nn.GELU()
#         )

#         self.stages = nn.ModuleList()
#         cur = 0
#         for i in range(len(depths)):
#             stage = nn.Sequential(*[
#                 ConvNeXtBlock(channels[i], drop_path=drop_rates[cur+j])
#                 for j in range(depths[i])
#             ])
#             self.stages.append(stage)
#             cur += depths[i]

#             if i < len(depths)-1:
#                 down = nn.Sequential(
#                     nn.BatchNorm2d(channels[i]),
#                     nn.GELU(),
#                     nn.Conv2d(channels[i], channels[i+1], 2, 2)
#                 )
#                 self.stages.append(down)

#         # Decoder (simple U-Net like with skip connections)
#         self.up_conv4 = nn.ConvTranspose2d(channels[3], channels[2], 2, 2)
#         self.dec4     = ModernDoubleConv(channels[2]*2, channels[2])

#         self.up_conv3 = nn.ConvTranspose2d(channels[2], channels[1], 2, 2)
#         self.dec3     = ModernDoubleConv(channels[1]*2, channels[1])

#         self.up_conv2 = nn.ConvTranspose2d(channels[1], channels[0], 2, 2)
#         self.dec2     = ModernDoubleConv(channels[0]*2, channels[0])

#         self.head = nn.Conv2d(channels[0], 1, 1)
#         self.final_act = nn.Sigmoid()

#     def forward(self, x):
#         # Stem + encoder stages
#         x0 = self.stem(x)          # 64ch
#         x1 = self.stages[1](self.stages[0](x0))   # 128ch
#         x2 = self.stages[3](self.stages[2](x1))   # 256ch
#         x3 = self.stages[5](self.stages[4](x2))   # 512ch

#         # Decoder
#         d3 = self.up_conv4(x3)
#         d3 = torch.cat([d3, x2], dim=1)
#         d3 = self.dec4(d3)

#         d2 = self.up_conv3(d3)
#         d2 = torch.cat([d2, x1], dim=1)
#         d2 = self.dec3(d2)

#         d1 = self.up_conv2(d2)
#         d1 = torch.cat([d1, x0], dim=1)
#         d1 = self.dec2(d1)

#         out = self.head(d1)
#         return self.final_act(out)


# # Quick usage example
# # model = StrongWaterUNet(in_ch=2, base=64, drop_path_rate=0.12).cuda()
# # criterion = lambda p,t: 0.7 * F.mse_loss(p,t) + 0.3 * asymmetric_loss_regression(p,t)

In [ ]:
def regression_loss(pred, target, alpha=4.0, beta=0.1, eps=1e-6):
    """
    Asymmetric + Huber style loss good for NDWI/water regression
    """
    diff = pred - target
    abs_diff = diff.abs()

    # Higher penalty when under-predicting water (pred < target)
    weights = torch.ones_like(target) + (alpha * target)
    # Extra focus when target is water but prediction is very low
    weights += 2.5 * ((target > 0.85) & (pred < 0.15))

    huber = torch.where(
        abs_diff < beta,
        0.5 * diff**2,
        beta * (abs_diff - 0.5*beta)
    )

    return (weights * huber).mean()

In [ ]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.auto import tqdm
import os


# You should already have this somewhere
def calculate_r2(pred, target):
    ss_res = ((pred - target) ** 2).sum()
    ss_tot = ((target - target.mean()) ** 2).sum()
    return 1 - ss_res / (ss_tot + 1e-8)


def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=200,           # increased for larger model
    patience=40,          # more patience for bigger capacity models
    device='cuda',
    accumulation_steps=4  # increased → effective batch size ×2 (good for base=64)
):
    print(f"\n► Fold {fold_idx} | Starting Strong Regression Training (base=64)")

    # ─── Enhanced Loss (stronger focus on water + focal modulation) ───────────────
    def loss_fn(pred, target):
        huber = torch.nn.functional.smooth_l1_loss(pred, target, beta=0.05, reduction='none')

        # Strong emphasis on water pixels (target close to 1)
        weight = 1.0 + 8.0 * target.clamp(0, 1)                    # base boost (increased from 5.0)
        weight += 12.0 * (target > 0.80).float() * (pred < 0.30).float()  # harsh penalty for missing water

        # Focal-style modulation: focus on hard examples
        focal_weight = (1 - pred).pow(2) * target + pred.pow(2) * (1 - target)
        focal_weight = focal_weight.clamp(0.5, 5.0)  # prevent extreme values

        return (weight * huber * focal_weight).mean()

    criterion = loss_fn

    # ─── Optimizer & Scheduler ──────────────────────────────────────────────────────
    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3,               # increased for larger model + cosine restarts
        weight_decay=5e-5,     # lower wd — modern trend
        betas=(0.9, 0.99),
        eps=1e-6
    )

    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=15,                # first cycle
        T_mult=2,              # double cycle length each restart
        eta_min=1e-6,
        last_epoch=-1
    )

    # ─── Paths & Resume ─────────────────────────────────────────────────────────────
    best_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_ckpt.pth")

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] Fold {fold_idx} at epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    scaler = torch.amp.GradScaler('cuda')  # mixed precision

    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0.0
        train_r2_sum = 0.0
        steps = 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)

        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            scaler.scale(loss / accumulation_steps).backward()

            # Gradient accumulation
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            train_loss += loss.item() * inputs.size(0)
            train_r2_sum += calculate_r2(outputs, targets) * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.5f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps
        avg_train_r2   = train_r2_sum / steps

        # ─── Validation ────────────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        val_r2_sum = 0.0
        val_steps = 0

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2)
                targets = targets.to(device).permute(0, 3, 1, 2)

                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_r2_sum += calculate_r2(outputs, targets) * inputs.size(0)
                val_steps += inputs.size(0)

        avg_val_loss = val_loss / val_steps
        avg_val_r2   = val_r2_sum / val_steps

        print(f"  Epoch {epoch+1:3d} | "
              f"Train Loss: {avg_train_loss:.5f} R²: {avg_train_r2:.4f} | "
              f"Val Loss: {avg_val_loss:.5f} R²: {avg_val_r2:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        # ─── Save & Early Stopping ─────────────────────────────────────────────────────
        improved = False
        if avg_val_r2 > best_val_r2 + 1e-5:  # small threshold to avoid noise
            best_val_r2 = avg_val_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New best model saved! Val R²: {best_val_r2:.4f} | Loss: {avg_val_loss:.5f}")
            improved = True

        # Checkpoint
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        scheduler.step(epoch + 1)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping triggered after {epoch+1} epochs.")
                break

    # Load best model
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded best model for fold {fold_idx} (Val R²: {best_val_r2:.4f})")

    return model

## Classification Model

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        # Sigmoid to get probabilities
        probs = torch.sigmoid(logits)

        # Flatten
        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)

        return 1 - dice

class ComboLoss(nn.Module):
    def __init__(self, weight_bce=1.0, weight_dice=1.0, pos_weight_value=2.0):
        super(ComboLoss, self).__init__()
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.pos_weight_value = pos_weight_value
        self.dice = DiceLoss()
        # We do NOT initialize BCE here to avoid device mismatch issues
        # We initialize it dynamically in forward() or handle weights manually

    def forward(self, inputs, targets):
        # Dynamically create BCE with correct device based on inputs
        # This handles 'cuda:0', 'cuda:1', or 'cpu' automatically
        pos_weight = torch.tensor([self.pos_weight_value], device=inputs.device)
        bce_func = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        bce = bce_func(inputs, targets)
        dice = self.dice(inputs, targets)

        return (self.weight_bce * bce) + (self.weight_dice * dice)

In [ ]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.auto import tqdm
import os


# You should already have this somewhere
def calculate_r2(pred, target):
    ss_res = ((pred - target) ** 2).sum()
    ss_tot = ((target - target.mean()) ** 2).sum()
    return 1 - ss_res / (ss_tot + 1e-8)


def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=200,           # increased for larger model
    patience=40,          # more patience for bigger capacity models
    device='cuda',
    accumulation_steps=4  # increased → effective batch size ×2 (good for base=64)
):
    print(f"\n► Fold {fold_idx} | Starting Strong Regression Training (base=64)")

    # ─── Enhanced Loss (stronger focus on water + focal modulation) ───────────────
    def loss_fn(pred, target):
        huber = torch.nn.functional.smooth_l1_loss(pred, target, beta=0.05, reduction='none')

        # Strong emphasis on water pixels (target close to 1)
        weight = 1.0 + 8.0 * target.clamp(0, 1)                    # base boost (increased from 5.0)
        weight += 12.0 * (target > 0.80).float() * (pred < 0.30).float()  # harsh penalty for missing water

        # Focal-style modulation: focus on hard examples
        focal_weight = (1 - pred).pow(2) * target + pred.pow(2) * (1 - target)
        focal_weight = focal_weight.clamp(0.5, 5.0)  # prevent extreme values

        return (weight * huber * focal_weight).mean()

    criterion = loss_fn

    # ─── Optimizer & Scheduler ──────────────────────────────────────────────────────
    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3,               # increased for larger model + cosine restarts
        weight_decay=5e-5,     # lower wd — modern trend
        betas=(0.9, 0.99),
        eps=1e-6
    )

    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=15,                # first cycle
        T_mult=2,              # double cycle length each restart
        eta_min=1e-6,
        last_epoch=-1
    )

    # ─── Paths & Resume ─────────────────────────────────────────────────────────────
    best_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_ckpt.pth")

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] Fold {fold_idx} at epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    scaler = torch.amp.GradScaler('cuda')  # mixed precision

    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0.0
        train_r2_sum = 0.0
        steps = 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)

        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            scaler.scale(loss / accumulation_steps).backward()

            # Gradient accumulation
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            train_loss += loss.item() * inputs.size(0)
            train_r2_sum += calculate_r2(outputs, targets) * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.5f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps
        avg_train_r2   = train_r2_sum / steps

        # ─── Validation ────────────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        val_r2_sum = 0.0
        val_steps = 0

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2)
                targets = targets.to(device).permute(0, 3, 1, 2)

                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_r2_sum += calculate_r2(outputs, targets) * inputs.size(0)
                val_steps += inputs.size(0)

        avg_val_loss = val_loss / val_steps
        avg_val_r2   = val_r2_sum / val_steps

        print(f"  Epoch {epoch+1:3d} | "
              f"Train Loss: {avg_train_loss:.5f} R²: {avg_train_r2:.4f} | "
              f"Val Loss: {avg_val_loss:.5f} R²: {avg_val_r2:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        # ─── Save & Early Stopping ─────────────────────────────────────────────────────
        improved = False
        if avg_val_r2 > best_val_r2 + 1e-5:  # small threshold to avoid noise
            best_val_r2 = avg_val_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New best model saved! Val R²: {best_val_r2:.4f} | Loss: {avg_val_loss:.5f}")
            improved = True

        # Checkpoint
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        scheduler.step(epoch + 1)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping triggered after {epoch+1} epochs.")
                break

    # Load best model
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded best model for fold {fold_idx} (Val R²: {best_val_r2:.4f})")

    return model

In [ ]:
import random
def augment_batch(x, y,
                  p_flip=0.5,
                  p_rot=0.5,
                  noise_std=0.01,
                  brightness=0.05,
                  contrast=0.05):
    """
    x, y: (B, C, H, W)
    applies same geometric transforms to x and y
    applies intensity noise only to x
    """

    # ─── Random horizontal flip ───
    if random.random() < p_flip:
        x = torch.flip(x, dims=[3])
        y = torch.flip(y, dims=[3])

    # ─── Random vertical flip ───
    if random.random() < p_flip:
        x = torch.flip(x, dims=[2])
        y = torch.flip(y, dims=[2])

    # ─── Random 90° rotations ───
    if random.random() < p_rot:
        k = random.randint(1, 3)
        x = torch.rot90(x, k, dims=[2, 3])
        y = torch.rot90(y, k, dims=[2, 3])

    # ─── Input-only photometric noise ───
    if noise_std > 0:
        x = x + noise_std * torch.randn_like(x)

    if brightness > 0:
        b = 1.0 + (2 * brightness) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) - 0.5)
        x = x * b

    if contrast > 0:
        mean = x.mean(dim=(2,3), keepdim=True)
        c = 1.0 + (2 * contrast) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) - 0.5)
        x = (x - mean) * c + mean

    return x.clamp(0,1), y


In [ ]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.auto import tqdm
import os


# You should already have this somewhere
def calculate_r2(pred, target):
    ss_res = ((pred - target) ** 2).sum()
    ss_tot = ((target - target.mean()) ** 2).sum()
    return 1 - ss_res / (ss_tot + 1e-8)


def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=200,           # increased for larger model
    patience=50,          # more patience for bigger capacity models
    device='cuda',
    accumulation_steps=4  # increased → effective batch size ×2 (good for base=64)
):
    print(f"\n► Fold {fold_idx} | Starting Strong Regression Training (base=64)")

    # ─── Enhanced Loss (stronger focus on water + focal modulation) ───────────────
    def loss_fn(pred, target):
        loss = torch.mean((pred - target)**2)

        return loss

    criterion = loss_fn

    # ─── Optimizer & Scheduler ──────────────────────────────────────────────────────
    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3,               # increased for larger model + cosine restarts
        weight_decay=5e-5,     # lower wd — modern trend
        betas=(0.9, 0.99),
        eps=1e-6
    )

    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=15,                # first cycle
        T_mult=2,              # double cycle length each restart
        eta_min=1e-6,
        last_epoch=-1
    )

    # ─── Paths & Resume ─────────────────────────────────────────────────────────────
    best_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_ckpt.pth")

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] Fold {fold_idx} at epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    scaler = torch.amp.GradScaler('cuda')  # mixed precision

    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0.0
        train_r2_sum = 0.0
        steps = 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)

        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            # ─── On-the-fly augmentation (train only) ───
            inputs, targets = augment_batch(inputs, targets)


            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            scaler.scale(loss / accumulation_steps).backward()

            # Gradient accumulation
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            train_loss += loss.item() * inputs.size(0)
            train_r2_sum += calculate_r2(outputs, targets) * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.5f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps
        avg_train_r2   = train_r2_sum / steps

        # ─── Validation ────────────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        val_r2_sum = 0.0
        val_steps = 0

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2)
                targets = targets.to(device).permute(0, 3, 1, 2)

                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_r2_sum += calculate_r2(outputs, targets) * inputs.size(0)
                val_steps += inputs.size(0)

        avg_val_loss = val_loss / val_steps
        avg_val_r2   = val_r2_sum / val_steps

        print(f"  Epoch {epoch+1:3d} | "
              f"Train Loss: {avg_train_loss:.5f} R²: {avg_train_r2:.4f} | "
              f"Val Loss: {avg_val_loss:.5f} R²: {avg_val_r2:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        # ─── Save & Early Stopping ─────────────────────────────────────────────────────
        improved = False
        if avg_val_r2 > best_val_r2 + 1e-5:  # small threshold to avoid noise
            best_val_r2 = avg_val_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New best model saved! Val R²: {best_val_r2:.4f} | Loss: {avg_val_loss:.5f}")
            improved = True

        # Checkpoint
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        scheduler.step(epoch + 1)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping triggered after {epoch+1} epochs.")
                break

    # Load best model
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded best model for fold {fold_idx} (Val R²: {best_val_r2:.4f})")

    return model

## New

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.auto import tqdm
import os


# ----------------------------
# R2 metric
# ----------------------------
def calculate_r2(pred, target):
    ss_res = ((pred - target) ** 2).sum()
    ss_tot = ((target - target.mean()) ** 2).sum()
    return 1 - ss_res / (ss_tot + 1e-8)


# ----------------------------
# Combined loss (L1 + focal MSE)
# ----------------------------
def combined_loss(pred, target, alpha=0.6, gamma=0.8):
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    focal_weight = (torch.abs(err) + 1e-6) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    return alpha * l1 + (1 - alpha) * focal_mse


# ----------------------------
# Training function
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=300,
    patience=40,
    device='cuda',
    accumulation_steps=4,
    max_lr=3e-4,         # 🔥 usually more stable than 1e-3 here
    weight_decay=1e-5
):

    print(f"\n► Fold {fold_idx} | Strong regression training started")

    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_ckpt.pth")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=max_lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.99),
        eps=1e-6
    )

    steps_per_epoch = len(train_loader) // accumulation_steps + 1

    scheduler = OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        pct_start=0.15,
        div_factor=10,
        final_div_factor=100
    )

    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0

    # ----------------------------
    # Resume if exists
    # ----------------------------
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    # ----------------------------
    # Training loop
    # ----------------------------
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, train_r2_sum, steps = 0.0, 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            # ─── augmentation (your existing function) ───
            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                loss = combined_loss(outputs, targets)

            scaler.scale(loss / accumulation_steps).backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            train_loss += loss.item() * inputs.size(0)
            train_r2_sum += calculate_r2(outputs.detach(), targets) * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps
        avg_train_r2 = train_r2_sum / steps

        # ----------------------------
        # Validation
        # ----------------------------
        model.eval()
        val_loss, val_r2_sum, val_steps = 0.0, 0.0, 0

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2)
                targets = targets.to(device).permute(0, 3, 1, 2)

                outputs = model(inputs)
                loss = combined_loss(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_r2_sum += calculate_r2(outputs, targets) * inputs.size(0)
                val_steps += inputs.size(0)

        avg_val_loss = val_loss / val_steps
        avg_val_r2 = val_r2_sum / val_steps

        print(f"Epoch {epoch+1:3d} | "
              f"Train Loss {avg_train_loss:.5f} R² {avg_train_r2:.4f} | "
              f"Val Loss {avg_val_loss:.5f} R² {avg_val_r2:.4f}")

        # ----------------------------
        # Save / early stop
        # ----------------------------
        improved = False
        if avg_val_r2 > best_val_r2 + 1e-4:
            best_val_r2 = avg_val_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New BEST model | Val R²: {best_val_r2:.4f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("  Early stopping triggered.")
                break

    # ----------------------------
    # Load best
    # ----------------------------
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val R²: {best_val_r2:.4f}")

    return model


In [ ]:
! pip install transformers

In [ ]:
import torch
import torch.nn as nn
from transformers import SegformerForSemanticSegmentation

class HF_SAR_Regression(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, model_name="nvidia/mit-b0"):
        super().__init__()

        # 1. Load SegFormer from Hugging Face
        # "num_labels=1" tells HF we want a single output channel (Regression)
        # "ignore_mismatched_sizes=True" is needed because we are changing the head
        self.hf_model = SegformerForSemanticSegmentation.from_pretrained(
            model_name,
            num_labels=n_classes,
            ignore_mismatched_sizes=True
        )

        # 2. Input Adapter: SAR (2ch) -> SegFormer (3ch RGB)
        # We use a 1x1 convolution to "learn" how to mix SAR bands into RGB features
        self.input_adapter = nn.Conv2d(n_channels, 3, kernel_size=1)

        # 3. Output Activation (Sigmoid)
        # Matches your PowerfulUNet logic: constrains output to [0, 1] for NDWI
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        # x shape: (Batch, 2, H, W)

        # Step A: Adapt 2 channels -> 3 channels
        x = self.input_adapter(x)

        # Step B: Forward pass through SegFormer
        # Transformers expect strictly named arguments often, or pixel_values
        outputs = self.hf_model(pixel_values=x)

        # Step C: Get logits (Shape is usually H/4, W/4)
        logits = outputs.logits

        # Step D: Upsample back to original size
        # SegFormer is efficient because it processes at 1/4 resolution.
        # We must resize it back up to match the input "water mask" size.
        upsampled_logits = nn.functional.interpolate(
            logits,
            size=x.shape[-2:], # (H, W)
            mode="bilinear",
            align_corners=False
        )

        # Step E: Activation
        out = self.final_activation(upsampled_logits)

        return out

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

class HF_SegFormer(nn.Module):
    def __init__(self, num_channels=3, num_classes=1):
        super(HF_SegFormer, self).__init__()

        # We load a small, fast version (b0) or a powerful one (b3/b5)
        # We ignore the 'mismatched sizes' warning because we are retraining the head anyway
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0", # Change to "nvidia/mit-b3" for more power
            num_labels=num_classes,
            num_channels=3, # Pre-trained models expect 3 input channels
            ignore_mismatched_sizes=True
        )

        # ADAPTER LAYER: If your input isn't 3 channels, we project it to 3
        self.use_adapter = (num_channels != 3)
        if self.use_adapter:
            self.input_adapter = nn.Conv2d(num_channels, 3, kernel_size=1)

    def forward(self, x):
        # 1. Adapt input if necessary
        if self.use_adapter:
            x = self.input_adapter(x)

        # 2. SegFormer requires resizing to 4x divisible (usually handled, but good to know)
        # 3. Forward pass
        # HuggingFace returns a specialized object, we need the 'logits'
        outputs = self.model(pixel_values=x)

        # 4. Upsample logits to original image size
        # SegFormer outputs logits at 1/4th resolution for speed
        upsampled_logits = nn.functional.interpolate(
            outputs.logits,
            size=x.shape[-2:], # Match input Height/Width
            mode="bilinear",
            align_corners=False
        )

        return upsampled_logits

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.auto import tqdm
import os


# ----------------------------
# R2 metric
# ----------------------------
def calculate_r2(pred, target):
    ss_res = ((pred - target) ** 2).sum()
    ss_tot = ((target - target.mean()) ** 2).sum()
    return 1 - ss_res / (ss_tot + 1e-8)


# ----------------------------
# Combined loss (L1 + focal MSE)
# ----------------------------
def combined_loss(pred, target, alpha=0.6, gamma=0.8):
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    focal_weight = (torch.abs(err) + 1e-6) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    return alpha * l1 + (1 - alpha) * focal_mse


# ----------------------------
# Training function
# ----------------------------
import os
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.auto import tqdm

# Import TorchMetrics for mathematically accurate epoch-level logging
from torchmetrics.regression import R2Score, MeanAbsoluteError, MeanAbsolutePercentageError
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ----------------------------
# Combined loss (L1 + focal MSE)
# ----------------------------
# def combined_loss(pred, target, alpha=0.6, gamma=0.8):
#     l1 = torch.mean(torch.abs(pred - target))
#     err = pred - target
#     focal_weight = (torch.abs(err) + 1e-6) ** gamma
#     focal_mse = torch.mean(focal_weight * err**2)
#     return alpha * l1 + (1 - alpha) * focal_mse
def combined_loss(pred, target, alpha=0.5, gamma=2.0):
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    # Higher gamma aggressively punishes large errors (like missing a flooded river entirely)
    focal_weight = (torch.abs(err) + 1e-6) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    return alpha * l1 + (1 - alpha) * focal_mse
# ----------------------------
# Training function
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=300,
    patience=40,
    device='cuda',
    accumulation_steps=4,
    max_lr=3e-4,         
    weight_decay=1e-2
):

    print(f"\n► Fold {fold_idx} | Strong regression training started")

    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_ckpt.pth")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=max_lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.99),
        eps=1e-6
    )

    steps_per_epoch = len(train_loader) // accumulation_steps + 1

    scheduler = OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        pct_start=0.15,
        div_factor=10,
        final_div_factor=100
    )

    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0
    
    # ----------------------------
    # Initialize Global Metrics
    # ----------------------------
    # Using data_range=1.0 assuming your targets are scaled between 0 and 1
    val_r2_metric = R2Score().to(device)
    val_mae_metric = MeanAbsoluteError().to(device)
    val_mape_metric = MeanAbsolutePercentageError().to(device)
    val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    # ----------------------------
    # Resume if exists
    # ----------------------------
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    # ----------------------------
    # Training loop
    # ----------------------------
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, steps = 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            # ─── augmentation (your existing function) ───
            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                loss = combined_loss(outputs, targets)

            scaler.scale(loss / accumulation_steps).backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            train_loss += loss.item() * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps

        # ----------------------------
        # Validation
        # ----------------------------
        model.eval()
        val_loss, val_steps = 0.0, 0

        # Reset metrics at the start of validation
        val_r2_metric.reset()
        val_mae_metric.reset()
        val_mape_metric.reset()
        val_ssim_metric.reset()

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

                outputs = model(inputs)
                loss = combined_loss(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_steps += inputs.size(0)
                
                # Update global metrics (detaching to avoid memory leaks)
                out_det = outputs.detach()
                tar_det = targets.detach()
                
                # R2, MAE, MAPE expect flattened arrays to compare raw pixels
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_mape_metric.update(out_det.flatten(), tar_det.flatten())
                
                # SSIM expects 4D structural tensors: (B, C, H, W)
                val_ssim_metric.update(out_det, tar_det)

        # Compute final global epoch metrics
        avg_val_loss = val_loss / val_steps
        epoch_r2 = val_r2_metric.compute().item()
        epoch_mae = val_mae_metric.compute().item()
        epoch_mape = val_mape_metric.compute().item()
        epoch_ssim = val_ssim_metric.compute().item()

        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        print(f"          ► Metrics | R²: {epoch_r2:.4f} | SSIM: {epoch_ssim:.4f} | MAE: {epoch_mae:.4f} | MAPE: {epoch_mape:.4f}")

        # ----------------------------
        # Save / early stop
        # ----------------------------
        improved = False
        # Saving based on SSIM or R2. Here we stick to R2 as the early-stop target.
        if epoch_r2 > best_val_r2 + 1e-4:
            best_val_r2 = epoch_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New BEST model | Val R²: {best_val_r2:.4f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience and epoch > 200:
                print("  Early stopping triggered.")
                break

    # ----------------------------
    # Load best
    # ----------------------------
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val R²: {best_val_r2:.4f}")

    return model

In [ ]:
import os
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.auto import tqdm

# 🔥 CHANGED: Imported WeightedMeanAbsolutePercentageError to handle near-zero water pixels safely
from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ----------------------------
# Combined loss (L1 + focal MSE)
# ----------------------------
def combined_loss(pred, target, alpha=0.5, gamma=2.0):
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    focal_weight = (torch.abs(err) + 1e-6) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    return alpha * l1 + (1 - alpha) * focal_mse

# ----------------------------
# Training function
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=400,          # 🔥 CHANGED: Reduced from 300 to 100. Your model converges very fast.
    patience=20,         # 🔥 CHANGED: Tighter patience for faster iteration
    device='cuda',
    accumulation_steps=4,
    max_lr=2e-4,         # 🔥 CHANGED: Slightly lower max_lr to stop the model from exploding mid-cycle
    weight_decay=1e-2
):

    print(f"\n► Fold {fold_idx} | Strong regression training started")

    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_ckpt.pth")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=max_lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.99),
        eps=1e-6
    )

    steps_per_epoch = len(train_loader) // accumulation_steps + 1

    scheduler = OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        pct_start=0.20,   # 🔥 CHANGED: Slightly longer warmup to stabilize early feature extraction
        div_factor=10,
        final_div_factor=100
    )

    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0
    
    # ----------------------------
    # Initialize Global Metrics
    # ----------------------------
    val_r2_metric = R2Score().to(device)
    val_mae_metric = MeanAbsoluteError().to(device)
    # 🔥 CHANGED: WMAPE prevents small baseline/edge pixels from creating 500%+ error readouts
    val_wmape_metric = WeightedMeanAbsolutePercentageError().to(device)
    val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    # ----------------------------
    # Resume if exists
    # ----------------------------
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    # ----------------------------
    # Training loop
    # ----------------------------
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, steps = 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            # (Assuming augment_batch is defined elsewhere in your workspace)
            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                loss = combined_loss(outputs, targets)

            scaler.scale(loss / accumulation_steps).backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            train_loss += loss.item() * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps

        # ----------------------------
        # Validation
        # ----------------------------
        model.eval()
        val_loss, val_steps = 0.0, 0

        val_r2_metric.reset()
        val_mae_metric.reset()
        val_wmape_metric.reset()
        val_ssim_metric.reset()

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

                outputs = model(inputs)
                loss = combined_loss(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_steps += inputs.size(0)
                
                out_det = outputs.detach()
                tar_det = targets.detach()
                
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_wmape_metric.update(out_det.flatten(), tar_det.flatten())
                val_ssim_metric.update(out_det, tar_det)

        # Compute final global epoch metrics
        avg_val_loss = val_loss / val_steps
        epoch_r2 = val_r2_metric.compute().item()
        epoch_mae = val_mae_metric.compute().item()
        
        # 🔥 CHANGED: Scaled WMAPE to display as a true percentage (e.g., 4.12%)
        epoch_wmape = val_wmape_metric.compute().item() * 100
        epoch_ssim = val_ssim_metric.compute().item()

        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        print(f"          ► Metrics | R²: {epoch_r2:.4f} | SSIM: {epoch_ssim:.4f} | MAE: {epoch_mae:.4f} | WMAPE: {epoch_wmape:.2f}%")

        # ----------------------------
        # Save / early stop
        # ----------------------------
        improved = False
        if epoch_r2 > best_val_r2 + 1e-4:
            best_val_r2 = epoch_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New BEST model | Val R²: {best_val_r2:.4f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        if not improved:
            epochs_no_improve += 1
            # 🔥 CHANGED: Removed 'and epoch > 200'. If the model breaks and overfits early, let it stop!
            if epochs_no_improve >= patience and epoch > 200:
                print(f"  Early stopping triggered at epoch {epoch+1}.")
                break

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val R²: {best_val_r2:.4f}")

    return model

## CV

In [ ]:
# import random
# import torch

# def augment_batch(x, y,
#                   p_flip=0.5,
#                   p_rot=0.5,
#                   p_cutmix=0.3,      # Added CutMix probability
#                   noise_std=0.01,
#                   brightness=0.05,
#                   contrast=0.05):
#     """
#     x, y: (B, C, H, W)
#     applies same geometric transforms to x and y
#     applies intensity noise only to x
#     """

#     # ─── Random horizontal flip ───
#     if random.random() < p_flip:
#         x = torch.flip(x, dims=[3])
#         y = torch.flip(y, dims=[3])

#     # ─── Random vertical flip ───
#     if random.random() < p_flip:
#         x = torch.flip(x, dims=[2])
#         y = torch.flip(y, dims=[2])

#     # ─── Random 90° rotations ───
#     if random.random() < p_rot:
#         k = random.randint(1, 3)
#         x = torch.rot90(x, k, dims=[2, 3])
#         y = torch.rot90(y, k, dims=[2, 3])

#     # ─── CutMix (Spatial Mixing) ───
#     if p_cutmix > 0 and random.random() < p_cutmix:
#         B, C, H, W = x.shape
        
#         # Shuffle the batch to get our donor images
#         indices = torch.randperm(B, device=x.device)
        
#         # Generate a random box size ratio
#         lam = random.random() 
#         cut_ratio = (1. - lam) ** 0.5
#         cut_w = int(W * cut_ratio)
#         cut_h = int(H * cut_ratio)
        
#         # Choose random center for the box
#         cx = random.randint(0, W)
#         cy = random.randint(0, H)
        
#         # Ensure the box coordinates don't go out of bounds
#         bbx1 = max(0, cx - cut_w // 2)
#         bby1 = max(0, cy - cut_h // 2)
#         bbx2 = min(W, cx + cut_w // 2)
#         bby2 = min(H, cy + cut_h // 2)
        
#         # Swap the regions for both inputs (S1) and targets (S2/Water)
#         x[:, :, bby1:bby2, bbx1:bbx2] = x[indices, :, bby1:bby2, bbx1:bbx2]
#         y[:, :, bby1:bby2, bbx1:bbx2] = y[indices, :, bby1:bby2, bbx1:bbx2]

#     # ─── Input-only photometric noise ───
#     if noise_std > 0:
#         x = x + noise_std * torch.randn_like(x)

#     if brightness > 0:
#         b = 1.0 + (2 * brightness) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) - 0.5)
#         x = x * b

#     if contrast > 0:
#         mean = x.mean(dim=(2,3), keepdim=True)
#         c = 1.0 + (2 * contrast) * (torch.rand(x.size(0), 1, 1, 1, device=x.device) - 0.5)
#         x = (x - mean) * c + mean

#     return x.clamp(0,1), y

In [ ]:
! rm -r "/kaggle/working/parallel_cv_speed_optimized"

In [ ]:
pip install segmentation-models-pytorch

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm

# 🔥 NEW: Swapped out SegFormer for U-Net backbone to get crisp high-res skip connections
import segmentation_models_pytorch as smp

from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ----------------------------
# Model Architecture
# ----------------------------
class HF_SAR_Regression(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, model_name="resnet34"):
        super().__init__()

        # Fallback safeguard: If your outer loop passes an old mit-b0 string, 
        # we route it to a stable ResNet34 backbone to give you spatial skip connections.
        encoder = "resnet34" if "mit" in model_name else model_name

        # 🔥 SUGGESTION IMPLEMENTED: Real U-Net structure. 
        # Natively takes 2 SAR channels and returns full-resolution feature maps.
        self.unet_model = smp.Unet(
            encoder_name=encoder,
            encoder_weights="imagenet",
            in_channels=n_channels,
            classes=n_classes,
            activation=None
        )
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        # x shape: (Batch, 2, H, W)
        logits = self.unet_model(x) # Native Shape: (Batch, 1, H, W)
        out = self.final_activation(logits)
        
        # Return out twice to preserve backward compatibility with your 
        # dual-resolution validation and metrics parsing loops seamlessly.
        return out


# ----------------------------
# Combined loss (L1 + focal MSE)
# ----------------------------
def combined_loss(pred, target, alpha=0.5, gamma=2.0):
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    focal_weight = (torch.abs(err) + 1e-6) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    return alpha * l1 + (1 - alpha) * focal_mse


# ----------------------------
# Training function
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=80,           # Optimized training horizon
    patience=12,         # Tighter checkpoint patience
    device='cuda',
    accumulation_steps=4,
    max_lr=1e-4,         # 🔥 CHANGED: Slightly lower max peak since we drop the upward warmup ramp
    weight_decay=1e-2
):

    print(f"\n► Fold {fold_idx} | Strong regression training started")

    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_ckpt.pth")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=max_lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.99),
        eps=1e-6
    )

    steps_per_epoch = len(train_loader) // accumulation_steps + 1
    total_steps = epochs * steps_per_epoch

    # 🔥 SUGGESTION IMPLEMENTED: Switch to CosineAnnealingLR calculated across total steps.
    # Eliminates the warmup phase completely to lock in good early metric initializations.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_steps,
        eta_min=1e-6
    )

    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0
    
    val_r2_metric = R2Score().to(device)
    val_mae_metric = MeanAbsoluteError().to(device)
    val_wmape_metric = WeightedMeanAbsolutePercentageError().to(device)
    val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    # ----------------------------
    # Training loop
    # ----------------------------
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, steps = 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                native_out, _ = model(inputs)
                
                # Because U-Net outputs at full (H, W), this adaptive pool evaluates to a 
                # clean no-op, preserving 100% native resolution details for the loss function.
                downsampled_targets = F.adaptive_avg_pool2d(targets, output_size=native_out.shape[-2:])
                
                loss = combined_loss(native_out, downsampled_targets)

            scaler.scale(loss / accumulation_steps).backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            train_loss += loss.item() * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps

        # ----------------------------
        # Validation
        # ----------------------------
        model.eval()
        val_loss, val_steps = 0.0, 0

        val_r2_metric.reset()
        val_mae_metric.reset()
        val_wmape_metric.reset()
        val_ssim_metric.reset()

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

                native_out, full_out = model(inputs)
                
                downsampled_targets = F.adaptive_avg_pool2d(targets, output_size=native_out.shape[-2:])
                loss = combined_loss(native_out, downsampled_targets)

                val_loss += loss.item() * inputs.size(0)
                val_steps += inputs.size(0)
                
                out_det = full_out.detach()
                tar_det = targets.detach()
                
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_wmape_metric.update(out_det.flatten(), tar_det.flatten())
                val_ssim_metric.update(out_det, tar_det)

        avg_val_loss = val_loss / val_steps
        epoch_r2 = val_r2_metric.compute().item()
        epoch_mae = val_mae_metric.compute().item()
        epoch_wmape = val_wmape_metric.compute().item() * 100
        epoch_ssim = val_ssim_metric.compute().item()

        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        print(f"          ► Metrics | R²: {epoch_r2:.4f} | SSIM: {epoch_ssim:.4f} | MAE: {epoch_mae:.4f} | WMAPE: {epoch_wmape:.2f}%")

        # ----------------------------
        # Save / early stop
        # ----------------------------
        improved = False
        if epoch_r2 > best_val_r2 + 1e-4:
            best_val_r2 = epoch_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New BEST model | Val R²: {best_val_r2:.4f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping triggered at epoch {epoch+1}.")
                break

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val R²: {best_val_r2:.4f}")

    return model

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm

# Swapped out SegFormer for U-Net backbone to get crisp high-res skip connections
import segmentation_models_pytorch as smp

from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ----------------------------
# Model Architecture
# ----------------------------
class HF_SAR_Regression(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, model_name="resnet50"):
        super().__init__()

        # Fallback safeguard: If your outer loop passes an old mit-b0 string, 
        # we route it to a stable ResNet34 backbone to give you spatial skip connections.
        encoder = "resnet34" if "mit" in model_name else model_name

        # Real U-Net structure. 
        # Natively takes 2 SAR channels and returns full-resolution feature maps.
        self.unet_model = smp.Unet(
            encoder_name=encoder,
            encoder_weights="imagenet",
            in_channels=n_channels,
            classes=n_classes,
            activation=None
        )
        # encoder_weights="noisy-student",
        #     in_channels=n_channels,
        #     classes=n_classes,
        #     activation=None,
        #     decoder_interpolation ="bilinear",
        #     aux_params={
        #         'dropout':0.2
        #     }
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        # x shape: (Batch, 2, H, W)
        logits = self.unet_model(x) # Native Shape: (Batch, 1, H, W)
        out = self.final_activation(logits)
        
        # 🔥 CHANGED: Now returns exactly one output tensor instead of a tuple
        return out


# ----------------------------
# Combined loss (L1 + focal MSE)
# ----------------------------
def combined_loss(pred, target, alpha=0.5, gamma=2.0):
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    focal_weight = (torch.abs(err) + 1e-6) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    return alpha * l1 + (1 - alpha) * focal_mse


# ----------------------------
# Training function
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=80,           # Optimized training horizon
    patience=20,         # Tighter checkpoint patience
    device='cuda',
    accumulation_steps=4,
    max_lr=1e-4,         # Lower max peak since we drop the upward warmup ramp
    weight_decay=1e-2
):

    print(f"\n► Fold {fold_idx} | Strong regression training started")

    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_ckpt.pth")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=max_lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.99),
        eps=1e-6
    )

    steps_per_epoch = len(train_loader) // accumulation_steps + 1
    total_steps = epochs * steps_per_epoch

    # Switch to CosineAnnealingLR calculated across total steps.
    # Eliminates the warmup phase completely to lock in good early metric initializations.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_steps,
        eta_min=1e-6
    )

    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0
    
    val_r2_metric = R2Score().to(device)
    val_mae_metric = MeanAbsoluteError().to(device)
    val_wmape_metric = WeightedMeanAbsolutePercentageError().to(device)
    val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    # ----------------------------
    # Training loop
    # ----------------------------
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, steps = 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                # 🔥 CHANGED: Adjusted unpack signature to catch the single output tensor
                out = model(inputs)
                
                # Because U-Net outputs at full (H, W), this adaptive pool evaluates to a 
                # clean no-op, preserving 100% native resolution details for the loss function.
                # downsampled_targets = F.adaptive_avg_pool2d(targets, output_size=out.shape[-2:])
                
                loss = combined_loss(out, downsampled_targets)

            scaler.scale(loss / accumulation_steps).backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            train_loss += loss.item() * inputs.size(0)
            steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / steps

        # ----------------------------
        # Validation
        # ----------------------------
        model.eval()
        val_loss, val_steps = 0.0, 0

        val_r2_metric.reset()
        val_mae_metric.reset()
        val_wmape_metric.reset()
        val_ssim_metric.reset()

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

                # 🔥 CHANGED: Adjusted validation unpack signature to match the single tensor output
                out = model(inputs)
                
                # downsampled_targets = F.adaptive_avg_pool2d(targets, output_size=out.shape[-2:])
                loss = combined_loss(out, targets)

                val_loss += loss.item() * inputs.size(0)
                val_steps += inputs.size(0)
                
                out_det = out.detach()
                tar_det = targets.detach()
                
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_wmape_metric.update(out_det.flatten(), tar_det.flatten())
                val_ssim_metric.update(out_det, tar_det)

        avg_val_loss = val_loss / val_steps
        epoch_r2 = val_r2_metric.compute().item()
        epoch_mae = val_mae_metric.compute().item()
        epoch_wmape = val_wmape_metric.compute().item() * 100
        epoch_ssim = val_ssim_metric.compute().item()

        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        print(f"          ► Metrics | R²: {epoch_r2:.4f} | SSIM: {epoch_ssim:.4f} | MAE: {epoch_mae:.4f} | WMAPE: {epoch_wmape:.2f}%")

        # ----------------------------
        # Save / early stop
        # ----------------------------
        improved = False
        if epoch_r2 > best_val_r2 + 1e-4:
            best_val_r2 = epoch_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ► New BEST model | Val R²: {best_val_r2:.4f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping triggered at epoch {epoch+1}.")
                break

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val R²: {best_val_r2:.4f}")

    return model

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp

from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ----------------------------
# High-Speed GPU Batch Augmentation Pipeline (Placeholder)
# ----------------------------
def augment_batch(inputs, targets):
    """Applies physics-preserving spatial transformations."""
    if torch.rand(1).item() > 0.5:
        inputs = torch.flip(inputs, dims=[3])
        targets = torch.flip(targets, dims=[3])
    return inputs, targets


# ----------------------------
# NaN-Proofed Model Architecture
# ----------------------------
class HF_SAR_Regression(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, model_name="efficientnet-b1"):
        super().__init__()

        # Fallback safeguard
        encoder = "resnet34" if "mit" in model_name else model_name
        print(f"Loading U-Net Backbone: {encoder}")

        self.unet_model = smp.Unet(
            encoder_name=encoder,
            encoder_weights="imagenet",
            in_channels=n_channels,
            classes=n_classes,
            activation=None
        )
        
        # 🛡️ DEFENSE 1: Instance Normalization shields EfficientNet from raw SAR scale shocks
        self.input_scaler = nn.InstanceNorm2d(n_channels, affine=False)
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        # 🛡️ DEFENSE 2: Clamp outliers to prevent SiLU (Swish) activations from overflowing FP16
        x = self.input_scaler(x)
        x = torch.clamp(x, min=-4.0, max=4.0)
        
        logits = self.unet_model(x) 
        out = self.final_activation(logits)
        return out


# ----------------------------
# Stabilized Combined Loss
# ----------------------------
def combined_loss(pred, target, alpha=0.5, gamma=2.0):
    # Ensure strict tracking boundaries
    pred = torch.clamp(pred, min=1e-6, max=1.0 - 1e-6)
    
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    
    # 🛡️ DEFENSE 3: Clamp focal weight scaling to stop gradient explosions
    focal_weight = (torch.abs(err) + 1e-6).clamp(max=2.0) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    
    loss = alpha * l1 + (1 - alpha) * focal_mse
    
    # Emergency fallback if a NaN somehow manages to slide through
    if torch.isnan(loss):
        return alpha * l1  # Fall back to pure stable L1
    return loss


# ----------------------------
# Train Loop
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=80,           
    patience=20,         
    device='cuda',
    accumulation_steps=4,
    max_lr=1e-4,         
    weight_decay=1e-2
):

    print(f"\n► Fold {fold_idx} | Hardened NaN-Proof training sequence engaged")

    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_ckpt.pth")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=max_lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.99),
        eps=1e-8  # Increased epsilon value for better internal variance protection
    )

    steps_per_epoch = len(train_loader) // accumulation_steps + 1
    total_steps = epochs * steps_per_epoch

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_steps,
        eta_min=1e-6
    )

    # 🛡️ DEFENSE 4: Dynamic Mixed Precision Selection
    # Uses bfloat16 if your GPU natively supports it (highly recommended for EfficientNets)
    has_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    amp_dtype = torch.bfloat16 if has_bf16 else torch.float16
    print(f"  [Precision Mode] Running with auto-selected type: {amp_dtype}")
    
    scaler = torch.amp.GradScaler('cuda', enabled=(amp_dtype == torch.float16))

    start_epoch = 0
    best_val_r2 = -float('inf')
    epochs_no_improve = 0
    
    val_r2_metric = R2Score().to(device)
    val_mae_metric = MeanAbsoluteError().to(device)
    val_wmape_metric = WeightedMeanAbsolutePercentageError().to(device)
    val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_r2 = ckpt.get('best_val_r2', -float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] epoch {start_epoch} | Best Val R²: {best_val_r2:.4f}")

    # ----------------------------
    # Training Loop
    # ----------------------------
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, steps = 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=amp_dtype):
                out = model(inputs)
                # print(out.shape,targets.shape)
                # downsampled_targets = F.adaptive_avg_pool2d(targets, output_size=out.shape[-2:])
                loss = combined_loss(out, targets)

            # Safeguard training scales against accumulation bursts
            loss_scaled = loss / accumulation_steps
            
            if amp_dtype == torch.float16:
                scaler.scale(loss_scaled).backward()
            else:
                loss_scaled.backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                if amp_dtype == torch.float16:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            # Skip tracking updates if an underlying anomaly occurs
            if not torch.isnan(loss):
                train_loss += loss.item() * inputs.size(0)
                steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / (steps + 1e-8)

        # ----------------------------
        # Validation Loop
        # ----------------------------
        model.eval()
        val_loss, val_steps = 0.0, 0

        val_r2_metric.reset()
        val_mae_metric.reset()
        val_wmape_metric.reset()
        val_ssim_metric.reset()

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=amp_dtype):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

                out = model(inputs)
                # downsampled_targets = F.adaptive_avg_pool2d(targets, output_size=out.shape[-2:])
                # print(inputs.shape,out.shape,targets.shape)
                loss = combined_loss(out, targets)

                val_loss += loss.item() * inputs.size(0)
                val_steps += inputs.size(0)
                
                out_det = out.detach()
                # 🛡️ DEFENSE 5: Fixed metric matching target resolution shape mismatch bug
                tar_det = targets.detach()
                
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_wmape_metric.update(out_det.flatten(), tar_det.flatten())
                val_ssim_metric.update(out_det, tar_det)

        avg_val_loss = val_loss / (val_steps + 1e-8)
        epoch_r2 = val_r2_metric.compute().item()
        epoch_mae = val_mae_metric.compute().item()
        epoch_wmape = val_wmape_metric.compute().item() * 100
        epoch_ssim = val_ssim_metric.compute().item()

        # Final check if metrics encounter calculation faults
        if torch.isnan(torch.tensor(epoch_r2)):
            epoch_r2 = -1.0

        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        print(f"           ► Metrics | R²: {epoch_r2:.4f} | SSIM: {epoch_ssim:.4f} | MAE: {epoch_mae:.4f} | WMAPE: {epoch_wmape:.2f}%")

        # ----------------------------
        # Save / Early Stop Execution
        # ----------------------------
        improved = False
        if epoch_r2 > best_val_r2 + 1e-4:
            best_val_r2 = epoch_r2
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"   ► New BEST model | Val R²: {best_val_r2:.4f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_r2': best_val_r2,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping triggered at epoch {epoch+1}.")
                break

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val R²: {best_val_r2:.4f}")

    return model

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp

from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ----------------------------
# 🛡️ REGULARIZATION 1: Physics-Based SAR Augmentation Pipeline
# ----------------------------
def augment_batch(inputs, targets):
    """
    Applies spatial and multiplicative noise transformations to regularize 
    the network against memorizing small dataset features.
    """
    # Spatial Flips
    if torch.rand(1).item() > 0.5:
        inputs = torch.flip(inputs, dims=[3])
        targets = torch.flip(targets, dims=[3])
    if torch.rand(1).item() > 0.5:
        inputs = torch.flip(inputs, dims=[2])
        targets = torch.flip(targets, dims=[2])
        
    # Spatial Rotations (90-degree increments to preserve grid alignment)
    if torch.rand(1).item() > 0.5:
        k = torch.randint(1, 4, (1,)).item()
        inputs = torch.rot90(inputs, k, dims=[2, 3])
        targets = torch.rot90(targets, k, dims=[2, 3])

    # Multiplicative Speckle Noise (Simulates true SAR sensor variation)
    if torch.rand(1).item() > 0.3 and torch.int16:
        # Generate Gamma-like noise centered around 1.0
        speckle = 1.0 + torch.randn_like(inputs) * 0.05
        inputs = inputs * speckle

    return inputs, targets


# ----------------------------
# Regularized Model Architecture
# ----------------------------
class HF_SAR_Regression(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, model_name="efficientnet-b1"):
        super().__init__()

        encoder = "resnet34" if "mit" in model_name else model_name
        print(f"Loading Fully Regularized U-Net Backbone: {encoder}")

        # 🛡️ REGULARIZATION 2: Architectural Dropout Modifications
        # We inject dropout directly into the Decoder blocks and utilize 'aux_params'
        # to enforce a structural feature drop across intermediate upsampling maps.
        self.unet_model = smp.Unet(
            encoder_name=encoder,
            encoder_weights="imagenet",
            in_channels=n_channels,
            classes=n_classes,
            activation=None,
            decoder_channels=(256, 128, 64, 32, 16),
            decoder_attention_type="scse",
            aux_params={
                'pooling': 'avg',
                'dropout': 0.2, # Adds regularizing dropout layers across deep layers
                'classes': n_classes
            } if False else None # Set to true if auxiliary classification task is used
        )
        
        # Explicit Spatial Dropout on the bottle-neck logit generation
        self.spatial_dropout = nn.Dropout2d(p=0.2)
        
        self.input_scaler = nn.InstanceNorm2d(n_channels, affine=False)
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        x = self.input_scaler(x)
        x = torch.clamp(x, min=-4.0, max=4.0)
        
        logits = self.unet_model(x) 
        
        # Apply structural 2D spatial drop right before regression activation
        if self.training:
            logits = self.spatial_dropout(logits)
            
        out = self.final_activation(logits)
        return out


# ----------------------------
# Stabilized Combined Loss with Target Regularization
# ----------------------------
from torchmetrics.functional.image import structural_similarity_index_measure
def combined_loss(pred, target, alpha=0.5, gamma=2.0):
    pred = torch.clamp(pred, min=1e-6, max=1.0 - 1e-6)
    
    smooth_eps = 0.005
    target = target * (1.0 - 2 * smooth_eps) + smooth_eps
    
    # 1. Intensity Losses
    l1 = torch.mean(torch.abs(pred - target))
    err = pred - target
    focal_weight = (torch.abs(err) + 1e-6).clamp(max=2.0) ** gamma
    focal_mse = torch.mean(focal_weight * err**2)
    
    # 2. 🛡️ FIX: Cast to float32 to prevent FP16 variance underflow/nan spikes
    pred_f32 = pred.float()
    target_f32 = target.float()
    
    ssim_val = structural_similarity_index_measure(pred_f32, target_f32, data_range=1.0)
    ssim_loss = 1.0 - ssim_val
    
    loss = alpha * l1 + (1 - alpha) * (0.5 * focal_mse + 0.5 * ssim_loss)
    
    if torch.isnan(loss):
        return alpha * l1 + (1 - alpha) * focal_mse
    return loss
# ----------------------------
# Train Loop
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    epochs=80,           
    patience=20,         
    device='cuda',
    accumulation_steps=4,
    max_lr=1e-4,         
    weight_decay=5e-2  
):

    print(f"\n► Fold {fold_idx} | Tracking Validation Loss for Checkpointing")

    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_ckpt.pth")

    # Separate parameter groups to avoid regularizing biases or normalization layers
    decay_params = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if len(param.shape) == 1 or name.endswith(".bias"):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optimizer_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0}
    ]

    optimizer = optim.AdamW(
        optimizer_groups,
        lr=max_lr,
        betas=(0.9, 0.98),  
        eps=1e-8  
    )

    steps_per_epoch = len(train_loader) // accumulation_steps + 1
    total_steps = epochs * steps_per_epoch

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_steps,
        eta_min=1e-6
    )

    has_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    amp_dtype = torch.bfloat16 if has_bf16 else torch.float16
    print(f"  [Precision Mode] Running with auto-selected type: {amp_dtype}")
    
    scaler = torch.amp.GradScaler('cuda', enabled=(amp_dtype == torch.float16))

    start_epoch = 0
    # 🔄 CHANGED: Tracking minimum validation loss instead of maximum R²
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    val_r2_metric = R2Score().to(device)
    val_mae_metric = MeanAbsoluteError().to(device)
    val_wmape_metric = WeightedMeanAbsolutePercentageError().to(device)
    val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        epochs_no_improve = ckpt.get('epochs_no_improve', 0)
        print(f"  [Resumed] epoch {start_epoch} | Best Val Loss: {best_val_loss:.5f}")

    # ----------------------------
    # Training Loop
    # ----------------------------
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, steps = 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=amp_dtype):
                out = model(inputs)
                loss = combined_loss(out, targets)

            loss_scaled = loss / accumulation_steps
            
            if amp_dtype == torch.float16:
                scaler.scale(loss_scaled).backward()
            else:
                loss_scaled.backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                if amp_dtype == torch.float16:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    optimizer.step()
                    
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if not torch.isnan(loss):
                train_loss += loss.item() * inputs.size(0)
                steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / (steps + 1e-8)

        # ----------------------------
        # Validation Loop
        # ----------------------------
        model.eval()
        val_loss, val_steps = 0.0, 0

        val_r2_metric.reset()
        val_mae_metric.reset()
        val_wmape_metric.reset()
        val_ssim_metric.reset()

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=amp_dtype):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

                out = model(inputs)
                loss = combined_loss(out, targets)

                val_loss += loss.item() * inputs.size(0)
                val_steps += inputs.size(0)
                
                out_det = out.detach()
                tar_det = targets.detach()
                
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_wmape_metric.update(out_det.flatten(), tar_det.flatten())
                val_ssim_metric.update(out_det, tar_det)

        avg_val_loss = val_loss / (val_steps + 1e-8)
        epoch_r2 = val_r2_metric.compute().item()
        epoch_mae = val_mae_metric.compute().item()
        epoch_wmape = val_wmape_metric.compute().item() * 100
        epoch_ssim = val_ssim_metric.compute().item()

        if torch.isnan(torch.tensor(epoch_r2)):
            epoch_r2 = -1.0

        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        print(f"            ► Metrics | SSIM: {epoch_ssim:.4f} | R²: {epoch_r2:.4f} | MAE: {epoch_mae:.4f} | WMAPE: {epoch_wmape:.2f}%")

        # ----------------------------
        # Save Execution Logic
        # ----------------------------
        improved = False
        # 🔄 CHANGED: Look for a lower validation loss value
        if avg_val_loss < best_val_loss - 1e-5:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"    ► New BEST model based on Loss | Val Loss: {best_val_loss:.5f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience and epoch > 100:
                print(f"  Early stopping triggered at epoch {epoch+1}.")
                break

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val Loss: {best_val_loss:.5f}")

    return model

In [ ]:
S1_test.shape

In [ ]:
! rm -r "/kaggle/working/parallel_cv_speed_optimized"

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

class HF_SegFormer(nn.Module):
    def __init__(self, num_channels=3, num_classes=1, model_name="se_resnext50_32x4d"):
        super(HF_SegFormer, self).__init__()

        print(f"Initializing U-Net with robust, lightweight backbone: {model_name}")

        # Swapped backbone to smp.Unet with ImageNet pretraining.
        # EfficientNet-B1 is lightweight (~7M parameters), making it highly robust
        # against overfitting on small datasets while maintaining high accuracy.
        self.model = smp.Unet(
            encoder_name=model_name,
            encoder_weights="imagenet",
            in_channels=3,         # Pre-trained backbone expects 3 channels after adapter
            classes=num_classes,
            activation=None ,       # Returns raw logits to match original behavior
            decoder_attention_type="scse",
            aux_params={
                'pooling': 'avg',
                'dropout': 0.2, # Adds regularizing dropout layers across deep layers
                'classes': n_classes
            } if False else None # Set to true if auxiliary classification task is used
        
        )

        # ADAPTER LAYER: If your input isn't 3 channels, we project it to 3
        self.input_adapter = nn.Conv2d(num_channels, 3, kernel_size=1)

    def forward(self, x):
        # 1. Adapt input if necessary
        x = self.input_adapter(x)

        # 2. Forward pass through U-Net 
        # (Natively utilizes skip connections to capture fine-grained spatial details)
        outputs = self.model(x)

        # 3. Ensure outputs match the original image size exactly
        upsampled_logits = nn.functional.interpolate(
            outputs,
            size=x.shape[-2:], # Match input Height/Width
            mode="bilinear",
            align_corners=False
        )

        return upsampled_logits

In [ ]:
! rm -r "/kaggle/working/parallel_cv_speed_optimized"

## CV FOLD MAIN

In [ ]:
from torchmetrics.functional.image import structural_similarity_index_measure

def combined_loss(pred, target, alpha=0.5, gamma=2.0):
    # Prevent extreme values
    pred = torch.clamp(pred, min=1e-6, max=1.0 - 1e-6)
    
    # 🛡️ REGULARIZATION 3: Regression Label Smoothing
    smooth_eps = 0.005
    target = target * (1.0 - 2 * smooth_eps) + smooth_eps
    
    # 1. Stable Pixel Intensity Losses
    l1 = torch.mean(torch.abs(pred - target))
    mse = torch.mean((pred - target) ** 2)  # 🔄 CHANGED: Replaced unstable Focal MSE with standard MSE
    
    # 2. Structural Loss Component (Forced to Float32 for precision stability)
    pred_f32 = pred.float()
    target_f32 = target.float()
    ssim_val = structural_similarity_index_measure(pred_f32, target_f32, data_range=1.0)
    ssim_loss = 1.0 - ssim_val
    
    # 3. Balanced multi-objective blending
    # alpha controls L1, (1-alpha) splits evenly between pixel-variance (MSE) and structure (SSIM)
    loss = alpha * l1 + (1 - alpha) * (0.5 * mse + 0.5 * ssim_loss)
    
    if torch.isnan(loss):
        return alpha * l1 + (1 - alpha) * mse  # Safe fallback
    return loss

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

class HF_SAR_Regression(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, model_name="efficientnet-b1", dropout_p=0.2):
        super().__init__()

        encoder = "resnet34" if "mit" in model_name else model_name
        print(f"Loading Secure U-Net Backbone: {encoder}")

        self.unet_model = smp.Unet(
            encoder_name=encoder,
            encoder_weights="imagenet",
            in_channels=n_channels,
            classes=n_classes,
            activation=None,
            decoder_channels=(256, 128, 64, 32, 16),
            decoder_attention_type="scse"
        )
        
        # 🛡️ Safe bottleneck regularization
        self.bottleneck_dropout = nn.Dropout2d(p=dropout_p)
        
        self.input_scaler = nn.InstanceNorm2d(n_channels, affine=False)
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        x = self.input_scaler(x)
        x = torch.clamp(x, min=-4.0, max=4.0)
        
        # 1. Extract features from encoder pyramid
        features = self.unet_model.encoder(x)
        
        # 2. Apply structural 2D spatial drop to the DEEPEST layer 
        if self.training:
            features_list = list(features)
            features_list[-1] = self.bottleneck_dropout(features_list[-1])
            features = tuple(features_list)
            
        # 3. Complete standard U-Net decoding (Pass as a single tuple block 🚨)
        decoder_output = self.unet_model.decoder(features) 
        logits = self.unet_model.segmentation_head(decoder_output)
        
        out = self.final_activation(logits)
        return out

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp

from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.functional.image import structural_similarity_index_measure

# ----------------------------
# 🛡️ REGULARIZATION: Pure Physics & Spatial Integrity Pipeline
# ----------------------------
def augment_batch(inputs, targets):
    """
    Applies spatial transforms and true SAR speckle noise.
    CutMix is explicitly disabled here to prevent SSIM boundary penalties.
    """
    # 1. Spatial Flips
    if torch.rand(1).item() > 0.5:
        inputs = torch.flip(inputs, dims=[3])
        targets = torch.flip(targets, dims=[3])
    if torch.rand(1).item() > 0.5:
        inputs = torch.flip(inputs, dims=[2])
        targets = torch.flip(targets, dims=[2])
        
    # 2. Spatial Rotations (90-degree increments to preserve grid alignment)
    if torch.rand(1).item() > 0.5:
        k = torch.randint(1, 4, (1,)).item()
        inputs = torch.rot90(inputs, k, dims=[2, 3])
        targets = torch.rot90(targets, k, dims=[2, 3])

    # 3. Multiplicative Speckle Noise (Simulates natural SAR variance safely)
    if torch.rand(1).item() > 0.3:
        speckle = 1.0 + torch.randn_like(inputs) * 0.05
        inputs = inputs * speckle

    return inputs, targets


# ----------------------------
# 🎯 STABILIZED GENERATOR LOSS FORMULATION
# ----------------------------
def combined_loss(pred, target, w_ssim=0.6, w_huber=0.4):
    """
    w_ssim: 0.6 (Maintains structural integrity and edge sharpness)
    w_huber: 0.4 (Anchors color/brightness while resisting SAR outliers)
    """
    pred = torch.clamp(pred, min=1e-6, max=1.0 - 1e-6)
    
    smooth_eps = 0.005
    target = target * (1.0 - 2 * smooth_eps) + smooth_eps
    
    # 1. Huber Loss (Smooth L1) - Outlier resistant
    huber = F.smooth_l1_loss(pred, target, beta=0.1)
    
    # 2. SSIM Structural Loss
    pred_f32, target_f32 = pred.float(), target.float()
    ssim_val = structural_similarity_index_measure(pred_f32, target_f32, data_range=1.0)
    ssim_loss = 1.0 - ssim_val
    
    loss = (w_huber * huber) + (w_ssim * ssim_loss)
    
    if torch.isnan(loss):
        return huber
        
    return loss


# ----------------------------
# 🚀 OPTIMIZED TRAINING LOOP (ANTI-PLATEAU ENGAGED)
# ----------------------------
def train_regression_fold(
    model,
    train_loader,
    val_loader,
    fold_idx,
    save_dir,
    wandb,
    epochs=150,           
    patience=25,         
    device='cuda',
    accumulation_steps=4,
    max_lr=2e-4,          # Elevated peak rate to cross local minima barriers safely
    weight_decay=1e-4
):
    print(f"\n► Fold {fold_idx} | Executing Advanced Non-Stalling Regression Run")
    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_best.pth")
    ckpt_path = os.path.join(save_dir, f"fold_{fold_idx}_reg_ckpt.pth")

    # Decay Parameter Isolation Logic
    decay_params = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if len(param.shape) == 1 or name.endswith(".bias"):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optimizer_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0}
    ]

    # Reverted to standard AdamW betas (0.9, 0.999) to keep gradient tracking active
    optimizer = optim.AdamW(optimizer_groups, lr=max_lr / 10.0, betas=(0.9, 0.999), eps=1e-8)

    steps_per_epoch = len(train_loader) // accumulation_steps + 1
    total_steps = epochs * steps_per_epoch

    # Switched to OneCycleLR to provide explicit warmup phase at start of training
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        total_steps=total_steps,
        pct_start=0.15,        # Spend first 15% of steps ramping up learning rate
        anneal_strategy='cos',
        div_factor=10.0,
        final_div_factor=100.0
    )

    has_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    amp_dtype = torch.bfloat16 if has_bf16 else torch.float16
    print(f"  [Precision Mode] Running with auto-selected type: {amp_dtype}")
    
    scaler = torch.amp.GradScaler('cuda', enabled=(amp_dtype == torch.float16))

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    val_r2_metric = R2Score().to(device)
    val_mae_metric = MeanAbsoluteError().to(device)
    val_wmape_metric = WeightedMeanAbsolutePercentageError().to(device)
    val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optimizer'])
            scheduler.load_state_dict(ckpt['scheduler'])
            start_epoch = ckpt['epoch'] + 1
            best_val_loss = ckpt.get('best_val_loss', float('inf'))
            epochs_no_improve = ckpt.get('epochs_no_improve', 0)
            print(f"  [Resumed] epoch {start_epoch} | Best Val Loss: {best_val_loss:.5f}")
        except Exception as e:
            print(f"  [Warning] Failed to safely load check-pointing artifact. Starting fresh: {e}")

    # Main Epoch Sequence
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss, steps = 0.0, 0

        loop = tqdm(train_loader, desc=f"Fold {fold_idx} | Epoch {epoch+1}/{epochs}", leave=False)
        optimizer.zero_grad(set_to_none=True)

        for i, (inputs, targets) in enumerate(loop):
            inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
            targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

            # Execute spatial transformations (No CutMix)
            inputs, targets = augment_batch(inputs, targets)

            with torch.amp.autocast(device_type='cuda', dtype=amp_dtype):
                out = model(inputs)
                loss = combined_loss(out, targets)

            loss_scaled = loss / accumulation_steps
            
            if amp_dtype == torch.float16:
                scaler.scale(loss_scaled).backward()
            else:
                loss_scaled.backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                if amp_dtype == torch.float16:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Relaxed to 1.0 to encourage optimization jumps
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Relaxed to 1.0
                    optimizer.step()
                    
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if not torch.isnan(loss):
                train_loss += loss.item() * inputs.size(0)
                steps += inputs.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        avg_train_loss = train_loss / (steps + 1e-8)

        # Validation Sequence
        model.eval()
        val_loss, val_steps = 0.0, 0

        val_r2_metric.reset()
        val_mae_metric.reset()
        val_wmape_metric.reset()
        val_ssim_metric.reset()

        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=amp_dtype):
            for inputs, targets in val_loader:
                inputs = inputs.to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.to(device).permute(0, 3, 1, 2).contiguous()

                out = model(inputs)
                loss = combined_loss(out, targets)

                val_loss += loss.item() * inputs.size(0)
                val_steps += inputs.size(0)
                
                out_det = out.detach()
                tar_det = targets.detach()
                
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_wmape_metric.update(out_det.flatten(), tar_det.flatten())
                val_ssim_metric.update(out_det, tar_det)

        avg_val_loss = val_loss / (val_steps + 1e-8)
        epoch_r2 = val_r2_metric.compute().item()
        epoch_mae = val_mae_metric.compute().item()
        epoch_wmape = val_wmape_metric.compute().item() * 100
        epoch_ssim = val_ssim_metric.compute().item()

        if torch.isnan(torch.tensor(epoch_r2)):
            epoch_r2 = -1.0

        print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        print(f"            ► Metrics | SSIM: {epoch_ssim:.4f} | R²: {epoch_r2:.4f} | MAE: {epoch_mae:.4f} | WMAPE: {epoch_wmape:.2f}%")
        
        # 🚀 STREAM REGRESSION CURVES LIVE TO WANDB 🚀
        wandb.log({
            "generator_epoch": epoch + 1,
            "generator/train_loss": avg_train_loss,
            "generator/val_loss": avg_val_loss,
            "generator/val_ssim": epoch_ssim,
            "generator/val_r2": epoch_r2,
            "generator/val_mae": epoch_mae,
            "generator/val_wmape": epoch_wmape,
            "generator/learning_rate": optimizer.param_groups[0]['lr']
        })
        
        # Checkpoint Saving & Tracking Execution Logic
        improved = False
        if avg_val_loss < best_val_loss - 1e-5:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"    ► New BEST model based on Loss | Val Loss: {best_val_loss:.5f}")
            improved = True

        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'epochs_no_improve': epochs_no_improve
        }, ckpt_path)

        # 🛡️ Early stopping guard updated to execute smoothly after warmup steps clear
        if not improved:
            epochs_no_improve += 1
            if epochs_no_improve >= patience and epoch > 35:
                print(f"  Early stopping triggered at epoch {epoch+1} to preserve generalization limits.")
                break

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"  Loaded BEST model | Val Loss: {best_val_loss:.5f}")

    return model

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

class HF_SegFormer(nn.Module):
    def __init__(self, num_channels=3, num_classes=1):
        super(HF_SegFormer, self).__init__()

        # We load a small, fast version (b0) or a powerful one (b3/b5)
        # We ignore the 'mismatched sizes' warning because we are retraining the head anyway
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            f'nvidia/{seg_model_name}', # Change to "nvidia/mit-b3" for more power
            num_labels=num_classes,
            num_channels=3, # Pre-trained models expect 3 input channels
            ignore_mismatched_sizes=True
        )

        # ADAPTER LAYER: If your input isn't 3 channels, we project it to 3
        self.use_adapter = (num_channels != 3)
        if self.use_adapter:
            self.input_adapter = nn.Conv2d(num_channels, 3, kernel_size=1)

    def forward(self, x):
        # 1. Adapt input if necessary
        if self.use_adapter:
            x = self.input_adapter(x)

        # 2. SegFormer requires resizing to 4x divisible (usually handled, but good to know)
        # 3. Forward pass
        # HuggingFace returns a specialized object, we need the 'logits'
        outputs = self.model(pixel_values=x)

        # 4. Upsample logits to original image size
        # SegFormer outputs logits at 1/4th resolution for speed
        upsampled_logits = nn.functional.interpolate(
            outputs.logits,
            size=x.shape[-2:], # Match input Height/Width
            mode="bilinear",
            align_corners=False
        )

        return upsampled_logits

In [ ]:
import torch
import torch.nn as nn
from transformers import SegformerForSemanticSegmentation

class HF_SAR_Regression(nn.Module):
    def __init__(self, n_channels=2, n_classes=1, model_name=f'nvidia/{reg_model_name}'):
        super().__init__()

        # 1. Load SegFormer from Hugging Face
        # "num_labels=1" tells HF we want a single output channel (Regression)
        # "ignore_mismatched_sizes=True" is needed because we are changing the head
        self.hf_model = SegformerForSemanticSegmentation.from_pretrained(
            model_name,
            num_labels=n_classes,
            ignore_mismatched_sizes=True
        )

        # 2. Input Adapter: SAR (2ch) -> SegFormer (3ch RGB)
        # We use a 1x1 convolution to "learn" how to mix SAR bands into RGB features
        self.input_adapter = nn.Conv2d(n_channels, 3, kernel_size=1)

        # 3. Output Activation (Sigmoid)
        # Matches your PowerfulUNet logic: constrains output to [0, 1] for NDWI
        self.final_activation = nn.Sigmoid()

    def forward(self, x):
        # x shape: (Batch, 2, H, W)

        # Step A: Adapt 2 channels -> 3 channels
        x = self.input_adapter(x)

        # Step B: Forward pass through SegFormer
        # Transformers expect strictly named arguments often, or pixel_values
        outputs = self.hf_model(pixel_values=x)

        # Step C: Get logits (Shape is usually H/4, W/4)
        logits = outputs.logits

        # Step D: Upsample back to original size
        # SegFormer is efficient because it processes at 1/4 resolution.
        # We must resize it back up to match the input "water mask" size.
        upsampled_logits = nn.functional.interpolate(
            logits,
            size=x.shape[-2:], # (H, W)
            mode="bilinear",
            align_corners=False
        )

        # Step E: Activation
        out = self.final_activation(upsampled_logits)

        return out

## Main Loop

In [ ]:
# =========================================================================
# 🔥 HUGGING FACE AUTHENTICATION (For High-Speed Model Downloads)
# =========================================================================
from huggingface_hub import login

try:
    # If running on Kaggle, pull securely from Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    # Authenticate via the huggingface_hub library
    login(token=hf_token)
    print("✅ Hugging Face token authenticated successfully! Downloads prioritized.")
    
    # Also set as an environment variable just in case timm/transformers needs it downstream
    os.environ["HF_TOKEN"] = hf_token 

except ImportError:
    # If running locally or on a different platform, set it directly
    print("⚠️ Kaggle Secrets not found. Using local environment token if available.")
    # os.environ["HF_TOKEN"] = "YOUR_PERSONAL_TOKEN_HERE"
    # login(token=os.environ["HF_TOKEN"])
except Exception as e:
    print(f"⚠️ Could not authenticate with Hugging Face: {e}")
# =========================================================================

In [ ]:
! rm -r "/kaggle/working/parallel_cv_speed_optimized"

In [ ]:
import numpy as np
import copy
import os
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# TorchMetrics for explicit final regression validation report
from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError, MeanSquaredError
from torchmetrics.image import StructuralSimilarityIndexMeasure

# 🚀 STEP 1: SECURE WANDB AUTHENTICATION 🚀
import wandb
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except Exception as e:
    print(f"❌ WandB Login Failed. Ensure WANDB_API_KEY is set in Kaggle Secrets. Error: {e}")

# --- GLOBAL CONFIGURATION ---
N_FOLDS = 5  
REG_EPOCHS = 400
SEG_EPOCHS = 100
PATIENCE = 12  
SAVE_DIR = '/kaggle/working/parallel_cv_speed_optimized_mixed'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =========================================================================
# 🛑 ASSIGN TARGET FOLD HERE FOR EACH INDEPENDENT BACKGROUND RUN (1 to 5):
# TARGET_FOLD = THE_FOLD_I_WANT
TARGET_FOLDs = THE_FOLDs_I_WANT
# =========================================================================

os.makedirs(SAVE_DIR, exist_ok=True)

def run_single_fold_speedrun(s1_full, s2_full, water_full, events, strata):
    
    # 🔄 CHANGED to standard KFold for mixed split
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    
    splits = list(kf.split(X=np.arange(len(s1_full))))
    
    for fold in range(N_FOLDS):
        t_idx, v_idx = splits[fold]
        print(f"Fold {fold+1} | Train Size: {len(t_idx)} | Val Size: {len(v_idx)}")
        
    for t_fold in TARGET_FOLDs:
        print(f"\n{'#'*60}")
        print(f"STARTING HIGH-SPEED BACKGROUND EXECUTION (MIXED): FOLD {t_fold}")
        print(f"{'#'*60}")
        print(np.unique(events[v_idx]))
        
        # 🚀 STEP 2: GROUP TRACKING UNDER YOUR EXACT ACADEMIC ENTITY 🚀
        wandb.init(
            entity="birdhunterx91-bangladesh-university-of-engineering-and-t",
            project=f"flood-segmentation-cv-mixed-tile_size-{tile_size_I_want}",  # 🔄 Changed project to isolate mixed runs
            group=f"reg-{reg_model_name}-seg-{seg_model_name}",  # Kept group the exact same
            name=f"worker_fold_{t_fold}",
            job_type="train-fold",
            config={
                "fold_index": t_fold,
                "total_folds": N_FOLDS,
                "batch_size": 32,
                "seg_epochs": SEG_EPOCHS,
                "reg_epochs": REG_EPOCHS,
                "split_type": "random_kfold"
            }
        )
        
        train_idx, val_idx = splits[t_fold - 1]
        
        X_s1 = torch.as_tensor(s1_full)
        X_s2 = torch.as_tensor(s2_full)
        Y_w  = torch.as_tensor(water_full)
        
        BATCH_SIZE = 32
        
        reg_train = DataLoader(TensorDataset(X_s1[train_idx], X_s2[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
        reg_val   = DataLoader(TensorDataset(X_s1[val_idx],   X_s2[val_idx]),   batch_size=BATCH_SIZE, shuffle=False)
    
        # --- A. REGRESSION GENERATOR MODULE ---
        print(f"\n[Fold {t_fold}] Initializing SAR-to-Optical Regression Generator...")
        reg_model = HF_SAR_Regression(n_channels=2, n_classes=1).to(device)
        reg_final_path = os.path.join(SAVE_DIR, f"fold_{t_fold}_reg_best.pth")
    
        if os.path.exists(reg_final_path):
            print(f" -> Found pre-existing weights for Fold {t_fold} regression. Loading directly...")
            reg_model.load_state_dict(torch.load(reg_final_path, map_location=device))
        else:
            reg_model = train_regression_fold(reg_model, reg_train, reg_val, fold_idx=t_fold, save_dir=SAVE_DIR, wandb=wandb, epochs=REG_EPOCHS)
    
        # COMPILE REGRESSION EVALUATION METRICS
        reg_model.eval()
        for p in reg_model.parameters(): 
            p.requires_grad = False
    
        val_r2_metric = R2Score().to(device)
        val_mae_metric = MeanAbsoluteError().to(device)
        val_wmape_metric = WeightedMeanAbsolutePercentageError().to(device)
        val_mse_metric = MeanSquaredError().to(device)
        val_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    
        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for inputs, targets in reg_val:
                inputs = inputs.float().to(device).permute(0, 3, 1, 2).contiguous()
                targets = targets.float().to(device).permute(0, 3, 1, 2).contiguous()
                
                out = reg_model(inputs)
                out_det, tar_det = out.detach(), targets.detach()
                
                val_r2_metric.update(out_det.flatten(), tar_det.flatten())
                val_mae_metric.update(out_det.flatten(), tar_det.flatten())
                val_wmape_metric.update(out_det.flatten(), tar_det.flatten())
                val_mse_metric.update(out_det.flatten(), tar_det.flatten())
                val_ssim_metric.update(out_det, tar_det)
    
        r2_res = val_r2_metric.compute().item()
        ssim_res = val_ssim_metric.compute().item()
        mae_res = val_mae_metric.compute().item()
        wmape_res = val_wmape_metric.compute().item() * 100
        mse_res = val_mse_metric.compute().item()

        print(f"\n================ REGRESSION MULTI-METRIC PERFORMANCE ================")
        print(f"  R² Score:  {r2_res:.4f} | SSIM:  {ssim_res:.4f} | MAE:  {mae_res:.4f}")
        print(f"=====================================================================")
        
        # Stream baseline regression accuracy to dashboard
        wandb.log({
            "regression/r2_score": r2_res,
            "regression/ssim": ssim_res,
            "regression/mae": mae_res,
            "regression/wmape_pct": wmape_res,
            "regression/mse": mse_res
        })
    
        # --- B. SEGMENTATION MULTI-STRATEGY COMPONENT ---
        seg_fake = HF_SegFormer(num_channels=1).to(device)
        seg_s1   = HF_SegFormer(num_channels=2).to(device)
        seg_comb = HF_SegFormer(num_channels=3).to(device)
    
        opt_fake = optim.Adam(seg_fake.parameters(), lr=5e-5, weight_decay=1e-3)
        opt_s1   = optim.Adam(seg_s1.parameters(), lr=5e-5, weight_decay=1e-3)
        opt_comb = optim.Adam(seg_comb.parameters(), lr=5e-5, weight_decay=1e-3)
    
        sched_fake = optim.lr_scheduler.ReduceLROnPlateau(opt_fake, mode='min', factor=0.5, patience=5)
        sched_s1   = optim.lr_scheduler.ReduceLROnPlateau(opt_s1, mode='min', factor=0.5, patience=5)
        sched_comb = optim.lr_scheduler.ReduceLROnPlateau(opt_comb, mode='min', factor=0.5, patience=5)
    
        seg_crit = ComboLoss()
        
        best_losses = {'fake': float('inf'), 's1': float('inf'), 'comb': float('inf')}
        epochs_no_improve = 0
    
        # Pre-generate synthetic imagery features to eliminate execution bottlenecks
        print(f"\n[Fold {t_fold}] Caching static synthetic features...")
        def generate_static_fakes(s1_tensor, batch_size=16):
            fake_list = []
            with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                for i in range(0, len(s1_tensor), batch_size):
                    x_batch = s1_tensor[i:i+batch_size].float().to(device).permute(0, 3, 1, 2).contiguous()
                    pred = reg_model(x_batch)
                    fake_list.append(pred.cpu()) 
            return torch.cat(fake_list, dim=0)
    
        fake_s2_train = generate_static_fakes(X_s1[train_idx])
        fake_s2_val   = generate_static_fakes(X_s1[val_idx])
    
        train_s1_ready = X_s1[train_idx].permute(0, 3, 1, 2)
        val_s1_ready   = X_s1[val_idx].permute(0, 3, 1, 2)
        train_yw_ready = Y_w[train_idx].permute(0, 3, 1, 2)
        val_yw_ready   = Y_w[val_idx].permute(0, 3, 1, 2)
    
        seg_train = DataLoader(TensorDataset(train_s1_ready, fake_s2_train, train_yw_ready), batch_size=BATCH_SIZE, shuffle=True)
        seg_val   = DataLoader(TensorDataset(val_s1_ready,   fake_s2_val,   val_yw_ready),   batch_size=BATCH_SIZE, shuffle=False)
    
        epoch_bar = tqdm(range(SEG_EPOCHS), desc=f"Fold {t_fold} Epochs")
        for epoch in epoch_bar:
            seg_fake.train(); seg_s1.train(); seg_comb.train()
            train_losses = {'fake': 0.0, 's1': 0.0, 'comb': 0.0}
            n_train_batches = 0
    
            for x_s1, fake_s2, y_w in seg_train:
                x_s1, fake_s2, y_w = x_s1.to(device).float(), fake_s2.to(device).float(), y_w.to(device).float()
    
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    # Strategy 1: Fake S2 Only
                    opt_fake.zero_grad()
                    loss_fake = seg_crit(seg_fake(fake_s2), y_w)
                    loss_fake.backward(); opt_fake.step()
                    train_losses['fake'] += loss_fake.item()
    
                    # Strategy 2: S1 Only
                    opt_s1.zero_grad()
                    loss_s1 = seg_crit(seg_s1(x_s1), y_w)
                    loss_s1.backward(); opt_s1.step()
                    train_losses['s1'] += loss_s1.item()
    
                    # Strategy 3: Combined Frame
                    opt_comb.zero_grad()
                    loss_comb = seg_crit(seg_comb(torch.cat([fake_s2, x_s1], dim=1)), y_w)
                    loss_comb.backward(); opt_comb.step()
                    train_losses['comb'] += loss_comb.item()
    
                n_train_batches += 1
    
            # --- VALIDATION EXECUTION FRAME ---
            seg_fake.eval(); seg_s1.eval(); seg_comb.eval()
            val_metrics = {
                'fake': {'loss': 0.0, 'inter': 0.0, 'union': 0.0}, 
                's1': {'loss': 0.0, 'inter': 0.0, 'union': 0.0}, 
                'comb': {'loss': 0.0, 'inter': 0.0, 'union': 0.0}
            }
            n_val_batches = 0
    
            with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                for x_s1, fake_s2, y_w in seg_val:
                    x_s1, fake_s2, y_w = x_s1.to(device).float(), fake_s2.to(device).float(), y_w.to(device).float()
                    input_comb = torch.cat([fake_s2, x_s1], dim=1)
    
                    def get_loss_and_pixels(preds, target):
                        loss = seg_crit(preds, target).item()
                        mask = (torch.sigmoid(preds) > 0.5).float()
                        inter = (mask * target).sum().item()
                        union = (mask.sum() + target.sum()).item() - inter
                        return loss, inter, union
    
                    l, i, u = get_loss_and_pixels(seg_fake(fake_s2), y_w)
                    val_metrics['fake']['loss'] += l; val_metrics['fake']['inter'] += i; val_metrics['fake']['union'] += u
    
                    l, i, u = get_loss_and_pixels(seg_s1(x_s1), y_w)
                    val_metrics['s1']['loss'] += l; val_metrics['s1']['inter'] += i; val_metrics['s1']['union'] += u
    
                    l, i, u = get_loss_and_pixels(seg_comb(input_comb), y_w)
                    val_metrics['comb']['loss'] += l; val_metrics['comb']['inter'] += i; val_metrics['comb']['union'] += u
                    n_val_batches += 1
    
            results = {k: {
                'train_loss': train_losses[k] / n_train_batches,
                'val_loss': val_metrics[k]['loss'] / n_val_batches,
                'val_iou': (val_metrics[k]['inter'] + 1e-6) / (val_metrics[k]['union'] + 1e-6)
            } for k in ['fake', 's1', 'comb']}
    
            # LIVE DASHBOARD TRANSMISSION
            wandb.log({
                "epoch": epoch + 1,
                "fake_s2/train_loss": results['fake']['train_loss'], "fake_s2/val_loss": results['fake']['val_loss'], "fake_s2/val_iou": results['fake']['val_iou'],
                "s1_only/train_loss": results['s1']['train_loss'], "s1_only/val_loss": results['s1']['val_loss'], "s1_only/val_iou": results['s1']['val_iou'],
                "combined/train_loss": results['comb']['train_loss'], "combined/val_loss": results['comb']['val_loss'], "combined/val_iou": results['comb']['val_iou']
            })
    
            sched_fake.step(results['fake']['val_loss'])
            sched_s1.step(results['s1']['val_loss'])
            sched_comb.step(results['comb']['val_loss'])
    
            any_model_improved = False
            for k, m in [('fake', seg_fake), ('s1', seg_s1), ('comb', seg_comb)]:
                if results[k]['val_loss'] < best_losses[k] - 1e-4:
                    best_losses[k] = results[k]['val_loss']
                    torch.save(m.state_dict(), os.path.join(SAVE_DIR, f"fold_{t_fold}_best_{k}.pth"))
                    any_model_improved = True
    
            if any_model_improved:
                epochs_no_improve = 0  
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE and epoch > 40:
                    break
    
        # --- C. FINAL ACCUMULATION & STATISTICAL METRIC UNIFICATION ---
        print("\n[Evaluation] Loading optimized validation weights for final scoring panels...")
        seg_fake.load_state_dict(torch.load(os.path.join(SAVE_DIR, f"fold_{t_fold}_best_fake.pth"), map_location=device))
        seg_s1.load_state_dict(torch.load(os.path.join(SAVE_DIR, f"fold_{t_fold}_best_s1.pth"), map_location=device))
        seg_comb.load_state_dict(torch.load(os.path.join(SAVE_DIR, f"fold_{t_fold}_best_comb.pth"), map_location=device))
        
        seg_fake.eval(); seg_s1.eval(); seg_comb.eval()
        
        # 1. Lists to hold per-image confusion matrix details
        fake_tp_list, fake_fp_list, fake_fn_list, fake_tn_list = [], [], [], []
        s1_tp_list, s1_fp_list, s1_fn_list, s1_tn_list = [], [], [], []
        comb_tp_list, comb_fp_list, comb_fn_list, comb_tn_list = [], [], [], []
        
        # 2. Global pixel-level registers for true Dataset Micro IoU summary
        global_fake_inter, global_fake_union = 0.0, 0.0
        global_s1_inter, global_s1_union = 0.0, 0.0
        global_comb_inter, global_comb_union = 0.0, 0.0
        
        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for x_s1, fake_s2, y_w in seg_val:
                x_s1, fake_s2 = x_s1.to(device).float(), fake_s2.to(device).float()
                y_true_np = y_w.byte().cpu().numpy() 
                
                # Fetch threshold predictions
                p_fake = (torch.sigmoid(seg_fake(fake_s2)) > 0.5).byte().cpu().numpy()
                p_s1 = (torch.sigmoid(seg_s1(x_s1)) > 0.5).byte().cpu().numpy()
                p_comb = (torch.sigmoid(seg_comb(torch.cat([fake_s2, x_s1], dim=1))) > 0.5).byte().cpu().numpy()
                
                # Process the confusion matrix counts for each image in the batch
                def process_confusion_matrix(y_true, y_pred):
                    # Flatten arrays per image to (B, H*W) for easy vectorized counts
                    y_t = y_true.reshape(y_true.shape[0], -1).astype(bool)
                    y_p = y_pred.reshape(y_pred.shape[0], -1).astype(bool)
                    
                    tp = (y_t & y_p).sum(axis=1)
                    fp = (~y_t & y_p).sum(axis=1)
                    fn = (y_t & ~y_p).sum(axis=1)
                    tn = (~y_t & ~y_p).sum(axis=1)
                    
                    # Intersect / Union for legacy global metrics
                    inter = tp
                    union = tp + fp + fn
                    
                    return list(tp), list(fp), list(fn), list(tn), inter.sum(), union.sum()
                
                # Extract and store values
                tp, fp, fn, tn, i, u = process_confusion_matrix(y_true_np, p_fake)
                fake_tp_list.extend(tp); fake_fp_list.extend(fp); fake_fn_list.extend(fn); fake_tn_list.extend(tn)
                global_fake_inter += i; global_fake_union += u
                
                tp, fp, fn, tn, i, u = process_confusion_matrix(y_true_np, p_s1)
                s1_tp_list.extend(tp); s1_fp_list.extend(fp); s1_fn_list.extend(fn); s1_tn_list.extend(tn)
                global_s1_inter += i; global_s1_union += u
                
                tp, fp, fn, tn, i, u = process_confusion_matrix(y_true_np, p_comb)
                comb_tp_list.extend(tp); comb_fp_list.extend(fp); comb_fn_list.extend(fn); comb_tn_list.extend(tn)
                global_comb_inter += i; global_comb_union += u
        
        # Calculate Micro IoU summaries
        iou_fake = (global_fake_inter + 1e-6) / (global_fake_union + 1e-6)
        iou_s1   = (global_s1_inter + 1e-6) / (global_s1_union + 1e-6)
        iou_comb = (global_comb_inter + 1e-6) / (global_comb_union + 1e-6)
        
        # Build raw master dataset framework
        df_per_image = pd.DataFrame({
            'fold': int(t_fold),    
            'val_idx': val_idx,     
            
            # Fake S2 Model counts
            'fake_tp': fake_tp_list, 'fake_fp': fake_fp_list, 'fake_fn': fake_fn_list, 'fake_tn': fake_tn_list,
            
            # S1 Model counts
            's1_tp': s1_tp_list, 's1_fp': s1_fp_list, 's1_fn': s1_fn_list, 's1_tn': s1_tn_list,
            
            # Combined Model counts
            'comb_tp': comb_tp_list, 'comb_fp': comb_fp_list, 'comb_fn': comb_fn_list, 'comb_tn': comb_tn_list,
        })
        
        # Save local CSV
        csv_path = os.path.join(SAVE_DIR, f"fold_{t_fold}_per_image_confusion_matrices.csv")
        df_per_image.to_csv(csv_path, index=False)
        
        # Log Table and final summary metrics to WandB
        wandb.log({
            "validation/per_image_table": wandb.Table(dataframe=df_per_image),
            "final_summary/fake_s2_mean_iou": iou_fake,
            "final_summary/s1_only_mean_iou": iou_s1,
            "final_summary/combined_mean_iou": iou_comb
        })
        
        # Formulate unified tracking checkpoint
        unified_checkpoint = {
            'reg': reg_model.state_dict(),
            'seg_fake': torch.load(os.path.join(SAVE_DIR, f"fold_{t_fold}_best_fake.pth"), map_location='cpu'),
            'seg_s1': torch.load(os.path.join(SAVE_DIR, f"fold_{t_fold}_best_s1.pth"), map_location='cpu'),
            'seg_comb': torch.load(os.path.join(SAVE_DIR, f"fold_{t_fold}_best_comb.pth"), map_location='cpu'),
            'best_ious': {'fake': iou_fake, 's1': iou_s1, 'comb': iou_comb}
        }
        torch.save(unified_checkpoint, os.path.join(SAVE_DIR, f"fold_{t_fold}_best_strategies.pth"))
        
        wandb.finish()
        print(f"\n⚡ [SUCCESS] FOLD {t_fold} RUN RAW CONFUSION MATRICES STACKED ON CENTRAL DASHBOARD ⚡")

# Run execution framework
run_single_fold_speedrun(
    s1_full=S1_filtered, 
    s2_full=S2_filtered, 
    water_full=Water_filtered, 
    events=event_arr_filtered, 
    strata=df_profile['spatial_cluster'].values
)

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset
# import pandas as pd
# import numpy as np
# import os
# from tqdm.auto import tqdm
# from sklearn.metrics import jaccard_score, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score

# # 🔥 Imported TorchMetrics for explicit independent regression validation
# from torchmetrics.regression import R2Score, MeanAbsoluteError, WeightedMeanAbsolutePercentageError, MeanSquaredError
# from torchmetrics.image import StructuralSimilarityIndexMeasure

# # --- CONFIGURATION ---
# SAVE_DIR = '/kaggle/working/parallel_cv_speed_optimized'
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # =========================================================================
# # ASSIGN TARGET FOLD TO EVALUATE HERE:

# # =========================================================================

# def prepare_tensor(data):
#     if torch.is_tensor(data):
#         return data.float()
#     elif hasattr(data, 'values'): 
#         return torch.tensor(data.values).float()
#     else: 
#         return torch.tensor(data).float()

# def calculate_segmentation_metrics(y_true, y_probs, threshold=0.5):
#     y_pred_bin = (y_probs > threshold).astype(int)
#     return {
#         'iou': jaccard_score(y_true, y_pred_bin, zero_division=0),
#         'f1': f1_score(y_true, y_pred_bin, zero_division=0),
#         'accuracy': accuracy_score(y_true, y_pred_bin),
#         'precision': precision_score(y_true, y_pred_bin, zero_division=0),
#         'recall': recall_score(y_true, y_pred_bin, zero_division=0),
#         'auc': roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else 0.5
#     }

# def evaluate_on_independent_test(s1_test, s2_test, water_test, fold_map):
#     print(f"\n{'#'*60}")
#     print("STARTING INDEPENDENT TEST SET EVALUATION")
#     print(f"{'#'*60}")

#     t_s1 = prepare_tensor(s1_test)
#     t_s2 = prepare_tensor(s2_test)
#     t_w  = prepare_tensor(water_test)

#     print(f"Test Data Shapes -> S1: {t_s1.shape}, S2: {t_s2.shape}, Water: {t_w.shape}")

#     test_dataset = TensorDataset(t_s1, t_s2, t_w)
#     test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

#     test_results = {}
#     strategies = ['Fake', 'S1', 'Comb']

#     for strategy in strategies:
#         fold_idx = fold_map[strategy]
#         print(f"\n>>> Testing Strategy: {strategy} (Using Model from Fold {fold_idx})")

#         # 🔥 FIXED: Changed back to "nvidia/mit-b3" to perfectly mirror your training architecture!
#         reg_model = HF_SAR_Regression(n_channels=2, n_classes=1, model_name="se_resnext50_32x4d").to(device)
        
#         if strategy == 'Fake':
#             seg_model = HF_SegFormer(num_channels=1).to(device)
#         elif strategy == 'S1':
#             seg_model = HF_SegFormer(num_channels=2).to(device)
#         elif strategy == 'Comb':
#             seg_model = HF_SegFormer(num_channels=3).to(device)

#         path = os.path.join(SAVE_DIR, f"fold_{fold_idx}_best_strategies.pth")
#         if not os.path.exists(path):
#             print(f"CRITICAL ERROR: Checkpoint not found at {path}")
#             continue

#         checkpoint = torch.load(path, map_location=device,weights_only=False)

#         reg_model.load_state_dict(checkpoint['reg'])
#         reg_model.eval()

#         seg_key = f"seg_{strategy.lower()}"
#         seg_model.load_state_dict(checkpoint[seg_key])
#         seg_model.eval()

#         # 🔥 Initialize Regression Trackers for image generation analytics
#         if strategy in ['Fake', 'Comb']:
#             reg_r2 = R2Score().to(device)
#             reg_mae = MeanAbsoluteError().to(device)
#             reg_wmape = WeightedMeanAbsolutePercentageError().to(device)
#             reg_mse = MeanSquaredError().to(device)
#             reg_ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

#         all_preds = []
#         all_targets = []

#         with torch.no_grad():
#             for x_s1, x_s2, y_w in tqdm(test_loader, desc=f"Testing {strategy}"):
#                 x_s1 = x_s1.to(device).permute(0,3,1,2).contiguous()
#                 x_s2 = x_s2.to(device).permute(0,3,1,2).contiguous()
#                 y_w  = y_w.to(device).permute(0,3,1,2).contiguous()

#                 if strategy == 'S1':
#                     preds = seg_model(x_s1)
#                 elif strategy == 'Fake':
#                     fake_s2 = reg_model(x_s1)
#                     preds = seg_model(fake_s2)
                    
#                     # Update batch-wise regression metrics
#                     reg_r2.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_mae.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_wmape.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_mse.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_ssim.update(fake_s2, x_s2)
                    
#                 elif strategy == 'Comb':
#                     fake_s2 = reg_model(x_s1)
#                     input_comb = torch.cat([fake_s2, x_s1], dim=1)
#                     preds = seg_model(input_comb)
                    
#                     # Update batch-wise regression metrics
#                     reg_r2.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_mae.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_wmape.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_mse.update(fake_s2.flatten(), x_s2.flatten())
#                     reg_ssim.update(fake_s2, x_s2)

#                 all_preds.append(preds.cpu())
#                 all_targets.append(y_w.cpu())

#         full_preds = torch.cat(all_preds)
#         full_targets = torch.cat(all_targets)

#         y_probs = torch.sigmoid(full_preds).numpy().flatten()
#         y_true = (full_targets.numpy().flatten() > 0.5).astype(int)

#         # Output the isolated regression performance block
#         if strategy in ['Fake', 'Comb']:
#             print(f"\n    📊 [REGRESSION PERFORMANCE PROFILE - FOLD {fold_idx}]")
#             print(f"      -> R² Score: {reg_r2.compute().item():.4f}")
#             print(f"      -> SSIM:     {reg_ssim.compute().item():.4f}")
#             print(f"      -> MAE:      {reg_mae.compute().item():.4f}")
#             print(f"      -> MSE:      {reg_mse.compute().item():.5f}")
#             print(f"      -> WMAPE:    {reg_wmape.compute().item() * 100:.2f}%")

#         metrics = calculate_segmentation_metrics(y_true, y_probs)
#         test_results[strategy] = metrics
#         print(f"    [Segmentation Test Result] IoU: {metrics['iou']:.4f} | F1: {metrics['f1']:.4f} | Accuracy: {metrics['accuracy']:.4f} | Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}\n")

#     print(f"\n{'='*75}")
#     print("FINAL INDEPENDENT TEST SET RESULTS")
#     print(f"{'='*75}")

#     res_df = pd.DataFrame(test_results).T
#     cols = ['iou', 'f1', 'accuracy', 'precision', 'recall', 'auc']
#     existing_cols = [c for c in cols if c in res_df.columns]
#     res_df = res_df[existing_cols]

#     print(res_df.to_string())
#     return res_df

# # --- EXECUTE ---
# # 1. Manually map all strategies to use your designated target fold
# target_fold_map = {'Fake': TARGET_FOLD, 'S1': TARGET_FOLD, 'Comb': TARGET_FOLD}

# # 2. Run independent test evaluation on those chosen weights
# # (Ensure S1_test, S2_test, and Water_test are available in your notebook workspace!)
# test_metrics = evaluate_on_independent_test(S1_test, S2_test, Water_test, target_fold_map)

In [ ]:
# type(S2_test)

In [ ]:
# import numpy as np

# # Assuming your array is named 'arr'
# overall_std = np.std(S2_test)


In [ ]:
# np.std(S2_cv_pool)

In [ ]:
# overall_std

In [ ]:
# import numpy as np

# # Assuming you calculated disagreement_PCT for all tiles
# disagreements = np.sort(disagreement_PCT)
# cdf = np.arange(1, len(disagreements) + 1) / len(disagreements)

In [ ]:
# ! rm -r "/kaggle/working/parallel_cv_speed_optimized"

In [ ]:
# cdf

In [ ]:
# plt.scatter(cdf)

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset
# import pandas as pd
# import numpy as np
# import os
# from tqdm.auto import tqdm
# from sklearn.metrics import jaccard_score, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score

# # --- CONFIGURATION ---
# SAVE_DIR = '/kaggle/working/parallel_cv_speed_optimized'
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# def prepare_tensor(data):
#     if torch.is_tensor(data):
#         return data.float()
#     elif hasattr(data, 'values'): 
#         return torch.tensor(data.values).float()
#     else: 
#         return torch.tensor(data).float()

# def calculate_segmentation_metrics(y_true, y_probs, threshold=0.5):
#     y_pred_bin = (y_probs > threshold).astype(int)
#     return {
#         'iou': jaccard_score(y_true, y_pred_bin, zero_division=0),
#         'f1': f1_score(y_true, y_pred_bin, zero_division=0),
#         'accuracy': accuracy_score(y_true, y_pred_bin),
#         'precision': precision_score(y_true, y_pred_bin, zero_division=0),
#         'recall': recall_score(y_true, y_pred_bin, zero_division=0),
#         'auc': roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else 0.5
#     }

# def find_best_folds_automatically(save_dir, total_folds=5):
#     """
#     Scans the saved checkpoint files, parses their best validation IoU values,
#     and returns a dictionary map targeting the highest performing fold per strategy.
#     """
#     print("\n🔍 Scanning cross-validation checkpoints for optimal metrics...")
#     strategies = ['fake', 's1', 'comb']
#     top_scores = {s: -1.0 for s in strategies}
#     best_fold_map = {s.capitalize(): None for s in strategies}

#     for fold in range(1, total_folds + 1):
#         path = os.path.join(save_dir, f"fold_{fold}_best_strategies.pth")
#         if os.path.exists(path):
#             try:
#                 # Load on CPU to avoid allocating unnecessary VRAM during scanning
#                 chk = torch.load(path, map_location='cpu')
#                 if 'best_ious' in chk:
#                     ious = chk['best_ious']
#                     for s in strategies:
#                         if s in ious and ious[s] > top_scores[s]:
#                             top_scores[s] = ious[s]
#                             best_fold_map[s.capitalize()] = fold
#             except Exception as e:
#                 print(f"      ⚠️ Warning: Could not parse fold {fold} checkpoint. Skipping. ({e})")

#     # Fallback to defaults if checkpoints are missing or unreadable
#     for s in best_fold_map:
#         if best_fold_map[s] is None:
#             print(f"  ⚠️ No checkpoint found for Strategy {s}. Defaulting to Fold 1.")
#             best_fold_map[s] = 1

#     print("📌 Automated Selection Results:")
#     for s in best_fold_map:
#         print(f"   -> Strategy [{s}]: Best CV Fold is {best_fold_map[s]} (Val IoU: {top_scores[s.lower()]:.4f})")
    
#     return best_fold_map

# def evaluate_on_independent_test(s1_test, s2_test, water_test, best_folds):
#     print(f"\n{'#'*60}")
#     print("STARTING INDEPENDENT TEST SET EVALUATION")
#     print(f"{'#'*60}")

#     t_s1 = prepare_tensor(s1_test)
#     t_s2 = prepare_tensor(s2_test)
#     t_w  = prepare_tensor(water_test)

#     print(f"Test Data Shapes -> S1: {t_s1.shape}, S2: {t_s2.shape}, Water: {t_w.shape}")

#     test_dataset = TensorDataset(t_s1, t_s2, t_w)
#     test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

#     test_results = {}
#     strategies = ['Fake', 'S1', 'Comb']

#     for strategy in strategies:
#         fold_idx = best_folds[strategy]
#         print(f"\n>>> Testing Strategy: {strategy} (Using Model from Fold {fold_idx})")

#         reg_model = HF_SAR_Regression(n_channels=2, n_classes=1, model_name="efficientnet-b5").to(device)
        
#         if strategy == 'Fake':
#             seg_model = HF_SegFormer(num_channels=1).to(device)
#         elif strategy == 'S1':
#             seg_model = HF_SegFormer(num_channels=2).to(device)
#         elif strategy == 'Comb':
#             seg_model = HF_SegFormer(num_channels=3).to(device)

#         path = os.path.join(SAVE_DIR, f"fold_{fold_idx}_best_strategies.pth")
#         if not os.path.exists(path):
#             print(f"CRITICAL ERROR: Checkpoint not found at {path}")
#             continue

#         checkpoint = torch.load(path, map_location=device)

#         reg_model.load_state_dict(checkpoint['reg'])
#         reg_model.eval()

#         seg_key = f"seg_{strategy.lower()}"
#         seg_model.load_state_dict(checkpoint[seg_key])
#         seg_model.eval()

#         all_preds = []
#         all_targets = []

#         with torch.no_grad():
#             for x_s1, _, y_w in tqdm(test_loader, desc=f"Testing {strategy}"):
#                 x_s1 = x_s1.to(device).permute(0,3,1,2)
#                 y_w  = y_w.to(device).permute(0,3,1,2)

#                 if strategy == 'S1':
#                     preds = seg_model(x_s1)
#                 elif strategy == 'Fake':
#                     fake_s2 = reg_model(x_s1)
#                     preds = seg_model(fake_s2)
#                 elif strategy == 'Comb':
#                     fake_s2 = reg_model(x_s1)
#                     input_comb = torch.cat([fake_s2, x_s1], dim=1)
#                     preds = seg_model(input_comb)

#                 all_preds.append(preds.cpu())
#                 all_targets.append(y_w.cpu())

#         full_preds = torch.cat(all_preds)
#         full_targets = torch.cat(all_targets)

#         y_probs = torch.sigmoid(full_preds).numpy().flatten()
#         y_true = full_targets.numpy().flatten().astype(int)

#         metrics = calculate_segmentation_metrics(y_true, y_probs)
#         test_results[strategy] = metrics
#         print(f"   [Test Result] IoU: {metrics['iou']:.4f} | F1: {metrics['f1']:.4f} | Accuracy: {metrics['accuracy']:.4f} | Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}")

#     print(f"\n{'='*75}")
#     print("FINAL INDEPENDENT TEST SET RESULTS")
#     print(f"{'='*75}")

#     res_df = pd.DataFrame(test_results).T
#     cols = ['iou', 'f1', 'accuracy', 'precision', 'recall', 'auc']
#     existing_cols = [c for c in cols if c in res_df.columns]
#     res_df = res_df[existing_cols]

#     print(res_df.to_string())

#     output_path = os.path.join(SAVE_DIR, 'independent_test_results.csv')
#     res_df.to_csv(output_path)
#     print(f"\nSaved tracking spreadsheet directly to: {output_path}")

#     return res_df

# # --- EXECUTE ---
# # 1. Automatically locate the best folds based on saved Cross-Validation scores
# best_folds = find_best_folds_automatically(SAVE_DIR, total_folds=5)

# # 2. Run independent test evaluation on those chosen weights
# test_metrics = evaluate_on_independent_test(S1_test, S2_test, Water_test, best_folds)

In [ ]:
# from google.colab import runtime

# # This command disconnects the runtime and deletes the virtual machine
# runtime.unassign()